# Deney 01 — Video Analiz & Hot-Moment Tespiti

**Amaç:** Uzun bir kaynak videodan (a) transcript çıkarımı, (b) sahne tespiti, (c) ses enerjisi analizi ve (d) LLM tabanlı "vurucu an" (hot-moment) skorlaması için aday model/sağlayıcıları **kalite + maliyet + hız** üzerinden karşılaştırmak.

**Tek kaynak politikası:** Bu notebook algoritmaları kendi içinde tekrar tanımlamaz — transkripsiyon, sahne tespiti, ses tepe analizi, hot-moment skorlama, önizleme klibi üretimi ve `hot_moments.json` şeması hepsi `local_pipeline/video_analysis.py` içindeki **kanonik** fonksiyonlardan gelir. `00_youtube_automation_master.ipynb` ve `verify_local.py` de aynı fonksiyonları çağırır. Bu notebook'un kendine özgü katkısı **çok-aday karşılaştırma (benchmark) döngüsüdür**; master notebook ise tek varsayılan adayla uçtan uca çalışır.

**Nasıl çalışır:**
- Colab'da Runtime → Change runtime type → **GPU (T4)** seçip baştan sona çalıştırabilirsin.
- Bazı adaylar API anahtarı gerektirir (OpenAI / Anthropic / Google). Anahtarı olmayanlar otomatik atlanır, notebook hata vermez.
- API anahtarlarını Colab'da soldaki 🔑 **Secrets** panelinden ekle: `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, `GOOGLE_API_KEY`. Hiçbiri yoksa sadece ücretsiz/lokal adaylar (faster-whisper, PySceneDetect, librosa) çalışır.


## Google model router (bu notebook'taki tum Gemini cagrilari icin tek merkez)

Bu notebook artik `genai.Client(...)` kurmuyor ve sabit `model='gemini-...'` yazmiyor.
Her Google cagrisi `local_pipeline/google_ai.py` uzerinden gider:

- **Colab Secret adlari:** `GeminiKey` (eski ad `GOOGLE_API_KEY` hala calisir),
  `YoutubeKEY` (eski ad `YOUTUBE_DATA_API_KEY`). Deger hicbir zaman yazdirilmaz;
  yalnizca `configured (GeminiKey)` / `missing` durumu gosterilir.
- **Tek anahtar, cok model:** ayni `GeminiKey` projenin yetkili oldugu tum modellere erisir.
- **Kota PROJE bazlidir, anahtar bazli DEGILDIR** (ai.google.dev/gemini-api/docs/rate-limits);
  ikinci bir anahtar limit artirmaz. Hesap/proje dondurerek kota asmak yasaktir
  (`ALLOW_ACCOUNT_QUOTA_ROTATION = False`).
- **Rota:** ilk basarili cagri rotayi bitirir; hicbir zaman tum katalog denenmez.
- **Ucretsiz katman veri politikasi:** gonderilen icerik Google tarafindan urun
  gelistirmede kullanilabilir ve insan gozden gecirenler okuyabilir. Ham telifli
  video/ses varsayilan olarak YUKLENMEZ; yalnizca yerel olarak turetilmis
  transcript/skor ozetleri gonderilir.
- **Kota defteri** (`results/google_quota_usage.json`) YEREL TAHMINDIR; sunucunun
  yetkili kotasi degildir. Gunluk sayaclar Pasifik gece yarisinda sifirlanir.

Bu notebook'ta kullanilan gorev: **hot-moment incelemesi** -> Rota A (`gemini-3.6-flash` -> `gemini-3.5-flash` -> `gemini-3.5-flash-lite` -> `gemma-4-31b-it`).


## 0. Kurulum

In [ ]:
import os, subprocess, sys
if os.path.isdir('/content'):
    packages = ['yt-dlp==2026.7.4', 'faster-whisper==1.2.1', 'scenedetect-headless==0.7.1', 'librosa==0.11.0', 'openai==2.50.0', 'anthropic==0.120.2', 'google-genai==2.15.0', 'pandas==3.0.5', 'tqdm==4.70.0', 'moviepy==2.2.1', 'imageio-ffmpeg==0.6.0']
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])
    print('Colab bagimliliklari kuruldu.')
else:
    print('Yerel calisma: proje .venv paketleri bir sonraki hucrede otomatik etkinlestirilecek.')

In [ ]:
import os, time, json, subprocess, shutil, sys, site
from pathlib import Path
if 'google.colab' not in sys.modules:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        venv_site_packages = candidate / '.venv' / 'Lib' / 'site-packages'
        if venv_site_packages.is_dir():
            site.addsitedir(str(venv_site_packages))
            print('Yerel .venv paketleri etkinlestirildi:', venv_site_packages)
            break
import pandas as pd
import imageio_ffmpeg

IS_COLAB = 'google.colab' in sys.modules or os.path.isdir('/content')
PROJECT_ROOT = Path.cwd()
RUNTIME_DIR = Path('/content') if IS_COLAB else PROJECT_ROOT / '.local_runtime'
RESULTS_DIR = PROJECT_ROOT / 'results'
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FFMPEG_BIN = shutil.which('ffmpeg') or imageio_ffmpeg.get_ffmpeg_exe()
FFPROBE_BIN = shutil.which('ffprobe')
print('Calisma ortami:', 'Colab' if IS_COLAB else 'Yerel Jupyter')
print('FFmpeg:', FFMPEG_BIN)
print('FFprobe:', FFPROBE_BIN or 'bulunamadi; on kontrol FFmpeg donusumunde yapilacak')

def get_secret(name):
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    return os.environ.get(name)

OPENAI_API_KEY = get_secret('OPENAI_API_KEY')
ANTHROPIC_API_KEY = get_secret('ANTHROPIC_API_KEY')
GOOGLE_API_KEY = get_secret('GeminiKey') or get_secret('GOOGLE_API_KEY')  # Colab Secret adi: GeminiKey (eski ad: GOOGLE_API_KEY)

print('OpenAI key:', 'var' if OPENAI_API_KEY else 'yok (bu adaylar atlanacak)')
print('Anthropic key:', 'var' if ANTHROPIC_API_KEY else 'yok (bu adaylar atlanacak)')
print('Google key:', 'var' if GOOGLE_API_KEY else 'yok (bu adaylar atlanacak)')

# Paylasilan durum sozlugu (Faz 6) - tum notebooklarda (01-06) ayni degerler kullanilir.
STATUS_VALUES = {
    'experimental', 'development_placeholder', 'review_required',
    'approved', 'insufficient_data', 'failed', 'production_ready',
}

results = []  # her satır: task, candidate, quality_score, cost_usd, latency_sec, notes, status

def log_result(task, candidate, quality_score, cost_usd, latency_sec, notes='', status='experimental'):
    assert status in STATUS_VALUES, f'Bilinmeyen status: {status}'
    results.append({
        'task': task,
        'candidate': candidate,
        'quality_score_1_5': quality_score,
        'cost_usd_estimate': round(cost_usd, 5),
        'latency_sec': round(latency_sec, 2),
        'notes': notes,
        'status': status,
    })
    print(f"[{task}] {candidate} -> latency={latency_sec:.2f}s cost=${cost_usd:.5f} quality={quality_score} status={status}")

## `local_pipeline.video_analysis` modülünü yükle (kanonik tek kaynak)

In [ ]:
# @title Yerel pipeline yukleyici (otomatik - acmaniz gerekmez)
import sys, os, tempfile, subprocess, base64, hashlib, zlib
from pathlib import Path

PROJECT_REPO_URL = ''  # opsiyonel: gercek bir git repo URL'si verirsen modul(ler) oradan klonlanir

_LOCAL_PIPELINE_MODULE_NAMES = ['video_analysis', 'render_utils', 'google_ai']

# Her modul icin: (base64 kaynak, SHA-256 hash). Hash, gomulu payload'in
# bozulmadigini/kesilmedigini dogrulamak icin kullanilir -- eslesmezse acik
# bir hata verilir, asla sessizce bozuk kod calistirilmaz.
_LOCAL_PIPELINE_PAYLOADS = {
    'video_analysis': {
        'sha256': 'eacc5e1ad15c78328ebcffddf527e948bbfe3ee494aea31c5fc1bc2ad1d164b3',
        'encoding': 'zlib+b85',
        'data': (
            'c-p;vYm?iymf&~&3S6aXn~Eh_?ZlZoa+EEP?Mz&g*eO4fz1!QRP$JY~M3EYT?6%y~RrAaC%hvv{Jr4i`KvLbgle?)T-Aw`q2L}h|'
            'g@eQ4aQIHvDqdw$T&IaFCUG8TN0ln^SZs@GvM=^BuS6m%xvtV87g=$Yu7%phrA)-?IGCL;spHagTtYYQHBAr4d?n&M5wqFySkzU$'
            'lFPWRihYc*mit&$vgBsd!Qf3%$yHJ8#Oz$`q&z5*R!S5%x!BiPl}^?GDNW)^ij~aQ+kIT_()>z%d@Z7Nu|LEm{-xx#EaPnO>o!#f'
            'S&Ds;$V{m8vs7cD)-soRGDQgw5Vzuh$i$iW@yDN_MN!I9L5Bo}InvM3c(AG~QEjD&*HsNI<Ls&^(`vg%6dP=mm`uDe04|JCCuuP`'
            '$apt6q-#W($3=Kqu?BEFJ(XS|77o=B8JyFmD4|J=(6jVfimfcAP{m+XR9lg2swn}WGJ+~U0iHs|dnwj1eUaus=NM+X5&JmJEBFt='
            'CiNjp*FewlAk7nbknq2}%8rwyOg9y_SQm#>Dp;BiKusiN4SbR#g$492;slnb6o-11rD`h`OzT)}q2n_oOOa<sQA%tS>xmMl3^pl1'
            'K?>Xe4WSpmJCyP|l{aFYr3d6boJc9Fy3B#m(tNU)`=UGwSpQh4s;prpN;w_8fJx(mMV1aD0lW!a)uG5$ikuDfqp@&GK;9*jS51Fc'
            'z|d78l2q-cgW+&E*p$UyESH-afiIUL-5-jw0%QR?@vIsQ^s`c$-&DOil*L*qWu8>&UNWphTy3*-WdQvQfAS0DV{B&Ln7_x=`E!w5'
            '@Y`4+2=gZ*ZIQ_9{oyELCGx{y@czT|4<Fwze|i4n$5-#gLfj4n{4<oFU>~G7<2W0R#V~>O%ZdX&SRS%?Ew@FM$P!=i5-dyk?{y0M'
            'dpKqY@d4&{jem<F@KU|mr0W#B<6_{$O$<v)uc2X56XP#&>5l;74lw)p^UrVJeR%b9`Tp(4cQ0Noe|i1#)!XH(|MB6~oA<BZzHzXf'
            '?hjuhpy|H2=AS$Mc_UYQ`X#<j`Nh}Q41JKu4X$^QUCX5^>T)evCPnTa%Oov90=Q3baTdejNKC$?m6<aNC=&@7@nD)CVDnQ4aRYlE'
            '#F$_LAv?taxjDzJ2E2qj19-m9Vap(>=kVwgZZg;ozx~!4`?uc!Vy?u^R_4G(2WTmisdx*+mN%*5@pE}2RYl<PBA>|pp*m7ySbx|V'
            'j98k7@L{$99(Jvg8Au|8`@x4+;%0^hD5}amhy+!EWRLcm3lS$Xw-HOMwccwWh+(^eAi-7)%WPDf76?Sxak#}<BTNlh>1VoWF9H)D'
            '|FPs+(27K@rX$-F0Ku-bJ*J>~5Z`FjKqJ&*q~!PT2K3AKMV%+F%Cab<&G6siV;=7`1y^-e!&KuWor_xo^KR%3XNOWC45AVISfXMP'
            '-!8<gYtUcftd`xQ6{=?#rlcK;`AAa%*0hr@@O(6INYXS%qQDQACJ&^?E{j!B);?Vtz=$$|v`Se4Y8Q4C;scbs^b4@3ixco0A?4B*'
            '+cisQ-yqx3E0Q{&rde?#p(jGTSEzIV@z?D;F{3t@F7cWPH8g9s+d`1+`MpSUSOa_%-HXR)bT9Ee=GhEgLJMF6P|RZ2&8<`M5;nBl'
            '!A6A*TYrx8D$U{&7t0`hH?-jOqG=}E$Dg8Rx??c9j*BebuacO^H5XApHIfa>9GcP#6b)uYV3LC`#CsHkpr}y+Om2!Y0b8XmS+anZ'
            'LU{%z%~gtqPVSg{#?o2KESnBq{q*(&5N`SF+jlP=)u6V;dWYnvU!XbuzLHfLADiD;$L-S0{$$0JNWfVCVins>OlohFUTrG^EZ0C9'
            '{0$5YZcf$?{t~YW5RzmBnC=-UMSQgd@)<P<b_3}_qzVle=U4pODk};`rHro-2#m5VLD3s<={C-JbXCwii>%)B&!i?0D!z^t{&JOm'
            '&PWRa?6suEvC3itsghtZ><E!vOe4wPLB(%LSywSm)+$c<3COmtmS}f@5uYr{UX3=J{Xt$WS7|;+b88H&d*g`VF*D*4w4wf+x;1}$'
            '93B1x996;|l<T?JfEihh!4@sqCukm$T7Cocv1<864gSD|?L2PaJaYSw1nOuoc72pVW`M01YfW12;|yd#B7T@Y`R41#&!4{_%Tp^p'
            'O%$|T&%fFI@EL61yb!-W|3#4TR>6u>a{`894w~m`U8dwqJZ6*M>Ob)gx4jbY>Kr!^ZF#aKpj(}z1)i{&uV9f%G$fe%LNnCV=uV&J'
            'Mi=-`S=jiiVOsi6M;$Py;6|bAw%?(I=0DWpLs^4KAV0xIUKTt0vul_?(3y1yvfO%~c$%YOJmA#{?d-`Ga9pmy3g?|P0Vgy=E0#DL'
            'sX1);Z8+I%viOR&$)hWoA1eBFg4+ccIr5oe<aSg_x3PRR!k(3W>hYM{Zyz6Uh}rylKBKYhgLpps?;dW~8`{TUNy!B*ew5xkelq&<'
            '$z1=nrN6vB>p5A#-~)UDR;2;JSyC)2(Bj;HAI*=`Ps`1oem~aNUq6{00zf1H;&@H1<Le?#mbtuwS%KxublS%|x%@24f+td3H^jtY'
            'h%ydPq)WhmV@Ic=Uj(b$ct}&2Wn|WU49fOEo)&ZlD_gTl1-sATF*FXK^OThqIkbQA3@%ox?&bnx##k}wDit6{74!ucTH7%Mk9XU}'
            '<g3|iHovq64YTy0&y|M%LTVvX<Snl8y0Vr!{{|4{g`@R5r{DFO9@hJ%I{zk<dX!+N-@hhy+W+uH0gWh78m`k_T&DM~2`MQfUMYTr'
            '>WR^1%|js`vgk-vp^c_My5l;4ubZx$$Lm8q?9)sbLaHm+2FF9w0aV6c0SwP_ndX~f4z_!d!MpF{Ov#beO&=_VN;Jr^l;2RCYjVRO'
            'DNqLlPKjt!XW0@d3yUVNC@8_?y`d0~jMrQ90zI|cGb@<R#^OwZvev&M4Ez~iUoC+RftIT?p6FR#EMZi+-Y)s6I77Qkoz2CC2d@D~'
            '4d4g1k5_4yR>!lu%RAqVBR$9yTk46KP=*e@{e;WHdy0c_3-3+TESp)U1^*a(Yq4<uVCooJEzy%03tScePVlA;r$nBRYSsj{Tn0v2'
            '9bd@>nipg6SK~b)vUm!1O`M<;0+5;WEfjnKIt?sbc&8S(Lof%S7Z~1XsT<v}SHrn@I-A{%1(I=z><?7lA5;KtU4Y?DUEp5|eigMM'
            'W>^qwjX5lsrHE?gboOZg6VE`iMjlPL3Bp1qw*HE|n&LCqWnid)6NQEgJS6wbfJ65@Xy*;MEQ=d*^)9XL6M$%B01?1%Z~BmL0G$Xa'
            'Vn~c6UR(ywJ79w^r^KhY1#rQH?mq>NHP7aQ_I?J@lh@LFVKJJ<2aI<{x2=9dyC+~FOLWm9GDo8Z5#XS&Mr}DV)H|rFAH3@8h3}yY'
            'HPd4d**ls#JF;sK=QYx3Hl4wkUk8Bh8gJBH(r$wE5(Ag5j}|?2z%Jcn7$Ubp>yYt+R*rD%0PVc(nDx*XgB|_-23;VCP4!9<&6@Xj'
            'Rx7%yA`I$ylAa-mfZj$j4M_z>R}y@0_=(`g&Rh6>IKnlCA9g##pCr$^AuhX^r^RASkDg&Y5M$!6Za%vkoSd~*>U>7BqyrNbxxS@2'
            'p=lI1Y-Z@qp<0x87gc@8<c0Pg>Br?|6ThuNyZ(H9kC0yyED>MDcnl(%VW!S7XL^7$oX=UDfCU65@foW;qpo8b86C^W@j*4Sj~EF-'
            'c&VwUCTWH|n9-4h+E3siUZDmnVC%6{(%EThPpnM$=NiM8`o&nh;0c%qtanX6aPCnC-&#7(!}jPFXqfOa1b;ULj))92%j+KamLTqk'
            'fg<bVdwIlgcf6%zvPA9xLJbC>f%cd3PDCpm(V7d5@>m%BM#fTS6Mx}QFKP^I()3z$Atjf4CN{KpX&-eonZ`-7G}uJ$B%>~f(+)m%'
            'n-SA2f<8dcW%D{3nf2NTWF{~TRD_lT8rF_Vtjc2$XXY)&yVi7I;2Me(e3YNo@=%Fa^aBi!6QGO$Fd#WLh5+@x_XCL+ksZ${1^WO+'
            'zBV=R6>S(Y7wqgWwLCC}o;KBFyi=Z?UloTXFa(}EEX4T?3?wkX5&#gR1=a+P@a^>Ji3rpd00DPhVThBVJw1Oyq9sY!m6akt(gCMq'
            '%tzpLgC&>A1WyeqFf^jzGLu13$J;Y9`M|!;57Rt}%Q8NK(OvFak7$N$hf`dYjc;rS#^ZE4y}Z-Ge~LV>j)G5SV9%I?+XP;Uyb#q*'
            'AqW~BeWK3+1P-ZVWs;U5;2M{Oig_UcA61eqNe`!ZxEr<XW>qf83uT@HOr0JZ8Pka@JdKVEfY(rY(fm5Pn6V*9L~^^MVGZ_gK^Aia'
            'gaT7LzmgI3ivZ?`hgW4EV4eq30rzg4W`UKOCZCY)2=Eosl%QVB=acixW+za8I+tBIyGmU^?@RBz0{qJ;Ua6?*FcDyS?=OaCEr*vQ'
            '@!g`e42%r4L~LAi1Jl=iWsS22xY6Ktd@`%Vup|e>HE!SuqdykkjPATiAZ<Mq#A2bhnTN}&#O#F@RzyYT&;VyOZ&SgQ6g&e&k$yG^'
            '@9}ryVff1(#l)$XKuA2WxJHFyh{b=vTH&%P+E}^)>=W`HIh0I|v*HGrC%xL*wdJN$fE*!z`43^NkyUlnZGs@gx8ht*&Y!fbJ3s;b'
            'd16?NB+AOC-u(3&1~n3oMbtnZ*@*lB<M5oNuw3C%#$_s1)DD|#<HP)YGUzUfYguN%N#Ic~t+U3~6w1BJd%Cm0ZqU(P+4dUS(_ZTe'
            'KMW}#vsQ#)Cdyp-X8^EL-%MxsA*RiCP}@8;r}u37)zf>rZ(u90;`Oc%z&|}D)zCIL$D}SAtMe7D8<>fdM3ms!eggbrk_ISk22Az+'
            'o>Oab^3TcEODml)@Sdi^=L>qd_|#0p0wXKU*I5lZKy1=dReY+??(sf8^6AAMx)L}_!NIxR#3kvhSmZJ;F|UcTl=NbPu)f)1QVt~#'
            '!4wb@0wDlKf+CEm_!S4JC_DwofbP)%8&EKRjJ{dA3ZN^tt(8pJpr&jh40%w*fer<;gaAoZO$;QllG_-gA|(@PU00OBL+wEv!tXrE'
            'QmnH=QS#TesLbI{oQuLHQd|{9f~h}3XQp|fEW@xELxN2Y=8X{rEuPi_!JJ_(nea|O6eZfHbj}d^x+AT*vk%dQWBdyb%D~}!>fxWh'
            '2h-4-{lf9)K{rjz+t&7RY(^d5;Rs~jDMDO@qMwnw;EXn6Ua81Ka6-{~XqaMvKg{0DwlxV8U4SV(`bST}jMX;*y^9+@0K_lCwfxds'
            'D6J1o&xQpqT()R>A#pDCiZ{X+9STlAxTS?|sP$#kW`l;C%;xggiAZF(owd!I(Kzm?Z3i2x^su?G{X<|*Ga#>h^X<Z?p0U(2(1f@)'
            '|262}EnEs;b|QY%gzuQphz~g06O|cA6QF4~vPaDVU9%vNozo+jW;F)J;_z8((t-WA#*FFFp^iZrXX#Zg6U(P9V1wXjz5;CI70zd9'
            '+j?AaWZ8+h_YY$kFq{nI-KBY*joU^g>@c*{>R5`2x5xetV)52&Lv9>ulnxGhIZg4wRMK1DlXWi2j~+&*5Q`oneIZ`T4bJx(a~iLK'
            '^<uLB2;avnY#23?lxPBw`M}mFX<fE!G&HtIrs6fUtyQJdg2`&AtVyEH2Gr5UwlL-h>);m005!a4<3QPt1|1{>HglegJq61h_TWrV'
            '>hQvexDF2m_|X^*9rFAqn*5U_EjfKV&L%{l$@RH=B<thkIb@;qcZ1zS7qRYMgumBE>j7^Id40V%r#pwkI?uz7-sYolwEMtm>GB`6'
            'bN}R4UNo4;4VD-^SO0}U$S$1rjfe%8P7-(2WZrZ;HOA2-5A(DS)8{Ajx;C7_(ss+dNL9c8!CT1pH@+fyi6Q+YjA_kIctZarH|p<p'
            'pze2_P9m&Lu_Y1J?f2wFzcgQx<FJI;;^A|50Pw4)csOAW3w%HA1tupy?B<2N=pmvMFGr-<3H&IN_k2cRCIp6|Os=1tA=7Tb;czn?'
            'ol7F-D8jvXqT|JM`D8Q|FN*yt&A~R%ayh{zt&W<3C*TS-k0?S1+fJ5d!+8mqwq3JPP|Iv^K6uuS!ab9pXl4m0hZ0-6xpFrW(<8y*'
            'oK4PWGe%P2algt^Ty)LGI&<!-lrl*{Z*bl@xP1p87>E3DSLiWdV9aKBjBztJEdexCr5)q;8H{^7DaC;znBQ*fYZPRP$6Dx(-BjxK'
            'Tb>x^dc+DfaYHW7M*zb=N9_)cV1)?Q%K}#D&D#&+3aO75ll)68cpFGc`*x|{QmUnjM!pwHoV3)lZbFwvMi?ir35>F3kE$;ePhk2q'
            'qD7z6J;EQeb62$3;1))#ym2TF=p^%WB7w2C3jk`I6C2{EC?}ZaUaz@tugJAAm3?O!@N7^YbwrgKyQj;M^SnrX#O>xNbSR53-vFLF'
            'KBoB2nG<dmdhPhp;#)Hz3#iF-=*Ymh;KS}q7pu;h7*x2qJ>~V1D{#K0oNj<(7>_Q87tbgEGoF0@KmR%TzyJH}^3jm<wrq~JpP7ND'
            'xowK!s|zpF;L^`dKaF>*t;zJjGj{T+`51kS$sV1#9@NMkzwLbcROTa!eXMpc8IISZ|JMXy#D6o7?P85W-sZy$jFmUwnjzfOPi=4-'
            'Hl7IQF!+2rYan|&`j6&UIMJgA=2tX<qIQ0TWqS;~<9WRQzH1?&W!K}5oCvTkYdTXVIfAPjj;{?2t%|&s;;~2Eup^l?O+Qud9a-@$'
            '#sr%+C3eFn9TALs#^E(*9Xixj1fD@1FufjcRLH2|ld<^c*=Q69PCQJq(SdNiSRV)TFdM_a?ci=1OCjVphq&Ure}R{cZg(fLmoaze'
            'UC-z{QlJNOZ5S-AVE`}*6D2E+O*eic@g<3OZ=s)%%ECS3WzGyt`X#H_ojm$d$zPpu$Q9!jf4R;{K>BeXt;SuGb<<{@D+Ay|0+Mrk'
            'Bz19f0e`bdzrls`cK#*D*PsQy^5=QySXj#UO#_w$JA5{7ZwRuLJ2u#Cdj9kT8FYhw#(~$fargSeB0hB;J37zI6ZvUuPbqTPlJ%a9'
            'p2(wx6J-f)Doa&YrZq$-3SZDTm&jaTijKvTdr8|!_={mUA}HZS3yoGUdQ43jG1|JkV7%&IwqP{epfS_!M~HEg>EXFQuv0c*X_nZ~'
            'p;C%Ij?#DUgL4M#;g~AAvlk$r=&iZ&ga5Xvdm!Kh-nJN8Rwf=3jDv@jy#U=-9b<%Nrrz{ej+nv6c+C3G4!N|TWw$TFQ|3|Y3^wA!'
            'XV*S^qz|MWs(>V#&KR&0N$}GJnqmWVd67cx9xHp-<=m5R%`oUmkf+00^Ujhgy`3<}0L`{R$04m|3X28!AlpL8d<5#QiDP<=eQSzN'
            'B3CN%sb=Ag8Xv*DV)&z$<NckDoSt&MM=isT0V4SvB0Dz`9lNg~iJx~6go(W1kcqIG*cK48_u#yp9Bk+iaIy2e*;oC&r-$MYbz4tl'
            '-?jrzZD1`AXWSReS@b+_!)yn17VSsvZe8J=Mb|^WtH1OP3DF@Z$%$5g!vyG!p7(WX`R|VG)u*y<3o>`_gVji-Zb&9AxerblG;jd<'
            '5kQz&E3|nJ&^B(pAazi*4`8?>2xR!McdaPwL0vksg4r5b+%_G~LZx?hd2#02EN7Q@U_f+$A`ZS6lxOJl!y>M}ADCwz^+MBD(!JmU'
            'YYN~EPizRc;)r$v8!;SaV|lyezGN(|^jz`EQSzRMQ|nHzn{(|z@#E(P2G-9wl}Q7R#1#-o(WRQ6%eygM84~~WWOndkWfV_r@-sO6'
            '=ngFQH5OpeDIat($_f5aDjk+G;rmu+rYFwqOhlN|Z649Zyt9!_hAbr|LoO`(`f8j0y$L+9i?n=GXW7sCb>tj~GoI$6t@EhW=An0M'
            'tD=gt4&mvbJtfaOnvJ>fSk!r1ErvUw=n#)il)_6+o8j#7Ej73kMV@}n<X-BE6R4*Xx&vhi@0jk+sQRw;;@mN0Tq>J0Wle_1u$!9R'
            '%Uk|=afTY<>~cPPlH5_DmnKz#8w_2TB#gY6j~&9v*wG!^qJGkP=g9BI4GAuK{u!4&gWJR>&=`$dH$#J4$qdU}K@JJfYUtiCo;;bu'
            'M$@-MPh26cGw=fW4sif2xG>yz8`wiF!2i-&yw1D854~~PbL(``s=d<7N-Zp#kaQMWo*@uk;!Bnj%{|p~x~FP;ej%Qt?5)!rD{sNp'
            '#Dj_rme$$Iq&XCz3^r)OA`7;hExN|{K4S+U2>VRvFcfTN78JUqPD>E*Rz58YZn4o_mp~W{)>#Zf&y#Uhe!nRu#X%G+CCh6`m6W(V'
            'kvZd{>SIV&g`(mJ!&gPUhIx~ywO*XUSwu=^8*-o;%yb<l(ixcjZhP?5xCd(EOoA_FXZfX>=<orc5LpdLw<SdwwAT5?Y8yX!`VAMP'
            'A$Gx5(3&Zdq*v@A>+)PvZmv}VsUb{Kj99r2%N5`hBWYkk_w#}8DAoB6*#NI@M(k0}b;~JM(TdJzPrjy@$CIx?Vys|(KDRP5)B1om'
            'MMUtttYiItD?jmXkx4anB>Ylew9)f+g+JB#nv<sWi2y76zM|rRNCg1`GvT6G+j<}8Rzu5mioOaSoFdy>ZU_DX*<)3)q#cdo7;BtO'
            'lX`z}T(=M(Onz1<G-9<*Q*&n)J#`BI1h;LK#H+h_g=!HE$0)$&L(GAbn;1}7pf$uvZd1l08rIcj^4Fn9#$c&6zM~GhuIe<ijwqjm'
            '0s*2y*F>1G`gBqk>oQ$YUs4{ySmS}h2wty_Y{BWLzqdNE73TI@BJ(SIqi$&4QK>g$DB9cY4fX7pamAhggUz9ZzHHdOH}e8`7VANw'
            'cpyfY7#QNWkR()t?>7NzB;CYolLn1B&E^r>VC2Lg`%uM0go~N>&JnNzpE*xAxFb!n5*0!8h-xj))e+E}f~*DsOeus#CS!_J#3BRK'
            'OR7V}S<Y0+Y)u6KIrE5X;bD4lohvDG%xzL>zLqdcV|OQ=dBzH);XY<ng3+dHFbHtNy6G%SZq(X+_N>+ea`LiWtJ0x@jw%HOwZ>TP'
            'XDkur0=u)!PK#A)jp^`QaN*Rgnq@kjz-ai>+n-*&`LKNV>X+BAetq?F`7eLNI%uN65)hxC8EFA_QMymDW+jk#2b;D`!J^c-c^Ba9'
            'W<iDrY*SVft-Ziq_3gLl!|}ifjUD7jHbaY3=)ZsU<Es}~=F80E`I~<ehvIWuVd5zmy;ZyyDn4Sp5e!;`z^A-*06S<*3I%jD2;U;c'
            '#?s(m$!KoqfSRE(%0qs*aF=(eZ;H{8u}c$NnKTbaJ~7Fl4q4Zf7q{LRcTO#`XMtC}8SdpZ9>`%sU6hO?sR2eKuSA(E)0{kKBo;L$'
            '($M^{fIz48)5QdI#xkhGytu)Nlosd(^>+c;mw1W0pLn)bTnO}`7hg-6B$y$yM+Io<5<)YZv9+WQrl*({cTpy-3XYF?ijpz*hUcAU'
            'uG3mExd`Wc;h}Ij=y3jQl}UYi@0i}c1I)V9b?oe$^fP!^N3JAj-<mvZGZcz<&&4LwtgHBAC)|#%eo;W**Jwd_P^rg&_(ts{g*u`D'
            '=Hibo5^e#=7NLAYwB^6ZjH(X!YW%@FJU|s|4GMG!a)~TkHY(FC1L-31m+~&$_QUVv&*GfSy!10TKaUL%u-2*+D~$@+2zb<pP6HMP'
            'Ec+@HI_IHUSJ63Skq_tM+&SMg#sU?I!F=A(&KUZstyPWr4NhU(@a<B^L+8RUv(`x9@y3*IgZ2WL2@*_R0rr5Sy|N_r0CuP(Dt6)Q'
            'ss85BGDsOY-JfBE@#=Z7L0b(*-4L2glg|G3dQ6y3(Ta400NO43VDzC7zHiNdj^SE})6=l6U2MOxeta^U#_!?03VV4nao!F_v?LwH'
            '8#@Ym_yPiN%X$y`T3Z?B_OK6~gbarlJ9R6j9qH#@Q)?zt12zY{8pKu80RKW=ZK_{SK?z2t0QR)5)+pHvtbdL&x&@7W8cj|2(Y>Wn'
            'hW(x|fcpb(@u&x;79%kbjqnPfVpn@h4yBQodr!M-$c|yu(~aBE)Nb!QEYRg@M!oIk2eSas!uOONA%tffloe9$`x=^R{%c3n&bc;P'
            'i>1s>ZtM4S`^WZA*D*fjG@tfin=Ei(2B$=fPAKL*jwcfb)GUgMDi{pV_XI;pae8tUXA<w0pyL;BnyQ@Ep}X)338xCPyzzWIvH<zY'
            'a650|qNT7zE*1Qng8BP7t9K%iS81l>)}}tYcUBiF$Z@C@$5!)=dz8`%Ej0Fb_}Jgi58GGcfH&Q8W@w3TQkf-e;PU+Dt}(`MwNc*i'
            'h{uk+66Ta`(<Z8}rF?cv(C*HJ+^KZe*xD_(T36$y$G$<1!3(WYJi3pt5aPL$?(E^zM#=9E>#VCe@6%&ZhF*Xj)L&W}vu*SABH{HG'
            'EtHto`i3?Fi^jAT)IAe+_7wLwfs`a1jS84!4OV|tm;1uXSG{Aqu?w)e^&l=2TGz!6<gut0ueJ(=^Un8ltT?sF$|!w$5evIf>yQ%d'
            '(YnO)!v^f=`M@!|B)lDy1bsi7wYzc;JL*3?E~YV}e3>Nj8jViua{Oini$5J!TvC90@rNvfp8az|Leie<9=n&wtuP<b7JRBOpk`by'
            '?X~PBMyUQj1*RStwrfO8mv$OGB#zfVjCYXSxjX#vY@D<&5_4a0?IC06_*j$qpp1$-7Q5l-A$rIO47Fuum-#+wRoZM}$}s@~YvM&R'
            '&tyq<6c{CdU>G%8s(QcY8}O$WGnA<z0F64WIBo}TmJg-wlyL7PIo3ImDwE1puszW%AE%C}*54nUf}Z!rqB^2YJxkRQVj8-~+r4xS'
            'oJpd1f57SQ!i5D6C>*T%SL9{P5(@0<;X@)=51p|MQ2A$edds-Sa<0fSGB)Z>e{wVAP1=@DpK9ITy{FO;mm>Br_Kr<*xb|pT-&wHv'
            'p_CSQ|F;!vVg2^CWP053rG*CoG&84fLjk=QTmLMueG>RDNN;@*7^*2$efW(9>M<4X*(wCPBF=8&BMZ5+;Dc0WoIR0py9TR?K6qpE'
            'QXa-!jX>9PuZoE=HXn!Ir)mmy_GvZ$tls)4T_LP-J8CuV2*!<D;k966QH?MghNq26<F@Y01#Tww8;c0fi}g1&M(x7L-A9FH@+c!0'
            'x6a6%vo+%H&^VqGo^&a^GmNI(T-?&&e<_1cCLxAVe~#97r*gDL_XgS@j<>m0j2QzUEzGf<YyEa%+{)2y5N_uL|C#tQ^geXmGLD79'
            'v&&_{JEoByoG3&Yz(p2*^@0=?^qEp2Ty7e+8F<<0Bs+YVZOPOPx1*gd3vbTsy^BwLc^)J0U6<>-1Nq$_uIrBH^UeR}yF|@R4@Zh%'
            '>62%ex&=)e=S9O;^r=XY2At+%jz~CHKz|L<MWIWD;6==i$4;+CyI{@f=03MOHKR;81gigVVX$sN0Q_55l(BWb;Ie@Ut_J**w!W^@'
            '7p@LD5`(|NqcEn};unoX7sFvG&er2zWV+ZHfOkC(dip9A`uGpJTDzLobU1Mb)qW{pSz~6JjcrFACF;W>2ELr4DzNcN6<G~jU2HZ='
            'qGN$M)&8PA>qvudOXA0Y+$Qj|$tqSd@u7FDM8^Qp%{E<cS=Eyg=BkKXRreyGYe45mEh=7i4N+|P`=Rj-=eUoa`F1**wp=XNpnz@|'
            '7xu7c<~i0Rlo)fwG7h_GkmTQeWa4_iUdZ3QGKuBAS{9X4wvEN&#jw?gGEnJMp!I6xf7y*efZhA@)%VXo{`g_}>#NuQ`ojn3b83Iq'
            '1=D}P<x=}m&pHpjE`0xY$x8m@4Br~)G=!<d%!#6+eP%NBd-65izSJK)S@wT<rWd&B+Vy1``|`xMXa=Xtv-VcbZxto#uRb(*u=8T7'
            '61lnm`kN}=ZIFJDrl@<P0nPo&;6pnt-{J?*XA?@nA7lRARVF7oo74GFG?$3C38#nj0L~^jEK$xSX}JCmf1bwusPt&wd;$(d0<rH^'
            'UFt0EvGYM9`~>vaskX0pC6xV&jV>Y11-f;a8Rdq2#`eCBa)S|^FFtk)j6cMa8Z;$VQ>Akyb2eY@{|Yx{aVx>)&zjP&;@!tL`m>hL'
            '-#q{EZ|`5fe<tq#@;3wH?_A%=a>8|PaLFuAK%5n(Q%0@AUSt3J5F=eTn?4BctX<DX!Ly<`V2XP1iER3AIVT~T%1(Zc<`?2O!@2bJ'
            'T&yB%$~2kWch2sfawhxevCh38=@0Chu{5{?X4J#OEhR={{X48`tJ706`gSuLj?WCEf!dbT4LJuHZN0-{ON9oAtv<e{Z}5T(CRmpe'
            'E{HEkKl-+}^T}_j;s>^3nq^q=fj*o${WI=9>8}V|ack16oJXUu%|kCgIfmuJ8{@IHesFs<mx!D){U<xz#Mi@tdxeUZ5d&eAA+C>3'
            '^|x^G_JsLRnG-I&_nVo0pUZSVmC3(%nmgW?20M@?0Ep&$<dGeCM`9a)rYzZx)>^AT`_0h*6xrmqt;y~<Da)Cy|A8_uBBnpR)%rYJ'
            'P@5<GPTQhAY@;Idx4?Ww8GIA0OHq!4XgML*`fk=2^igj5h_+J@E#*31o4ATIa0jrWWxAfw&Up;GV4};gu{z<cw2jRc2Xh^-K=4OD'
            '*`DH3QMOhad#_rSJ5}M%lr2@MVVS~nB)^~wSfn5hIH#Pk)-^V$t`ipDeElBg3%&5>M3LLdbYC>ywOb-FzK+uj(Oqb@s<0?H0L=Nm'
            'iQsllK+VW8n6G51BPACoM_~-o0<Z$4h$7c0|3G`hS#ecMw5L(vP#W}99|W~AG2w*vC&zR&rmK^i=7|SLXCMFe{M~EqQ}1X^F}FM^'
            '@}w5guE=XJ%z$X>kXC}uGr-?SL1zEp6#d&3C1%9JZaNDLmNOM665}ygt`{itJ8rp0`Xe?wsrC|WkRyE`Jw3vlj`0ByUB|m;ob{MN'
            '@0})Y8#+_*f1Tk~Z6Mzf>4!xGbIv6{BB1G6=jLYZNV32nlU9fYN5t!98*@waAMX9`(!w_;-LKjBWtp-5D(dB>|5+DxfnU|Q<a1dX'
            'l@!RuW&7?b6%%3&KNj8}e65Q}+-3GUH`<(TcX~}|@<~1Hp&4y5BS7tfI%%B53Z*$qU@4}B-RaNNjGC{TjP(~%n%)=s>lc?}?;(By'
            '<8shh48p6D*VYYV?;|BZb6b{*o-`GyMy3d`YojzK6R?fcUm5_(3QL}@!52`E&u7YtBvZ%?L1<uf(+8ISg|=FZ)geC}3fwA<1IxGg'
            'hG_1@_$2@;4?=R?8a60SO!-5qC``VAamqp6*08-mvRb)8`dqxVMgC*)WM*V?bEA?zvy86K;wgzQuzE2GfJ%7+xHVl?Vj~GH9hkK#'
            'ZESO`FeN8hdSAN7N;T(WI=d#|T4GsO*8k)S*CRDDt>5S2sx2)Z_^zuKqphIG6=U1I9(UsI#kxBu_nn))<hy;CaCW-dBz@AP6YNBs'
            'pSp1C4T98lD_EYW0{Mmb9(MpT=%GO0HYw8$@J_{k2UZ)fB{Q4y@M>GsI-88*CH{?DjlI|aq^(y82$*pL#w6_l;5hv;7d8kuOzZ#('
            '3Y=QzM%DmZsm3AVby+CmWAfcr=IfsI=K4=h&AxXC@77HZNa^-)%6~RJdf8U@_R5BeFF~OEb<|Zqsa>5p{MMv8U5|O(RSu^w(#pG<'
            '{_jRJ7B1&BXfL_~<R?oSExO6#T9evB={zp%_yY0}oHe{D=G3kQVQ_R$uLUb);06%Nsfs}<9b>dRz(~dmJ9I5$)UDTa;6Rb3e;d|~'
            'XzqIJ&tkJ-;u!k(mJR8x7;;ZxkftyMD2piY&YU_Cb^@b^RT|;xw@;H!obz|xC+<GvUp&ZLa!ivy#DeBPW)Z|^9WTc7AQ$)N1uUG`'
            'gYHy-#l7z4;>0mUE9SfikB*G!qzwPI5@_8`z{7ZI9Wra!(iG~ChJgE(tR`enOpGeU^8BY&Yac(6fi&EtpJX!OrW3pE5$dS#1SvS('
            '@ybfI)Ky1KR_}Pv`;@O+|G3r7JM;4^##+JN(ak8A4pL(Yjnj*D=Zd`MbfU{IqXUYr7yW(hM{R?yXd82ql5H`%ABy*l+It~-t^a`2'
            '*`VMbFJFpB3o(2qetrJ#&FeS+>Ynk68@ysnE_{S1N~W-TMOj&+VbAf6cVVI<AvP$@(0|dvshqCw<iln`GnxUJTMQASnCek=k@EK#'
            'H>Gb-#`)0=7jRf0(o<}r1|tp;;AbRNXee;h8ckg?^Gmq_hd@xlr(!KRj1dD3GQ9UgW?0sL*}|!^8JW}p@1}!$Am``bVK<$`NmM*}'
            'l>jkSpq-8wItDj_E)LAa8P+VlJM%MX(Z{wIb-eL=?uS@CgiA#Ek+@0S4La*hh&yuzIHRBkSqzk^>E*ZhHn&NYFshf*dWf!^dpQ;l'
            'PjI3;bz^&t&O7-V`rmy|lTY2Xrp#pLtvZseYau?$?F6UrvIsYycWQuenH^$Y+yb(9m+d<vxc;ekbHvXz{C~*6M5+'
        ),
    },
    'render_utils': {
        'sha256': '92f838cddfc00ca9563f6a9a9e198d7c6fc5c6514382d719e4c719101c3b0289',
        'encoding': 'zlib+b85',
        'data': (
            'c-rlK+m_olmgswZ1yWfvloCpH@g;7#l`O}0IzH{#S+>)eUgbJ6B~c}|C{n{qmE{_(HGgnk=5x;1%nzJjIQs&C00^qej(d78?e5qV'
            '32fXpHufEdhlj7?x{Ri2EPnW5y@@YG5ohzb7;KVFoF-W;mT|g?i&B)!sEFrcx)sOwCtPTv3JsIZHk*px)iRka50VVul_JVS{LgiA'
            '8KrSnh2n9ZM$<t86*pKrtHdndY(>5h)iNnXnqMTdK2(~E%eY7u+ew<wqI9^~9xz;ytT%a4m0WzC&ui%PJSpN?m2Sl%FW^HgsFECn'
            'DA)Na7FAqU{o%pEiyUfZ7cinC7WF30qj@Qy+NPeSNx7sRZ>h*oJgY>W0oWqOLJV|~q*P`uu8IVJ9+<CdYA_P;CWYxM>&+%jV7~IY'
            '+SHZUMAdS5aCmrluqg7im`oORRTuGOB1Z*av%HF`B+tr&1Nm+lmCH1ls-J%;^GrP#v3e?(b(N(0S3TVn`7AC=^`=_l7+_G`@;s{I'
            'Dp|*>UG+N@`1glAi@6+j7}{3Feuke12e02edh^Tc$)6rQ|K;gxF&5Vc0{$Gv?*T)gc2-5{VJHsg@nxLm8+<X@q|q#1<|)h$ek$V2'
            'B)$Ut0mJ}wIt&>?w1Hk<;%{-7WM#crB(nsoL-!(lxrktv=`$3Z*E5``2@Y%vAa>BRU%r3y?9KD1lV5*)_WI|iuO>geeDd@;^)aZD'
            'Y#W3kDA%wy_^^)WNxjCWG%7CQ!R4I+EXFhs0MO${KfihQ^2OxMk57MkI{Eq4%O9RSH^v&w<3&`bRWOn-=q2<Cv0ws-TSCX+<4M?j'
            'UF0y3ELz9#K6n!UCHhk>UPoCOxMj-Z1D5&b)c6p=P%q=jEKl=d5|t&w_{SfQ;m;57#{@BrN>8~TesIgJqxX}myqIH=yUzO=Y;t^m'
            '=h*sCTqN1#65gMlnjZki<6;KQ!j~VSw2X~!X<Su6A<9iO1KfiGvf_^a2`*qhCi!9l=m|Ji-ZY=5UGw=<RLtTutxNIxap!~{|IH`#'
            '^ufnY=$oS6Kc7PrWK{q{&5C408w2KVy+Iu9=@L}xNffX1OuT{iqhcFpp?I~7;%r+)=DR;7Wn64a^VLt=GAfb?c8m@1$Gk3Mv*52l'
            'YvUp>L-C}DE`XR-u)3l6BlP)gP>Sb3J>jth=UIW>B2<9(qjU?6VNYV%pj$uyUApfsG2%%J7hXReyoxVss64EyMH?-a(doVWSmXH4'
            '@!h-8^!~lm(+AVzd-v|neiz-Foy7O=%onG>o8OJ^J&3+Jd2sKW`1IuT!SP8HExx(`V7{229N(YL7soBob&(?6WmRp;(a}*6T@5di'
            'YFSTVo{AZi0HQXW<?Ex1JikceBdk;&t)nD6$`>g-$PtvYqg#w7O=fWhNNP^P%O9Q(tM`?!*k|mVr<Exjfk2~yDyI=`F+4)BJ=i=k'
            'z){d7kiqSSsGzDbd<1m6kCO0wDgGF5;J9V><UJhHvjjPj^#%|e4%nk2Pt$2MTZx?Z(oj6At9%V@&cq*Hz7Yi+=*UE5`IUqRlUEY!'
            'vCe_7fCIQnsx%(p;Y56uNuh+KY$BS%aX0jm!U=5f4{`d*C(L86-a!lRKlt0z!f!q|Ej$^X?xPDlBZEI734;~6fhH{O&L7;F-?=+K'
            'S=^s}bNA%rcX#5u(QI*d3PfS?yL)#|;(O8kZ>IO7i0H=r_&EAzKK<r2x*H!a?%e&{G=b6KhzIcx_rAZnSXTdV@5$em7RXM0G8wQ2'
            ')wwbsxB0z&+7WyBheyE7z5D^Vx>qk>Jbf*>yFWht^RF*oJwfiS*K9?WNSG^ugK9}f9K8c^gAO%(U8yq<e_Y8^PqrB(c$_f|{5)@t'
            'Li$OkFg_LX^d%Sx%z3ehkwrb|9~>M2rwMF+nWwlzBR<%n59_#x+%vW~=iehyRt1b9h=5W7C|bSu)dc_2--5mvd<){ztQs-PjglV$'
            '2ccyiL(}UdOG+ejvN27oNFLCnlA1)ZNg)XeH3(7n3YvpP-Ckl9?qPojpOa0mKTPwhxajq%da^Jee4$1&(#T+a2U$rZnea2~Ba62v'
            'B4B5br>NtnMUfXaON4(Ghu<e4v94oSO+9!yv)8)KFN)nmap<YMjxT2@;n8e?><5)f5Sik6&)>cN&W6(0zvODqLlisVEP{#D%>#z0'
            'CE)pFo+t*cyBTkm7a-#Rp`TAA63DeMQGymCuqPk?z!^7N(&}YU%ATg)D&8VlCox++!vfr}uVBrL!tLG}+#LRSf$ZNwdtLyH?3}Z-'
            '?fGiS6`@!JvJazc31!#tllJ^|0aZdDEJBv^^iL$v3Be5H4gfJSGmmX&4*<z(7St@oO&Wz{ADl9^5_?Nx0r(+oaS8AVH+l#Dohj(&'
            'a{70Y*vK=GPLRo>PSYNeS)v<&Lp)3K<N|sRb0T#UL#`vy=n75{;G5!%Y9fJ~0^I>#YZ)OSZzdYiXtu1g72-t##HW`=>*+ii$&y3l'
            'o_hE1-Me$YABriUtG!6vjbXh(^+k^$_01)a-<R=we(EWL3dablT`YP!!Q-RHk1#i|&^&)YNh3g)$vi3A#Pg?Ug_Kj%NB|+C%P2{S'
            '=OfKfx;3;}Fv?PFk}QKA_WH+11DLeo0Rj7-4>`eS8O$N@oX;{*0&4?oBta+D8LGTuI)&o*z_`JYgi4Yq6@Q=;nqd8a2O(*sL{f@*'
            'ewCq?3)N8B3|?hbP5_=1aEK0&%Y{!;S4Lw6ZkH+FT+SwaLgfo8!Rk3`#9(D()U`#}M^x>>%Mw<8M#?p40nU?Tv4{&)@!`A-)tu44'
            'YR1OGobfY_yXRC&aWPO#ei&ZFRnOt=^vQwefqFYK=qc|<t|fT}A~MXGmAJIct1_>FQ%AO<+`<lB6JDTB2w`bdfF7YTtk>X%{e8Gz'
            '!QZd}aQx+%SXWpvI9ez9ihgp1IiO{dk;V@|HsFu+K$yV4-^4Q{AiU9<`8c!q)^qnPHNX#(lBTT3+ST?pH_GZ`L5#<4+c!)so--yS'
            'XOig{(sO1_H|>xFj|Kd$W*o-(7ifM`<awngeZsw-^oNp&0TaP9E&rjKifzP>@q{&ij+*)NbE@ZR22Ch8PN7-&c5}#j+ac<}Pp#^q'
            'xpvJV=<vdw<Klg?pK}h80WgnOr+715o9ue~EbD|84qKgRi6&tqn)I7~FhbyYt3YeSAlm5Tc%*t7q!~Pn<l%atKf^AD@4%<KTPzq}'
            'tKRH}icU6cK~-$+k<-a$w=#sEeAZbT5(TSs)~4rmHNJa~4nRQH4Tt~^uVHm@NLGc(bX&z`PnG9`(3m8-U~LT4a2oXcVh{S+?D4BQ'
            '!wxfp93&ac63z}EYx1~)<K&{K;hc`jL`)-82$so*!0uWznj$9Ef6t4|-Z{SYJexACG1aHXH%v8gvY&{hw_lz3{R}lsPw5||?$9R%'
            '8@{!>wJG3W?k$2}{`~0Gvk}jEnv_viiWTfMoNl!uXt@CS!T`5(7NFTgia(er^w!(SV1GS*ZS@h*tyxw&$x4?n*h^od0dUppAKZXw'
            '9%1TDypCuFYTyJvBt;s*QbhAa^sen~Mk{o_>%(l|EDSxO+2IdSu}TvH7l|@10bRvKJOj>X8*P#l;SL>HObNTQgM-eU<~=U=?KRKP'
            '7lAQSlrq#NTfv(GIDNE`VrSKo8;z<Z|GcL#KkOXE!(xF(;mH)h6-lhuowNtodMSEM5s<YOuu9VyfAojuFA(6iVj2^xr5UyyMk)UN'
            'U;mA#dL3QFNiJJt>ub48Hl(PPFR2^bkZUvW5IQapaiIm|Kw}~y2P_Q>X5>`$Br7Xm%dLI^GR*SznwrDT4aH-!;Fe-tgP>EFv0_G`'
            '0<J{_i!BY(3n*C%QWgW-3k(SI?BbnlZIHa{Tih}nV)Ysgv4rY&AJ_)3bt&>#v*+YMR#elCvOOQj)E!`s=@Up?AYIv2k;199jNiv{'
            'Ynpv%!=i6Cs&y>2lC4S*tC+(8*15nPgOXiah=fr-rLwSUWIg9qB6bZW!+1T7=V&Lw6-z+qX)u6cUKG)0iSX2H_JD>DweGzf{~lp|'
            '3n=(F&w%Pxyh>#nk(nPB3!1r#in&m)kgMbVmjo~pzXHw8uRw?*S&3E70yW^7=sh0ccpkrfJL>BtGqf$t1gZltc>5NQlA}gif%BzD'
            '>JRa)IH9KT*V)NAGK|5<P<lzA^!fN)d@TUjD1d+I&FSdeTpfHTmttBcDbf2X0YP+P%vuknNTLHF?38)69HCekie;Q!EGzkAn%2dn'
            'OfJ?Dy?`Fz+3GBzA^`rySrpL)Isk|sY{Qg25RNLKgek7;v_RPtmW4HK1Tu&2;_jR+BcL9*Wzfw5;H+VN&W=Zy=PbZRLXAuO`u<1d'
            'U;zShj8gO&RhXZ%(8+@h;Iu@8!K}zP0ja-mfCW%$#EOdo=}=%xsi<;7aT^!Jtlg4<>_cpENF4=4O5^c0&F^k>&2zgO0n*LlNnRuu'
            'K*v&GYB$8bBcUEA*$mY=ad_we#AaKd)F=-fZJF6^&lqpn`%RhvH+43>n3Tor{A|JhK6HDg3jEba8(!d8#)OdL>jpV@&;w7m7QSu`'
            'PWN_b9&?AFvUi|33+ITTX|x^pejQx(k4{Iu9|ueLw`(z4n&-vgv{5um-3nvqHU9HOu@k*%yaMtNXCq;t9q&kcG5d#oYqLl79Lc@0'
            ')0a_cj21tS#5B)S5HRc63bx?{_*^)3=JK_dUv5-xP=_ZASX5vqkxZab@Di96)RfD2a6nU`%>MAH)MK5zM|uHx8J&xs8jYBx$wttS'
            'xYyEGi<C|?paQTD+N1BYba8e<uyjZJOjgb!pd{x*g8(3x;h95s4*`B4L>}(=+MR$M1J+R8xxdAhY=9dxj>r#B9&Fv;8+hZdCO<@K'
            '`)A0{%;R%>fMxJ*3#_0E;$K9((V7knWI0CPvl5-Z#>d0sdn4Xz*9|7^Mp0D(MvvheP;VIA_|Eb185V&3)NW9IP(6y^FCZmA{>$;{'
            'XkPP3$8bcW&UZA2odZ)pA>WvUKAi$|V>yhy>Q4SgP5dWJx;1??@4t00+Gbd1lGQU<b9v%N#YH(n;j%h|-yl1G6(VbUL91n(Ula$D'
            'S13Zt?o7&fCU>;>7iQ+i^bhZE?R+?V3@89RQ#}8fUo`u^hD|SbJZ;CpRt%PaIY8R73<%;NMQ!$FoQfEUCh$7g>7jUq)U>pusK(}i'
            'N^R8X*XW{JRzO$_(W_u4W)bwEk8DPkXM+!Mk)tlQlB)M#mq3WWfBEJ|@lMUcJJAD@luygJ0BX<Bhch6!70CI-tG#Pa$vfyHGcgyy'
            'EE01}oH+0|SDAP>Xn@14$I%)&epDSW+X?$+9?u5=C(q{Ppv(u02pITMBIbsSi6Xj)P=2AIt}=uxIUm)Lp?HHV6M&9DaLj6yh1RgA'
            'U`~*fhEgV~$ngaP#5GLKNG$~jMitENGQR@32SghQJ0S{4{;m~{i+EM&08If?UdevN*#*oLG|qjS#tZa@;dMz<HQz9sPyvn`RCD4-'
            'P0^*f1E>W%%d$`e<AyZ^;pqYmav<peVia37k{FJy6~Tw<RSb<TqszEt`v_WJO4*MVZ>JUnTOlk4M8m0RFD-uYy#4)H95<wUwnnq*'
            'nJFFx1M)dFQJa1%?8YC2;PCJjV)*$%qYuhwC`mNC2@tslZG;NPCp_!YxPW}WVOVH}I2$rqL;OMo#qsDSjm#oYlD5+CfoKK$MO-`J'
            'Nby4Nc(<JmfN&p=&P{ccq^Vd++E9MeG?{%5^p?T`=!QNzp?X=4m|WVEO6vzL6W!{ySChNEK_#O-??02?k!Bu-i8@B}<FlWihLYH{'
            '-tXt7^ojuwRWHypYkbm|?qrgBq4^FLC?ne)84qz=u`~BbQcj0bCbaq4%ARf|xOly(w%AOC!nPE@0G|MrfDEjmPMqcS#gc^L1a=bc'
            'N}N#I>vL!=%~8K0g|si_L=gD``UpZ-el?U9Fs8o~=#6p4;EJCgNR!t>OlVZ5{;;J7HUSNz4Mq#}XmKR2Nv*5&W9Zanj<B~`{?8!8'
            'e*gUG@$re($R;XF*(;jnf((+<Retz}o_gry6IWyIGqBC%xMSMzxGMB3Dk`Z|v4(@%42q&iGNFe6(`OHO6~hWmLUD<r0`!AIDwL1R'
            '20+t!-wVb;%MJill`~8)ApQXM@}#D1p?|K+z@x84AB-fc<C~T8>jZ?5Qg0yYrdQ|P$U5E4HE+IDhYzA*DS{VAj{@`HSR@6UcO+a{'
            '(7qJ!jG?_lq~{*;0g7`W0ng)dg9igbLxu4}I(ZymFoh-HTv2SPT!~iaI!h|4S&Xsg((Mu+@rf+gL)#&NS)nzcHqn<*P1=z-nE++P'
            '|Axo*q$v_6*$a~DdULWxpY%h)za=z$ft?x$WfttEE^B}oUBu%>AOm`?5#(riyx1LG6Eu1jWhPenJOzB0Shc`&I5n2vMA+dhh57Bd'
            '#GOu-u`GENC6!S^ZH~X>I4RObb5BizBy(eHx1){=klK7j_CDC>jgMHziS|*anRAf&&oJ-$80+zo3)EhiW;OP&j66v9u-AM>HKRfU'
            'uR5EQT%Z)T%z~CZLxC%$1Fflb+yq%tNBuUOM(<=jfxOz4f+l}0@bIbLiU0Nb<qINy9QTBfJp4PZr=tg7L}|GdGM0-tJ9a+#_kaC2'
            '8Cq46W{_F`2%T=Udp4@$l`@_eet=@zTx;f`i*-6jw$j$;Gj0KMjVyV-poYko1q01ggqf=uFsE-H5-LqP9KmVL#;jq3U>-(K13NRC'
            'K;Guk<QW5N`kNQL0`xmy;6E5OMz|pL(&r6w?&l`M152;CpP%g5eRAY#`?);hskPrEnq!BD6r9&ng`tBAi+M&alSr8})T+0IT|YX1'
            '53<i7UM$eQLhCPNC$}C8j$lK=JGqrY!f9z}V)=~J)F^hSGlxwP&_Txp(<dGbZBj9`1Jahz(?eD*DI$5=CT0&RN>M9HucB>9Vr<z+'
            'wYWhlDgP-$BmtyoN95&D)Oq<vAi|J5S<=>m^&g6ts6NwDBRtoc4qqT=tzMS1XY&)uHje`4YY;GwiN#tO_lbQ>`-~H%9GBMj(Q05c'
            '=>|HHCgep2tj-$NLeB`158IA^9Vk>EiL>hfQE)b?wi|XJ0_h=VC+ay_L6fZd7wOf-I%MPBE)>JzP;G-iZWut#K*<DbC*{IBD)t5`'
            '0S2j}%(yAgh8(j|5Bj7HigN9#6Gz;xqMC>gMSKyz=c5*l%*%QmWlfj}8-x%?pq(ZHQU|tz96q3N9^n(IB99KV!zy`(9;pF{*)pE3'
            'qyP<w4IEp7o{}<d7r07SAZCiDH_#0PjSp2#&mK0Um*-&~v4uTaC)>QZJ-IXGLRTV8elro%EQdjZScs5<fD#GGrI(zqhh-cUvt_Sn'
            '5dUrYRqyTm>;9<uyZ7Db?GS%`*EddG1W60#h4-3w%$HDY1pl4=@3-0cSKs{!%a8bn1B&(gw+Hozl+`UzjT+h-tkQ#!!>Z9~^S#*v'
            '`Tln7mx~>FxsA4o$XE52@W<f5PVT7_K`t3{7bHW<o*UgWmFJ~*ZhGJS{)@NsKCII9X?Snff3Kf<-{A+C83W!BO)Hz;vEk7}n;C$j'
            'i;iwYmOJV9nI%<i6R(Q%qN__P;IDsrbA34hN({b{@X@~})srSPeT*B6IJ#{sR43T=oo&LACAzyUq-x*;-&K?-5@mH2!-JC4#!d~M'
            'm(mm>?8cA);7k(0zW7Sqxqo~tzNXog(DChke8<A7Y^pCiuYG2k(?mXnT}Of0h273#LzsVTF&=w3>#7F6{fxcKQ9Gj9lAp0gDc)hp'
            'cR;w24^hX;6{@MVb9pFo^eMT*{bqTfFp+J0p$zrm3v;)~C)_%De`oVPLN8+VQ>mS`{|6CBht&$a+ITj?0Tz%#T`7#z7@`XO!zy&0'
            'F$dGvDrHS1kSif}CIW;hOUGeGa!bHrAXy?GoeY)2k^tdYLezxO7vL){F5<$1Z}Rb89=@dl(atZO@{?TPu$i`QtyD`3MT(G!G;f8{'
            '#8HP$5k}7rDclZh_3DibvDrU+B_%q$48ko@I&6I2OkWcH(sU~E>nvc?dW}mUTnz64#faW_e8!DGJoZSdH8GiGi~$m&j6fvcDt1Vj'
            'P$Fr3$9YBfPnS6*hCAYYfGZTw18F#qM8L2DE(uMbYbKE48m-d!340_917-*FxIzJz5d!w#EUn42i9N`ae8hY2Dq@yxbi?-ROP-pz'
            'QpSEAiNW?F<RE9rVlnMtIR`htn2`vK>>i%PGVV``3b^Ue89YZnEVXad_H>SN+NFT0WX57`9g?QTKEvDOsFORDf(k{nLBA<~1W%co'
            'awtG=ZDy6{pCfcIEyqr(mT8)s*MY=Wo?02?v6Cc{PIL+=pnY**06q)I(UaX=1JqvI{{QWwAcB#S-BXyxL$A;((Zj$;io+J)isJ*5'
            '_4c-H*q7+D3U|7;b0?sS0`o9D<vJM$f8qn*`D8;dV#{WEf$9zehOzy!GMT2a@r~zLi7;*GyM8-lM6PZFL~f{&9A((-nUPvn84+Sp'
            'X2fQFiZN5~Igu^OCCQ*Ot!M5qa2-*0XqXW0#hHUsSWZ44+{&4HPkS^Ot%9wk-CxvN>h7YR<?b%rUjBAjXMR{kbuQU8n1RoW1&vIA'
            'D8Rn^jT-Fi+?2cw`Jp69nydrrEo{D$>l^$oMeRt&`rMGbJ{c5q&*uj>E-bo*THWpwow#w8v7Xh(zC~XwkPR|kg#xLd_4VxdoCm?;'
            'L_h5KETxNn8v`2LqOL81>#N=S>!lHKdjl5suB@mj7Qc{;F0J?6gT5ys=vG*62{NV-vshOaVrg~ibt6JS8UCn+gniabqZO7&pn9{d'
            'g7cOEv-bvYhdAocFsSuAq3XWtMKC!g8nk+GaP!XAfXS3xHPXM!si6ymE)35ut0sf^XrBYz2!rH+E$UtNY4=>>?vdvj85BzuN7JSe'
            '{BUa`aqy)We7-*b?X~nPM+IptN8GzbK0L)cts_hiaqy+~SqVUL`sKys)zcSGp1u+r*wr!cbf34|Z1W-Xos<QeOMwenZ<I*v^~`J}'
            'b6C$FaM)o>jy44PBF2Oub26R-7rf5rs$576Iu|y@{<0~lD~BMjtCDG&luBDd=CvMG8qgbj?4x{ta(sMjV+#<PMlUcrKArDI*QQdy'
            'ctud?cX-VWj*jnOP5{%Jb_NJJK{YL)GI^qwJ@p`c8<5dFX9<N8+bB=@yQegxO&KkkGyu*}{}2^GR{~@t!T|M^fmBS3-h%LD03t%U'
            'LEgUFf}lJUuhB76h9_iOA;Agq5R@S%x;O08x~|hI*`RV679U+^P&NSUMOg&~s6}D?(dgLdaHDV;p}XuxB*2F=?)ACyR4|>7$d?M!'
            'j~wjvIIv<NBn{!owCiDCwQh=MZp2;SALnQoBQBp_5_5(*TCjg3>d3CYzbBNlNJd%=$ph9H5CEMR1Q&Nk(+=RK-4hU8O%}7xRw4g0'
            'Ny)$H49nOBCRABXtAuo5yI}INa_f51(C&O-463Zx7)C9N9$S7WDh&q>)tO-IgqmUe-fw@V-ha*91{D!sixYQJ&Gz?}MJvCC`fcaR'
            'v#}&Xvw;Ks$efn8vkAdw!N8nT=!dnKUlH_nbuYphOQrEmH`Jh8AOiMz;ItGD=C#*cgYcKzhdgQ)Z?j!|z4z@FfA=MsMbMH?ay%@Y'
            'EIoW_6Gg4=Y-|K<8nkFkGJhW;L`2FsgS9OvnMX}W1@?1%VsJeKKn;uZ!1MQpp5FSrIK1|rd;nXcLVpLx?YCl2L7JHU?y#c~D#IkO'
            'qlEHKYcvfqiWqD4g14ERX{rQ!nLL4$>|z|$)nf1<P&RLs+;JLHT%8VcYhoCrP{2AlrK52YKpAJ%WP^bvb=;g;H#jT`c@|ADP9-HB'
            'h6b$Cs-{tPQAZc@gXJB56Z9lYDvVU-yp=}IDt&go(Nj<g->6w<=Nz0wdj@({`Q%KKiws@D;drE@UPle`z7a1oI$fo0T=b+4TBx+e'
            'vn`JXg^C(82~re_h^yJK??i;sIYQY2ttFweSy@E&i99gWW3cf`5uyKBDSg-Eh=Y>#9P@_t23C*Up&KPQY#-?(P$<2>+*ER;6OQ#Y'
            'Vizr+MKn(HOit8H%6QaY`R7lF1Wl3!!U<R}^J}k-FqqfkGMUBWV793#P1$-=<7NPHXB@zsJP52Kml%%;CI<?|gmjl<Zfe-<2Rm2>'
            'se%tVU~5sTCXERyEz4yzml0EAJH?<p&gSHJIiZZI(D%G9X#vMlX`wKFtR6x!1v!~UJHBT_vr&B9vwx$Sm%=D~>95pWAoVoZpP@s`'
            'ht?@n4vFnCG!^EIp1aIio6+mE{@Z(OnwLzUR+xMi{U8A|=XKi+{6@aoL!ZcEVYN4){fp&;+v(ig<O+80m&u37&1fNdlu@An(B7U}'
            '^S!E5Snni_X3=VBjI%@ONCxSSVES&YL4(Gu=bU_`qSLz#I$CIHDD1}DiV#=(f!KRuxwr1OdU=~h04;XBXA@4DxlZ!u`1362aoS_C'
            'j;kyY+kDkGSZB%fpvrwL?pxp|lFY)UqSnrmQ3C8?Q{$YO6x=%n2B7pn7FG-`p4Lstz7e~Dh)aph#=c=5Y4+Xdg2Tm2S<ITTX4@yg'
            'v+73l=I>$GPx{Wc4vx-_s>3mcO)|EI5i@(JtDL*rnopL%yHx@||1#fZ(K?x_By45xHq6Tn9E3_?_{j{;ZP}b#f%y%r&}|taHd(Sz'
            '*PLr&gL#JG9UI9BmvDvUl<Vx4tgpgI2W3f&85DByobmt}5X=}gp!dt<Vws}Km2O9PX*Cc5o5a{4?TAs?$|<U;C)pzjq^9H)>Xm7i'
            '0NdR6--tE&Z^@9Hw`jKUN@|jLksTJq$t)&6wdGc&ex&cr^Yk^{vp=Fc%MJTWvq!RuM>k~ZN;GI593D95-qc4r_XifF()zG+v|}DE'
            'nUF2==**ZzdK*jUUCZC@#z-E<I^qzSZ{WFV+UbYpE1WxDAF=N5#|+8ey412z*S{8Uia%&)^}^ZHOY4QA*PGS8vgdc2rRMj<<KF35'
            'WmDlNt<w=^T&T9XzA<9sGfp+~FQK^W0ln-02Xf2u`2Qqs+3luB(<e`DR3pig+u^SzeB%svsuIwy+9YuQMiw29mURMrcAVlpHTzlg'
            'dHq4We)UG&$el|GSY=1+RH<0A)ybH6X5Yq}_8Bww96JX-rokZI4vn^d`Y3a2d3nGp!EaQSx3mLE<ti*<;+TX&!#HD`hS)IEztKe0'
            'hRxQZmc7VR*D@NZ3TV*@h11z~g%ef2u^CY(ccW|Bq_h$tJbD~nn?UJRV9q6HjOuAt)t*BYwgsmGDeK*gQ2+1${-3~jd5!OS=9?Y9'
            '>6>rfQrdW-HG||YtFsx?)@1plZKRT~B|qGKOtuVKcaHb@;hmb-(71o5MI5ar7tzLb?Yn=kF_JYMZA@i<z;P*Eo#ZUYqE#;Qf?Q|W'
            '6LzzNL$QgWQ0(n2!t+M4knB{9JEEYR#H>X(6ww`)5!9}86*oQ^_%QO8l;^${u+D(7Su3S5i&mJ}3@zOZ2n`S<ZmL4({r%1YT$VM_'
            'Vjy2iU}wkYjW*i?12gVGP{mp>1}EnVz7w1(`Ahg#qKiO<nRU=yVOEqDz(NHpV`0?9_Ikk(G4%`l>pT2wgnvB@oLre4kvAbc(YG*I'
            'L=offxy8P;6ZKA*1Zkx1w;2}<yT+Q<N}*c})_>5{Cq(zx*GuuOG=<+&{F1vF{Pl4SzEx)Q6IIvR5&nw)neVtVK;o%6k8LiEH$Y>z'
            ';OX>Oy_)@T;Jj=b2YtI^PeYOzatX_XBrl;Mfe47IkACq9jJaykXp0%cC+`(?IyruDypQMCHMY9O?@rOHK_wB@!nOZqw)P)=n~nnb'
            '%_$N{zQ#u9%RAlMD-S{T65c%^+`D`9IJ_75cxbbAqmNa)b*)ePkG`FUbB8I%s(n@LTYJpYz~jdKE24&Y^!hb=6r-rgQ3@1dK%v2m'
            'TbLtsldqIY0W-VCDP^P}Ku;qKszCtg7ad2Ga4_aW#5yd<RfqG;u*HS%u%vvCbs_SF<YYOo48kB0f<3UK%RHHLwL`Ue<(UO!ciZSr'
            ';`yP77mIkt;mi4=(bTM>BBHx0qYF+(dBG|e@i^K%uB!nBO0M%uG$!%Na}O+C32!c#(oVL(U<Jnd$w0kvLljN-WCrX8dl{MSf+vz`'
            '#}!LpeHHiq0XX+e%%k-MGVt4cr6RkQoIk`=bOvXyS(W=NTjb~FJwW^({o+RA^6v2X_-o_qBKR5h_^Y`5r;#8BpN2^EcfN9ecFS$8'
            'a?o0o#`EuCYU1KiHh;a0=K0kKFw45J@z)~SyhhKQkvKKqo6l!|y8E@DLZvmHA4r)#5-%dWcoQ#e$n+EIqkb+DASv6&gnyxUO&qB4'
            ';$>YSf~ud5loossGKzZreV&52{0!LbWEP5FkPXIK01wXVRSZDj5l8z^^4~whzq~&~@dy(#*`R4kXgs<HP=V$L$}6!i#4FE$8a`JK'
            'uhhezLh)4ExUGee^Z1Am`&Vcsyw>!27t&&Bs@sPbHtm8JE-~DNfdQT=BK`^g@yBl-$j8A+cufBTxq$w=<|EpsQ~lA!4$Fl1;J;G~'
            'EF3)We|9sI<R3bB(uoXDiIRB{;d!)03FS3wJ)WYHLrycJ$i4|Bf)a{1vA*+)Pj5*$Etd->3@8u15BZ0!^XHA;w-eaN;>J1cspQ@i'
            'Ae~!yN{y-=9BZ=RX3&fweoxGs4=h`{-&YeF+{A8$Dx7IBUCc2MriD2XZj<Flw^^P!;Lf{Q^Y(>99q_F%P(_Q~Vuq7$8|atf1vv!('
            'LW}Aux1}RE+9=LssC45>3Ig1}grnR84O4h?3e=4#=|D@j4~?r|{(>QQM*be`En}ev7|7-Mr95mG5i&gL=p1ljf!VBH*D15$Weg`g'
            '-uH#~;0zD^8c58&<3TqmJF#XmY)|0;Kb(A`?$fJz^)IohgzfI2qpag7wTVZyZpyA|L*H&|Y6GaFwbsr|Lt^fF6q~I1JdRCnm2In>'
            'qlKgqV@pS(TlCI!GYa(brw9Lp+f$+>jZUs}5nNxteLHKP5C8b~?et`~yVl*^1vdh5P8|utU#~q%KUQ>)N%zd5AB4KD;=u;tsnI=E'
            'i^C^Ll;#&`EQQm|)e(}VVApz&=7L@5yAIDd#O3)KUN@cg-hFqIf1KnMzm`MkQyz5GgtZ;!{MWEaei+j@1tXKvqjiD$W`2{cblgQs'
            'K@9o@LTFt_>g6NsyVi_LB|Hi8mmht>+Tu2Mt1WK2R&CjRquSEEOzmuRa{t`9DME%ou)?{w%&W;dXKNqc;jV5}E8~;{c=)4!O|@i5'
            '6kQ;I2ThU8W1sE&=LDm9<sA%|o(2_zalB#`p-Py7+m#YUkF2YH72r_E4uLpxWi%b~%P5P|ZJCr41Yv&QtA5G8K}i`8w7M+5;>)6%'
            'c>CEhFXK$sHX@SNl%os#tUPk*S!tC*bwilC(YvoB?tg&szR0T|Q2(xzXk}zBT*#zZQs-6jtz$xl{mhecB?bddYPII08sy7a8b<{)'
            'cR6LG=46tgbSR8uWU-lCi@C{YS|pTR*Kj}3Dp$Fl>$r*#l^tY|F?)RT2)XgdC|-LZ@N8<}a3#Bh-0<zC@>Ok@YO;*L5>QOj*hNsY'
            's}o&LwhI`4i(0aj>MQHCCpWg)D(C>Xp>d~}%JGMaQM@TZC7T2i(Fow4fricG0L5sHDO7V{P)lFe!}hodoZHLL9iD`HK5}iFNlCZ4'
            'WI$_ZBq0XZS5nQ|@856-?BWn4>Lg82)e?!|n&xB|0Qa&@$|!}ccafwcaZRAVD0V{=hliM<49<ZM^-3T^3UG?yp`|eCsnz+`n7=*x'
            'lf|Juv1s7rr(IvP7Nr7ixOc68?=WrD7CJ+{c6;LD%eS7T*xNG~K=0al1MRn0ocq{%-|nmGZ1)jB`Y_@Vy36!^SJv&Lz72)&K*NJI'
            'x>EPeVTvA`FzF82??0sz?%Crs>>+Ks@%Fb*>45K?vuUN#9WSXd&%U5<td?E)%r1ETNlUgLBsqQ13*TLVjN7dP)Mkl;7j1qZ^K%0k'
            'CeT3MBF~Z!csH9^0mGKiNn5FWk3b#f$xzr*La+zfabuLLI7_kw`VZ%Ew2eUI9tMWFze$GbOwgF|$H?r|#bPLz9o&?fieoIBdTGRV'
            'Yc_-;y7MCW3#DvdMcXV|afrBZw*nAPV=Z^$OuAkeWNV;lTzPKpHP4`Ie7xQ8TC}HixWI5*g*t}q-pYX;N<PR1@A<IfLxd#oy2XZ2'
            '5#vzbl>uJ$wn_@Tk}fG>=A3&k0%s<bS=>Wi2QAvHrnZr`>`k$61}Xga)8}jvwD(5O+^}krfOo_V*31DQ<cc}^KfnB=i%Nl&$?H=&'
            'Y_qS%#ToWpFWN6iM|Ss&oMwIrhu=+@Qc)Umes7ki-WpD-b`3ok5~$`$p`Z2LUBZ9AQv;JTmm67`l&MsGF;X)ro9KaM`-e5+=A?a('
            '9aQy%ETpNTxge~muM4?W(cc}1=Eqw&YwOY<;hC-L*lyUAilfQ{J9Q){b|dqHDrDHB*_!C7LU&g?aP>#Ww%p<WZIy1O@`loWv+oU+'
            'e74>jsyG&EyR=oxuvy;-(?o}k!naWcIG5J=a#oYu80B-9EQ9j><K4!Z6wILe^3}6HJbUrzIXn9i_oBT4%{*I4sl;m_X?Xb@`jrh7'
            'U1%;Z)F$1zRI8B=q8y56tOlRwF)QsUyERbA=g)sa5%|rU*ElL1cQGV~e7EOx=@s5rSy!AJKa?4W<K}`09z;?a4vtvA334=I1=w$&'
            '?}e60=MGGLBsIug(bu#ve3|Z-ee0wSzxCO+O><r{4quX2uh`eH=wu_LWSP9E)AVQgxhFXua{nJHkAKDxb@WO}Uei$0Vw5_;jVd<O'
            'neJSw+v}vqVT;xyLH_4DDdLGFmA-`8ZeR8aSwAsW!eN@M6T0`c;C$IGl${cpG%E}Xy`uO7yVp_&VKdBBSn2}QfTmjo;#zUlRO)=B'
            '_6jAv&XpIxI~iwz@g_Lm+2dCol$TUEjqaHfhv6c5J*H5s)%e<iy=x2S=z;*O<2GNp7X!mo$CtQ+ux1(+IK_0qQqSpHf9Kv2=FFv@'
            'b6w@k>el&zv76|^X_<4bdDC}F4A(38yT@1AtC$ed;!X0EaqkFkCUoP!h)MpUjN%C;dJq1P00l~c`pNscGt9xP^go{%qxi{OmZau3'
            'W9U~IcXC!Ot?@p6vAGH9ke;B8&h^tYGvAs@+CJx)Zui>IZ&3lC9IQz0rV7;MG`FaQ*G{S+N?k)5)+}xVT~+F|GUm_v+_Qd#_TX)B'
            '@+NJ0`G(s7rE$aWflj%+hYM31`}Hs1zj^lN`P0d-KR$c?^V3(8pI$zB`uz2+H+X=TR@2>=d_AT<u#|b4nxs_G-JSWO*gbSa?M2Ys'
            'sHt{(M_>E9pmi%C-9V4Bj>}U@!)g0HE+g@SI`l`~5#C7cd!Q<Aa%ARKW2|wS_G{G}B-G_KZh1STQ^;R-471(7j1B8%xF<T!!cneh'
            'LD|<$${^ipjPHL@YmIX=;X)L8Y^n{0cso1gk?s}2Sv2@?etn8A^n2#JKf2u<>;Z9`SqPN%saq}8wJ{Fc-C+a~m+>|d$tn@Re=pKR'
            'OyPu4n*|O4yhElIIZOzAbI-tDUS#oVgt1k4WK?iIJVd!Zi?Z=`t3}vW?pfHZpCl+4GdPyl%Ue{&Ma!mq3oj3>b<k3jL8qE`XymjT'
            'It%pI;M@aBMuDTdR!j8i28UmjpM!>H3Sma?T-nSE)h*=qwUN~f$hH9Ig=-gap=xo?nStiL52pUwnt>+KSQ{R{?}A5PJ@8mGY=M)n'
            '{m}Sh3s~c{GG=qV7bP2`e#p+teFuzfz}L<@USN(7KuBKA_NhzggqaL+qr@5B%W0qBRHElQl1*ihI1+@JWzgt>Erm^!gn=&VK@kPo'
            '4AswBvCz+SMK;nWXrqWQmJfQ4@jl1Hdmc*l5dce4n~0MK@Sn`<AcvqXRKUQ9Q;x(il~v*^3szqo9f?zTd1~>>`dWoH6#1ze%t7P*'
            'ZkUVh3D%#7>&@N3<he10V=qwBh?E3YahaqBo9(@VH#A+H2Xu2KGlhzI?wi+~*i30KsLbMuzczhT2J2{pwETE<IXb2rQ}l~ybo`y^'
            'bvl@h=q{-=nZ7^0f0qjLqT(ZF@Rq0mB@4Pl>XadFlJ}Fvn!ar7%e$w?8wYHJpQ30cJ8RFf&D+v9o!q31KQ_aO1DpM3TDNtRfot27'
            '8i{=gdO28KUeB8~-SZ?E?%(=`a8E?ALLF&6S-}V@)b*4iS|#iw|4=9!%Yfde)i(N&AO~Ed)+i~Xb@XAV_P?DXWh`zjoH)m>DN<?;'
            '7_R3_W$-DH1<G`QuI!h@c`Gj1$u!jtxNYSCIk4kW8as*;ciXK#ZZOdke7i?zXzJ+*+-_blJm|=u;A3zcg%BzIFjwAQ{BUPmtv)8o'
            'gCnC#(3;ZRA*WZ0GT3>ES)<N`B23Ke<b9zBiiH(?0V2oZQKmZMkuN<|xZnu@YVJL}*41`;SS1j$B0+Zs8)bHVh2KDrDpa^mE{bTg'
            ')UomUHRsEqk3GXq5?TjNXrCPJ+@W*h{<%{aJ4EJ+6}dJFD`zM3J{-%yz_g}qTp*RR6MEVK+aEA@&`PxsdWL4NOcNtBHcww8!frdZ'
            '4%G0V6$6Ky_PEWihfRdsxXppva1;wSZZU#pq*9xUb)l&6r8a+Rmvou$v6rQ{>)4F-*u$vX<uzY9ZZo7##J|Nfjyo7nZw8in%Bw2f'
            'tMSz2Cw_}0miI-kjR{7|vYeC$N#TlE#HOmmU$RW#sW-{H4CEs!14YKcbw|%Ml>RO_q{QmFJLaXYt&uaCP}E>?ct%$Q&(Cpng6r1Q'
            '?1F2zzq=9MKZtXVqV(BiURURb?It+q85#K0rZ+byZ9pB|XI>iNnAksMv1~f5!AMHtPIrKjW3O??UAHW>yU+?I8#fW2p%q0oHo~M_'
            '9hwpOHNK;&jZZ@vg=Dm<_V4Z^7f+Aezl4BJ>;s2AkR|BHF1W?=>2q)I@->%Q@M=tEbj$90gz!*lj_uPZbcS@UcrxM!`AS0n!NMgd'
            '6162`NG6O~U*NzN!IK0p`d&rzBKnK?S|6TOyh2-10*nHA3>xAYou70Jv9%0240IGHAO6Pc6)7K%on8;|uimXin52Wthfnqd;t;GC'
            '9<ha2u_d?MT5NG-Qzg034J2Al?SHHg1D(k)7in}sjPTbe7Rkiz1i7`$O2jb?)V!vl%&21K4H(<9oGIU?=od5FN(Wzaag>V|E!=?z'
            'Yoviw8iO7utBUX>OUPNZ0FlFE7n033b<uF@+zED5qRX(2^?P|R!vV&*T4H=M^yC~FbKZ5+MG55jP@LeM;wG!z8qArR6q+=?<^q1u'
            'tY-NYT~7W1*tTI^&Cn|`XUfs*1c$dd)qFctAXP8YAUqFP!{4eRFS9t8M;oRWhM)Tzm`K~uxkc~;xtc4W??sMRVj~Txl84e0?=sf('
            'A<4=J1LtNj3JsV+DPFOj!gQy;skcqbsA49y^E_HcAentgBZXUhH7$y(vR}24t|?FFb#P$#F4`!&23%aktZ|T@8|oEmw?8C>jjhoJ'
            '9VIY_kjjXMYDgpAr{z(_HRv*t4m&xos8k53(xEx&vnBa`81ppIH<av|LKXMJqB^d+-R(0{XQT3gndhA1c05BIVlJ7I!}3s+%va~e'
            'S;GVPJP1iAW0M{&aT1amIn<>8Z<A-O-*0EsF>%ds+wTD2IPx0-%@LfvXjXB$x3zYMOdnUTd+ky5;o#M|SM4vD`R%|BNVKQGzMPdy'
            'gZzdov`8>9xa=9xrj9<Dz5YFFS?|$wnXl!nIsJ3Q>d!fjj+i;QBSsc}h3<DnqDHQzFE`W}KoWqRnAZkrf|H0z1`v~rd|d;9c<3ir'
            'ab^>&k0n}GZ!CS9OoFN?m^>EO(HuH~9yq9GF}PS`?6Cbnro%hxY4_WyI)fyG^ABgvy{OOe0Oh-9d+uu|17YBG50!Pj-&_D*AE1%#'
            'pA&lh(;ZI_B6POzhqs#HH-YW6UK$q!8q=moUVGvi8VKJ!l+o!$i^?JryYKvcpuK$2?tyaiQ_~Q1_zAn2g91Y#)f^L^nKLaoc;l7v'
            '(<y1HYcSWPslD-*)K@|8(dPsvpVoBL2i4-?nxbB)d$g{I7`M^M+HkaKT^r=72c5?GTfkwRi$d+oPl6&t8AqKy09_DlCGw<ZZh~x<'
            'K83Qovn~B1-tl7o?kKD5ip~7_oq(EIrT2=F>4c}}u?w7>-CILrDx=SrJz%Yp`yorCEozU;#Dc5M%5IS^fQ`4o$m6uIuad<7o0Ow='
            'B}w)7sp(_{*50NSrgp(mQMnyJTHXsIaSIh{;P+!ND|cyAgO;a)S*t@K9JJgK+)5tZaNs%}*ma~@+F!*?5O9;$S66D~4Y*MOtauso'
            '=pl_qap_@1_z(cDqBgHuwLe>G6DVV=QM_eQu`)7^|K5|dIT2fjX`5PliR7$~t+WhmZ?enq9R)@qc{zjAU{RS&lSj$wR8X4oRt2p>'
            'H@za8*T&7yn#*>X6TWT=mxL6*e0-Xd!6lq(ex)T|d+Gh)j>oVV(H=t<Mwy?`>&~wd?HYs6I%RaleC>9mLxe-VuH%yfp;O{Q1nBzK'
            'Vi1){xiN~597D4gsadtv#%LCijJyP+z3M@4J(&^Y<UUq5vV*jNkg3FddkfSvG=m<Eg`dqc6p=cIdizr-PW%2DTh{AHsb74oy`h*<'
            'H$kgclj&Ao+it#$D)rI^gnl`tr#v}8<H1K$+Nx_Y|G+_1s)^ev2<@aUs55tNg1tEr3gchv^<|#SVsvVkU%C@(wgL-X*zQ{m+G7{?'
            'BqsW}2*mEl!|ryvLlXCeBF^jC3QosKOleEz_Sf#loa2-Dj!WPnF<P%lNqXm=d54nw9re@QEnvr(?ld(X><^vX8#3A5(yd}CZx>P5'
            '-N+50c0a<R4bu1$bBf?ad1*X=eIW4#*o3OsicOBa(KjYj)xEbSA0JP)K-}tSOdj9sNHS)4*#H*<4&*(M@0q+1*NXJh*?Ek>xlmo+'
            '9GA_(%@Ji-7q)(SVBJlnF7_*8vQU;7A*YOg;%hOb1V0U=uf;lfkIE$uz~h`d7|zDl61u}0Hiyz8r(11b;rKZw<spq;UE5~|<V}=L'
            'oI{+Du;KV=HDsV7=lf&L6nWro>S>ykOYB#+ttd-h&s1JfOp&}TurGl;lEhlKFyfp+i#j76lX(2-#fz735HA4JF((`<0>=$0$1Foh'
            '1-p6`^WtQ=*4hX_@0z^Q#3PvfxLm?a=zv2uU|>pJzQ3@g%O-F=bYGMDYQ9fMPpdF2SXcrTqSJU50lNjgnC9~>`s;}+3=)9$5)R4S'
            'g771L22{wYA}q8qm?b4nD>?OWv;YLWqWp=MNko<3r8!y!G!m<k$_GGTz?lV1W3T@%lu4`PczESs|M+MCN0Ul2jIVeeIJFi0q(c;a'
            'S875WbkTv%KYH;*M1)tktiCBQE_bRo(zQ`91W;DP4xw&QQ{xVWJ|&uW?2e`04=_WSpm+;85{A*Iz|yix(p1ElK!=h<F<#Hmk6<op'
            'TyLH|JzY&_-iagQeM@uq4k?=4@D%qJTSg&hIo==&s@vCj_Ta@VqD+!Hjq}wqq70g__0S|Sr|YoHnAI=E%Z<D)JD-w}RdC#}G+gzM'
            'q-R`z2y6vk?OxJ~pn-;V8>MoND+x}ujNQ_iqNnjP!l&XP2N=T;^EBp5?=b^3?!gsa1KvdFeyPYS%s8gT#F3F1iDm`+hPw1Bj^>nP'
            '8ZozA=2z4qQErk9DB<w4h2{R2k8or^3DnwaPIUsCJcIc%vTm9ot&Hi8>F^8FbZ8{3@&j(}Dt4g~c~?~RdT*P%K=f3ToKmTO*l+b#'
            'Z`se?;RbL=pF3LiciTRH&C7kqonvVV1(u(!7Yy21;Txf-_kv7cMxw5)K^T%;rZ6E%0UK|gms`4`h?4%Sqd7(@`r5!`VyzHvX_%2W'
            '$@y2@E^DRD^BH*B$&w6jb|VVBoz#h|w=NuOg{k@RRz_aC{T))16p($`9e!>^o^#OoC&ina!}zUX=iV_oG3Z8nf2~mT;ctjVH@YW7'
            '(&_%9by9My<RXbNOMC@~PJxE`G37&UrYHgyY9BUH{eI}p4aVP0gLGtGZ#VVdYbTA-jX$38GMbhu%X4#?&9skqyXX-d7zn1v$xaLl'
            '!m9<{?dQvO*h<=neKzK*ug`)O`2FeqJa`n$qg6sFA1D!_c~=5pL*hXtU<>MPU9M|5??x(JgVBYCwEWi#*#CPh+utab?T<|$l9(kc'
            'z`VbQvi)ZcWxGWb+sy%Ne2B1m5cS>a7R1sDAhQ^drZ6)fpF7O#QHH6A*64ksJajnT$-u^2Mwd0$Ir><z&^z#bp=Ey$wfYR9R=+g{'
            'q9gv{uNeFA8{!^pjQOj@J1|n)tHtlkrE1fXmi9a}wASGs9eG<T-v4b86QMbV|KRY6ugN*_Hw95t$DcoVLVfazsOBN{Iif7iB7XCq'
            'G|a-vpY*p0x2TT)9%k`BILty${Shz6pt)#ThhHQde<6C@__#tX=0l*y;@OMWk6wtEFCP8*nRxQ_56_-IdG^Y>QH|6_!0V+T7Jooj'
            '=Z(m-XbV__X?rl}^0NnO+%TTMV%UajmL~f_>!Ooea-YapGR5hs%U}BKD@s^d{);kx*ftMjl_T-|(VM5QwDn-9Js-3O0n#Nvw^9Pb'
            'EM!wMB%(cNTGA~hky6N$CO#Kwo?SGS6U@#TrwhDNmP|F2Ml?+&zzWR}(wK%ODayEs7m!eTPH8c5EFO-vqbF@05+YkX@U9otoLZN@'
            'Hi9VQ2is{{CXgFLY~+F%{u}$1<6UtzjWdD|RF~%ZW@w026R8%NXdfS2y>|;tGb|u>AUURq?dqjeD>YcoJ@(<hH@x6KXRMK_Uf<IP'
            '<02v_Fv%pw?srLpv6vf<(GaAEL>z&S+hC%Uc??^O74S*g#A?`To|}TBZs<(U=M;tc%A<9P+XW+9TK$#Lcuf4Cy?Ffe`O}}CzJ4aY'
            'fBMJg&;I=Km3tXLj{{+9Is)dvNnrZJJS!p|ZnP~pxf^U}kd8yqX<;S>A)p~%NYeMkU!&ugix#%()i(`aV=DKdc!{0Kvl#~FfRssR'
            ')E218DlhAl)z>ZHPO|pj5MJcz*xxE3$LC(9JVia9I;-1$SS|$n>&M3Q*qMC)M$tXZ<hQMf!Klm5_+KwNs5{8#a|QN%zF?k@3*d3I'
            '1n3v(=neMzrW=X_sJch1;P`SnDh8<QzWOLP_TS>V-tDd4<)MBXfAjx7PQ|~$PxzyplK)3K70*)iC9<4uwBO~6e2{OnUy}BT%A_9>'
            'UoU`*C~u($G|g-DdZSDFv>ljbITTMqv5L2$)**v{g}$C-ofX63P}g7$HP>KGQx(sp!X*@6DK&L)3s7W$Vo0(q;kZ-!{{x#$%uN'
        ),
    },
    'google_ai': {
        'sha256': '6a2cc8b2916cf3485b8fb85ca1b4b4ad217c00dec1f962cecdc219a2c7bed49e',
        'encoding': 'zlib+b85',
        'data': (
            'c-ri}d4JnDmMHqaKLwg~-&m}YWM`>#x6|r+DqBfu;w59tRbB1!P?SX3Y)Pb=r0ghe-_QQe0)PM)SxF`L&D`JAZ>A%Y00<l$oDFAj'
            '9OuxF;wT*Z^QOrBbk_3k{Ui`?<M?JCh;Cmr-Uf>>3dNp)9}C|VNxaH}L`>sEWVc}|mP!1NV4Ssgc7FNkS26ghe<4oJjxLXTqW8D{'
            '#o%J6Rr%k}j{yu3@ED7DvVzYAcR{iiQJe*%IG%~oDx7B`%u*5GM`CyrM1I&loQFY_HQMdAI~0C2*}3(T$vB<_6ETEgf_Z26hGyJ)'
            '(SFsM&i(XucPPSW9!5c13~v2QEc{6T!-{aRjMFq6!I!gGG+=GBD8AphzYWK?g2qZ=dV!cGL7<i?#(pIHakld3^R-9<e|*~%jS+MW'
            'f~9WV%EBPoIe&k$H-P^?e}4pPncM^!yo?uO6ok<YJV_#$^k^+;#Bn%*b@_8K^`jPmxQUAsjj;<4!Z|EJ_&duan1tiZhZ)1`)g+{!'
            ';@)55%5pe{Jnys9o><QPF#_4o4t92gc-9tUm~<AT!WUx#rD&u9Jj+^PGzpeLgwtpW^>e(2=eY1-;zIxDZQKhUo*)DuxCzq?w(UDH'
            'j-zRKvw}(HKoI|2#hDMoP5sq8OJRr?Z7~VcaSVH%0yLNYD4d7TG!=KDFNQp+HiN%Chp`*H(EtL#JD6)mpI?akFuR5E;6DIh@L@R*'
            '$FQleFY!HggI6!xVi}HSxYkJkoe?e4EF3cqizQ4CaUNQJ-=+=3mhd!%Rs@CK#h`!E9rTXmGUfs7Vgl`7wPE2|g0tQeBOkvJPp9x3'
            'p3x}dFd464IwOGgjJo`>E$;kzI3eHz#8!Yr2r8J$a+Qg64ZA<i@FW6abx)5(8Z7)sK7jszYKwV%lgdK^=y(ShlL$Y{g2ggJ=)|-w'
            'ev*Y#7zw)kxh+yZ0^oiORKFle;NXt<?BN-}1B{S`i6naMlO6GFzl}hRQ$m$ASo#Ufyr}^VlZ^ZYbQ{M&7GQf2nv!hjU0v_&oXXP-'
            '$hk^0Jlp`e7972Kut4ndqcwnzXFdX?KuiwSbT*UBqz*t}e(s(3-}GTOx^KJv(~E&Pd-JA$*zX>Tw`XT>f#LwYIyyVNJn5Ydx`Y1N'
            'DdKh@&U@XXlb&$i`RmA^iAj8utmgj0cibA<^!fkow8RM=i~doHkO}z2-)7k|J=og=`gKDfYEOc@JxQnhWw<wq$LSv9JPc7d1VH!#'
            '1il9c4DqtTbPh;hQ^-fD{tHfEhlpbn5TGznuowjsJ&GowwTF(#a{%MO^j5fAu-!cW77=5St`b1C0FQNJ%Qc2KxD>YyAEn6-fgG;^'
            'A@StReIRp4;b1(#MxX}SdKskcZx3bDc!Z?rwZ$}+eu_AH)IS_th%`o|PBVX<N)$p0xg%(qi+DU^f(euZjvXQ0#nC%BJJ4eehr5ly'
            'hry#6P-t%sM`k`m+8iu_=s38Ng+F)1zk~xshW>>U5Uv*V^ESM>bwmTsL5l$C;tp&F74=E@{NdU8dGC;r_bzSI6cy_k0lk?Xzy#sI'
            'BSHZA7&hV0c<C+Gj~O#X+<N$zs10oRD#~~}PtOK${1$M|CXAUle)oNU55UkCXVWPhZQwySE9e;k;{Y-NV@VO<1u>QwsGxrbn32x{'
            'h!C-9N#%jd?Ar_pzLF5hfx%Kd`)hI9gVQW7dZ$MhOwLiDP#<c4Q2dt;@X$k1VB(nv0r7z#%<LNYa0uJQoQVLnn-a$W5~H>o5e0}N'
            '@LnI7>O2jQ%O%0s&%!urZwGt1bI;q~vux1<Di;h9FbLW?5l{FKFl(Wx;YHvlFvA;QI<O<~b!MDf9vFGvQW}I7%s4K;2nPnI$%6-z'
            'RuXVaW^tdkC?XFy!`Aa59QlC1kgk?^ju54gS$cRRhp&lo(!oJS2JlOPGN{?W_{fs7;6t`OWzb(wUE%@!1=ycy_yJZ61Ho3s2?~1h'
            'Gm6vEGy<Pa?;9<6n9^6Yg26%DP)A_78o^fG=0o8OUh7|C83XD7s>4TtrCeknYQ;oCfKGtmFb@}q?l_v`FM%Hh%>E=?o5bbt0h}z5'
            'E7lJ6MIj6ZbZ|l>PNV{NE5p;X2qzJck4(Jx#~~a-k%f!E*&cX!Dq1`(fL;$-J<^mLSRZa22;{wE@4WXG1oHD=hf+Y~BMFD$@a*(W'
            '|Lx^@_ebENN4+=Q%i{qo0wo3#PFHY#65!^*g6Y^Eh}Ci)`;)Y(A&Eaxj$mN{<;f=en*cE(;Uj+tr#y)fis{ZM$nJw6Qs)HdEE1}8'
            'PlAzX%_Ihi3%V1)+B=Tp>=51Zyy*%_kLM{-!H*&w7g2F%M?M?*Y4GZ$`s)_X>pUE(KmU=&k@_8{>Q@3=6)*H(f%?UWuKvR9;fWxH'
            '#lO@vCL|@Pr%h@J$364;=f^mb9hZKFvr+xt!=F1l--s_MjC>=00_6y>86>u~NP!NHVW;SHr2(NY%q$!cO$Cz7#JWXn4@C(;cGwQW'
            'W*BO^is+z2zZ((;;7uA6R*UI8TrLsaV0DOGKFIQ{Sf%kQ0X{Om^`jfsLrA3f2K5HO{uBpHK~N4zu>xY>{=zWW`Q|{J6JnswWn*0T'
            '@bK*Nba1iv{`~CUdni8u#>q2TQej|O^aBSZ>3(qsKtRqR`b3UF0br$V$c)B|Vo&0<M?JknA}9)1aQIdzw*ZlW0h3iE!YK;F2tfoV'
            'aIdT_4iRoRb2`BnNa;Z8Cos=J@dAuOdZg%i1GqH|Huq6M=k3P6uqaS~!s!%FJdib5?9ct9-nomDo`PD;+Eogily^s|_{aj<JKf{s'
            'vtK+3D(^oo&jwxZ{7lN%9q|SkDM2|Wz3FPS2x)IqCEdYVN!OK-hbaNxgAPNoK}e)?vcfY@f||d^0ihdcPt@&p1v&=&br%NrNXVZi'
            'i>U5tT8N`*Zea?K_dwPHR6`Te<>(cldk82C{YJ1`i}uc&^Ip#z^n2&tQ5O~u45WY9gQYh}`*ftxg0y~xG%gK4ikY9mgb#ox0#{DL'
            'Ip7?OnS?VKRsh+`=+PRLv$?PJCWi;iK-GeM0SOG6uA({g6E^|M7Gbgofa{KdYt2^kIdq?sqLtvVfE&=`CjJO&yBg{y08|Zd2w{vG'
            '&sLG_*LV_u1Nib(jd$zM<{=Q$g#fCy_Sa!?$-Z3jB3Oq}+u3pX0Q&c$F^-q(1eshgaTQ%a9X8(ciNB&mLqZ`l7seMUQoaV1pVKzZ'
            'W7d!ntSH51K(Ny>Xx%910qMR+jXVWGfmHKFkoiE1eIj-#>Npag&%3{PC%vP7*BhL9#s)L<%klaG5dHGP`?-63*}H(p4-5y;g5moM'
            've-CH;Y^Tb#!GzQk;&yYp2H0B6(g;e1pi#YR!p2Gj{tH5qW=V&z|O5m^v3QuU3`dUAAT**1{^03&%+vqc!Zf9{?t3^dH}EsK8eqE'
            'c6P=flco}#y8ZV=_<N}GH_m|!!G!4#_kic%IOl$d2lM_GP{tRxC2oP6au>!(Q6_C<zB@axBjRZSCt80Mr$DCkm)^i~QrNxK1U3&z'
            '9e3OV@kpp{;@)RFg#hhxMEV$YKP2T$Na_S?4q6O`!G~ZBbjoA&^@wNzWhJf@#knnFJ}2G+)C3KGG$2~j$h=5PN7MjS9f@kjMmaqp'
            ')C=SSxZ_l8qKCX%{4YVkR&IoDLgQ{4tushg0g%Bk!4p08KhAx%Dq!GnfLT&XLwP#&pm5StJn2A8{aZNqK*^&CP@M#b2S}fLQZ!~c'
            '7Rd{cFF#%QGciL>R~Rk4oul1~iOgul+y<D?+E4u%kfb*7SK=3Ie;$P&$G+(PdfMk^M5`u2aY*83aD;D&rUgWb+Ibrw8ed!zx*+)z'
            'grA@N`6^)H{XT(H2-{6!4B+9IWJtsmX%lOrR=PkF+XN+77zTyLmKJQ;d@F6YUqf=~eL#+N2nTQ^S3p)bR&#-;f_y$FngH@Cl01;Y'
            '=c|+~lJ{_!TVqhf0F%)Mf-G<zf<O$7Lgb}sJYO-A1AZm#or~V#d2ir#kNe#Vlh(=e1w6qy5Dm%h-hoyFq>1fr9`HWzUGG=NmBeig'
            '2(cOkM9Xx?U-==ngPy;hT@EgP>@iC%_6K$LpZu}&v;WX<o9gu#$q1<5ApR@*6Hci8;qlq!(VOG$c~76I{!w*DP-?s^o&`~<<K^Hd'
            ';0o`0r+G*JBkCIIG6=@E9%^*hr~CIyXnyg&*E{^lo}kqnE#z@j?t$vq8Gi%Fx#T<^?7>m*bkOe}dmyO2=^vw{;Yc><?5Ky{Y4@ZD'
            'L%_=<XyY~mw`e5Je_nN4fA?D-_gjCy&VPFC*7d`F^VN&T|K-5pIy^h>{^-g5{;7BTzHYPiiF6nR0cAjMi2~qlvrto)hCEv?+IDV$'
            '9w8wOXF#oh3G99+ehq-W+=L(JCZTd<?_F%vog9c=jaa*)aTlz85v2H$3B0keY}ajyqu?e;4$R?lfiNX?T}d?nnmt$zvKfDp;N*~)'
            '`twP6gIs_o(KbigRVHfz#~fA^?y6=a<{h4Ne+5Y|T7kF&=N;spC7i2u>eG2djBnor8E`;jNCps$`v%Q4#0U1&SGwad=t1-hWll{9'
            'H=U*8zQ_&Gpv;e236@po4k2pHRst6_z!)KjhsFRWeg!@-fDecxl&w(my?dd7^dvCHAO^?t1wbud>Z2B34nasl#mb+b2U9IRbeUkG'
            '#K)EdVB9fq*gK@=iodgxvC5R`LSL$Z$dTU0_mPw*hDoqYKvdWpf>uxeGMQ(+3}pZkH;Krq3lwSlX-ZR0AZ^hs6_C;({{7<YRPzSD'
            'q)Sot8tWA)4l@t@8Q<8Tc||HFO)?q^yyX#iTS>MOy4phfGZJbb{HafBGzdF4w@A3d$WPXaOM$YNi!1cQ^U`^orJjfCiZcV`N7_q#'
            'z(Z*1zk<}m>ntU2PJx=>Pmkb>Gy_LN(a-bX6l9*)NP{`(yEx<l8b_KUAC(@6e~|##6xR4n-m#`7572+|tK7FE>gl1IOIbLFHckOL'
            ';6L_@d_wR#|D&RjnjN&uK>G+9)>GH+AtwrL)qS3q@31G|HsrsBz6?+93je|eBa9fj|1Wd_ts+}%Smzh21Z@(k;VjJ!F_W<sA2y5+'
            'xOF4a+kCKmoAMz6chLHpfXc=vtu4;*Fbt&)o*4w+7~z3|gS7%VnlyFPpW8}SHs)tcEypA}Pu5&Y>vD1kI4FD5IL0+|fJS_WYz@(k'
            'lMkK69vLRb9^NM9-@w1Dm0qaR-PFghmHJaOjImNl_MO8RhzKZY;`B^WAGEvRL9bfvgDvdRsXNGV)y92?z@XkUOx!{fbs!Hc1S~W%'
            '4)sA1|Ern;c1zPf^`J|?@`*5}@@NtPq@a=r=-m_I8#u-O&B8xG1&prvMGLRQ?*LLoC^e?eUo2WrC%E&V;q=HDBA-aeJQh?5{BNoN'
            'B%8{=9{J46hdD4F|ElO2P{%AG+Nc&}E|H0<rgR|9jX!g##BK;wQh)ZOJm)bsc_1(VIyF$VC;BHzoZ>}kJBaSW1W*wr3v8tZ+JXY{'
            'fYu2Eoc&q5BKT~>AETWYRU1NYMdLuY2<FUJi6i1A-NKJnz{+4f0U0Iq5m2-ZnW`E9+B=R5y9bA<;VK;(d0<hJZBJKo5DK#KZG&l8'
            'QJE&itxAVB_^bNtv(~GZAekkgpu)zmuz~8UR#9*L1ojl(`-`U7ZSTVWc)Nl`rQOyp8S56oJT!GuV@D(<qb-nfi2}Ra(uTnp&5h81'
            'r3XZfDi~q@pM@PAB-6By)e_P<&m#qS0jQXgUU)JfnuBN+-k@y*gbuWH@GhcYMEi(0SUfQ)BhmG(w)wL_#aD}nGq8cTeBnjlMwy74'
            '$h#fkPZ99)-i8Jw&y%(os`cZ=|1y=xXCqvyCYk0cuM~zzSCu@RST&l0D%4#ib%0nnkJ*Ht$;*lhz_)y{rgcYy;uBcA#Ac9f>AqyF'
            'UxS*B8|>o5^}y6L%5gyNzJKSVEtkZKlxT7TkI~GfBZtDG&;#R$509=}e?F!TnpC1ium)YHd^`@sgPPGFlE>E|SKf_R89ddqcyxGl'
            'gV%QStglq(Yigq)-9ZztYTk4aG6qzPS4@zIceAvWzq4Sf2Bw`~Hws|(<v8$`8Y>z@0Q1Sj$Ne`ElBr`tLas1m3p=WbAJO_IZUP_&'
            '@K~awKS6tZ63l=;@Pzq<;?|ib7N=HC>8TUV6Q^j*HJ@shtPd>Ops1_l71SX|0ag-3Pw5KPeM#6<4?`&q+D#>evuF$vGbzY7b1M={'
            '@(L|F3Xm41tQyBs3)G+GQEc*BCaa%E9qH;UzCpJ-wVz6zqC0)sknt^udrDa>^JtfD(3M}LXs{AKH1&V<$FceLkSvIFU<6ReOiW@R'
            'eSyo?$NWl)=_ua{r(&q}ZurCyr-;Ig=n4aoh^i54r;JpQSVd`&?akxyELWKE(w}@k$w9@zLo-Elgd=@<{VYGu`u3lE852LfwJBwO'
            '=54DX2`IQnRR{TCOM<1gWtFUeCgEs!HX9>2a6uGANWGXs+TV5`-Tyg`pL6)sQPUF47_`QThrbcq8|p`eI8%}<yORldduXePV(fIo'
            '^#Q%L&}R=X*^@is%C~ft?T__cA9+T!e<x4QyKOQhloU?2RC<o{fX||5hK7L!U*H}z*E9<jk8RCK9_3S7@uRvkV~uiT&7PH-{K0{}'
            'WhLnD8vMAS?WOLn*b}=_3pSoNg>5*6DZufWgzl~+ZKrg`$4a}!tso>S-Ff9RG_>zB@Emh`4@M_L`Enf_f_mByD1*2E5r<Ku*i$Q`'
            'k9r9Tss=D`gZa`!pR=4v*Z@?gqG<irM5|e}r;@FkEym&e1|<V!<P=DZ+RoRwsp1F+_R~k%yHVkb{~EQ<+i-$2V^a)~t<egN?`g^$'
            '&i&#yQs(t?j;SZSg;iI~=$tEes>*y+i}_8tDi19(ak6IiuFt`bI2O_w%Ntq7Gmv)`#y%KmBy}2pX6Q#K2@I{T4$+=386O!$G#eDl'
            'IG)S!5q6$q;w0VpeT;+fQ#8>60SjC5R!Vvf>D%P|elhrfBvAUqAor)?qy)e|uTR~v4_zgo*r00o%Mh=q*bCeeRM6QGB#%_yki(Sc'
            '=>NPI1=)R^%!Hgq7~PSjy+j5I5_k^ZW`S-MIsIV0tkm2qZ~|sHBca>24H5HCL<29rm!Q4htv;^y5ESVe01LD=Bk5ofe01AIrCrYa'
            'DI#Xv*q3p8$2K@A2eSA5oFc5_SzA0*5+vi(Bpf|fOwry&b*Et4W+ery*>K}4hAcqGfkCW%4f(i;Z{**A8FT+*h_--OB1jaFvWnPY'
            'l1?iGtNydn$fZPgb=c{j4htX^$Gd6lka^xUY_@<&nPHJzk=<eYjdo50^=%*)#-MmnH=l&B<+Y}|2}Q@Mg^w;)=m@?ThhnzkSd)+K'
            '8dK6{vRDF?{pZ(UKPOgL7qAd`HcXa_;odN#|4o(?4koB_8@?bpnxX)uSK@BU0d(jYsC-FRQHbGR6zw)fr;H1b2GM#Hq{}#>kfR8Y'
            '5*iu)*V60y9yBx1f7pqhV!*x;Fs05uxfU-M@B{IL-H7Jl4OxSvTI8>?c;V{^urJOZyX0r@4SIh=7c!?9xZ|h>7lZT5!@(uUqu4AD'
            '2J<)=Ot*V`es+0!#H}QXuno@B(FD5fV1PawPL`$mQSanO?rR19=-1u1APwVd{|2#b?5W@ZC0wBg!?^Kn69XR6nX`%TZHf&42SDBw'
            'cBo9+X}C?;(FKr_j>E2mj%naDz2-XPU`GfMq#ehBk1WGZ8DzF8{K4=a;5j)5>dKD)HZCj-anyr}Lra`~<?D=IXLYY(vDDkRNO1~l'
            '*YhYmH|N8kftLKkcw_k!9`Jph`OF-JAm$t}<c#O+8vy@nJY$i8oh{*|BhRE~tWCL=a4)=vx1$BLZwQ?g(LE3t=+ye>%EtgdDW={9'
            'QUF3x{;6XkJyY!<U6>uCTW%8N4ed0HQ`Hou6Qm!h!(zY+$-(!8u$|}M?|b|E`-Hfi=lgk^5)SJ)uO^3iHFcC%QwDj-FKBNiwUqDw'
            'yDx>e_(!?LAO)@#ff=hxQsOq0uo5m5EA`cY^Q;Ow-|v5&oqqNuR`h><0ldG52l6JTrNYL94jWV9{?FUr8xd<nMqE+7zt8^*;{8To'
            ';D3_s)P?BL1%M0`juvd3#hAT{ki522i6hUy4AvKMWdCb{q9FJlUH-jyJUHncAZjeaQuHE4I+8P;`1R~v6a9kMD<kqi52R4cGa?f+'
            'bWirt2#L~iXo|)a=z4S@P_}6-*^?ekI(YXu4rf^?3f}du*oGdd(+jdvOOCq51`RHnE=fS07rY72j6HZ}?13fbpbvMAY~xYfgHu9E'
            'wNQfeYq?Nd@<%_9g3(KU6943IxZ3|~R>_zF*qKL-ex4+6!ORGNeHv0ETq{^T+m{FPg$XVCnGb-_u_f^*4xx^>1?}o$(0g~>J3Z<B'
            'T~Z+<UKZ5Q7~mx^cq~B&Tor*^IE#xQi_kfD7Y9BN?RH!I3;%Z)ZDF%;Jx2$Lk40zv3cyn;7}1EAaNcF0cFcjac`%eKwgddaZ@p~2'
            'cs6Q<*_KpN;im-ovsVVlUwSX`x1oeA{4(w-J=06ax52`<>_@N3k5fS~q;x8<Ssec0uM-McU>KDS9<Am1t5M5;`QLs$ywh($qX6CT'
            'Ch_l3+P9RFrihoAHile->bG0E#OI@EinRZI>-qlkS5Je9G*tzsR|-!17G*L{nu3<`O{ZBwu!>;B%Y}6c+|B=oFA+O@4M$=bqN#!@'
            ')Qz9uuyDqR@7?)H>aRnP62!ai`|feS5{iEhlDNgjU^*vZY9$pqkmeTjY|+6eXkp2dt;oUfzuw;tk1x?E&`_?(!H4mDMWM79d#myS'
            'xlQepv%>H#(W;l<U?N{ldXw8FH)_8Gbr2nN5TS}K+NVgQ*CJ+n&5VSGwJ*9C$6axQVUQnd_#Se1808cAdY$04dGZzP?=#l(<WiW0'
            'gN%O@d>S=*3hKZyR#CR%U{nzBzTB({{QsS1oQq2T)o1-kL1Ba2pd#N@yDp87OfUtcp?^B)onQ10F`*S(P7jC}y%xR{>)B_JY_Tf&'
            '(})t~_}cnT@sirYig&tgX)&T$#^e6QK)cb~mLzpCaO<aD6h|L{DDs^aCp7Xx<I*;FbH}ZK0qvpa%2xHVZh5eI>2iRBc}<6OXmwSz'
            '4_=GE!d6h4kMipYzm`h~Q%~EHIr7v~w&snCp-2`6H3r6GN~J2pG&|d(108`~?=i3k%nWub6U-oz5g0e5Uek^ga^-4vco|qQ^x?|u'
            '_vUNdH4HGt3#~X6jrZ{FK)i6<;tZWURTjG`tR5Xa+4U1qHC<82)O0l$(=bUhit)t|&%xRIi#&k=XOUU>$XCTOT;ff>a<7{DQHlrS'
            'i{AhM3mo*kF8Z2u2mPNZO+eY!*{m=cHtP&#&61ex%Fq1h{djr&&i+J2*9wzWvr=4D;~tt94~?>&r>Pnq$y<lUX5G5^gjTAC>**Vd'
            'MwLpYKl?^sT{JezmC#{l(7kx)Q8y!`^)}8HF$L>uDWSM=l0|S7h{`<0ahx?xF<y>D>+$-@VWst)bmOeP*=RMNdE|6~NrCL|^XAKj'
            'cVXhgAw_ZmK*%<<R2dPfmSi=|6Dz<B0zX<UtJ{?fJi1wqMrug1Kp3bapXaKqel1a<SyCR#pOM(0xa0dEX*E#6Z8EKMwM|TrO{^`t'
            'Bg_OLI)B9*ZrfmnM2FKv5Z(-a>Yu*rpT6~udp~3Pl}!lrP<5qYnB6LF3H00gAb?sUI@$FeH(=@4na`FJsyqu{X+~R4sd(hR!fdvi'
            'NcEw9sPAkT`RgtqbeqS`1mO#RrJ9(0;g3`jqA&c7DkAn!b-|%_QcB_|zkq;;GDn{Dl>fx=<VeYSnEF&_4VGbdlE$3({^PO_3a7`)'
            '#YP%q>L2wdzuc<wIF_3gk4U*iHSU(%)u3OwVKoMo+f^b#xmndd7OWE%FEXs&6|Bgz^ls!tmYvE>=Wx>fn<u?+rIIul^iJNB)bvs+'
            '6E+LNxWfh;Xvy8WtWFf8b}768g?kp54PC$<xk<X5QY}bMR>_-FOqgo=A)*mX;j2o33d%@>Ex)LM7`#({5)<@52Un{c%u+j|ikFfG'
            '$87rH(XH3#XpE9Vr#<Jl&<;6V%G7}*tDmYLtPfJJl3PbpG+Y_1#G%?>-rqMT94aw3X478yQ8>jo5vdqG|MKp<Ig+0C-}Ej99{R*%'
            'IHuuml#?}2&8Uq@aGY}Q!Qj~Y>Fn|xar^npw$hcoX&B5Wjd=8rCPnf!m7{;bsrM)xXDwFfsE3%x@ocpu-!072%Lx}!m4G2k%rKf%'
            'I>X{1;!XejVxa65<2VH(6OIq848uBa4x;jq9T(H#-mnb}VnIJ<!8#psuuGOD;RuyfO6V}WL4uYg4Q+L&Ku)pCgY3FPaZe%fDaH(B'
            'lqR0gj#c&%QYcJJb{kJr=2DdbTT>r#7odY*$paT)P~p(X!%L{DriP;~fu4d9OP%1A4l33g<Jvx-yxQ{qX-CfpBc(APlE;URqDB(`'
            'z7)SOESRPeNwzqr(L9_5Vwaojm6G^tszTAA9fel#8DRAI%;8W=+!*+@W$npi+*UhX0wy<{J;%LjJ-gOn$-EAnm8U}>l}{$m$M#<R'
            'c1s6VJLdo_;IMmocG^Ge9*e`X6F4|W2b^`0vhCK`@7NH>;j4MQg&!E-72$u_jb|7@!F_iR#8o&+ui*iO-R`O%9=adzKDObUC`#kc'
            '(Geaez%e119+wg`Dw`k^d$u#ttp#6Xv|8FMcz-Kp9?a90khCKm$z{fVvNm$gUS-M>jY2|+GM&*L?uT&7R)`_Mb8<O>=W!s4NJaTa'
            'b>csqlYNY=P%fQSrc>juT-<4MlNrfmtXGXgF^14Vi_*b+aN-#V4$=$K;Wfa6E7&xL_R1lS-+<1pdgRGm5=O|or?Ou>mKUek2`x{?'
            '%so_})&<ASE_7-z^#8X2`{>pzfiFMu735zvR>J4QUIjjT+^Gtes|A{UpIr+<yP1Zj3`$cdN6!(y)}B^%Rx_58@47UCohG9q-ldG)'
            'pG8qN>cPf;`5s96C5KB^5jr14oVme62Fl*1D&yt0A1Dmsp(KaiMrD1ZS*<m=+kPpE@3$wWBMuC*6E_)}nHFJ`w4kUUj9s8s!@{M;'
            'kiMV9OYt$k&6Um%t-s1>`Na~g_ml%YNLra;-X>a!ce+}vUZu!pbV{o7&d{~u5J$pPBuLdx`ltO9SZ}(BXEL>=e}l1mnK(N={#C~%'
            'EfdggleHw^AY5Rb8kMr0Dle2b&j^0P`QP8ha~=#f40yziN@kS<b9@_&XF8|}&nG2N_t(2ArkRvk+)+lOVb#nqRt;VCDVsa*706mP'
            'jfPDJjKu$imMWQc8~$#nB--s&Hr9$l3Ej&=QZ8C|wqf*71YT^5yw;J)k8+8Hz{#tgs!3z3znK%h?J0Z-UXOLhaSH;zQ(`s5F!U;}'
            ')C{j(89SceW_85XqH0>Wsf+Rz)wSXl^A#%jfOp>ZqsiPG8%NCVy$XYwpOKnr#Z{Th!|}=9JVJGE>E_X&8c4v2fW<uuX|m?S9aYX@'
            '1g5iiz!jO4%4nP|IU){1l{!H8z#BEr5h`TOAY}*G<x*QZYz8x-!>LCSMrytv$>Gv01lltSIIZ3u=v5KsmWiiR&Rho~`^^m}8dyab'
            'ECs^4j3?9!^VD+y6a_pO7z0RuZJ7bez4G%?3GU*R^k$GVhI%8+n1HA5QKtkj?@Lu;7SNZB-E_sj5Ks-ChHIv$t1?bpsS&SrJM-)L'
            'Yi$Y4Q-!H~U0xpi!g;&szFIxJF3K>aIGF0|@A_U@b+2r5FOHh2zDqCkD#WR9AM0VUM>+h)fJE_S2_iZYhuh0POM;^vmBVeVhs7Rs'
            'P65z+8<&CNG;<OL1DKd-$k=ud6&%EK<-e{=Ioza7Vj());@JHFzF^#&vQ4C684uViSK0OoHtZ`L>)OnGNb$!YLCiu*xZEK%=PGYf'
            'JZUtIVkQg<utq(Nfv23_Jbt5r*woW&O487*ARn$ZcsNz#mjs96D%zMcI+_%$mD(-L;^ew7uUmC}V2)_)DdmaB?_-!5{V>%n+ARiZ'
            'nJ6$caLMTOsZ;sU<03wBTLube3FM6_kQtsWHsD{yQ6zXJ6{0CifR|Gnt`HlMo?BTtIB{bX%ek-=;*{`IGc<G64m-KxPNw{MI~BeW'
            '{-&zr;S^ZPGrnP*F3-;>pzgjMrm@NeYsUIzbMF@D5U!=(>{&5%j&FuyoIkAWp(HMY9rB~=`sPX^EB4{%X2)Fg#6-eI>~<k%W+{JH'
            'ZLoixC`^0&pweHOnWdX~)=Os<VbRUVF(pt(b617U1#a$E?_!k+!i=%378{{`-K(nJx%gmq%%2*4Oyo8MbSWqp4Rs4s5-{-(0|A%}'
            ';Ax{DcBHDQ&nao$jFqj*Ac5cqNKvH)^IV{j&qEI~&N<EXP|(s+RR_iBmmhb#JVvpFr%7EQ-eDbNK@`mNu@A)>_Hb?(4ZkCA>^YXU'
            'OTsw@W>r-zkWSf)Ii~FPQ6q%`lh-J}&ck9vIh9u#`8RXuF~%|n<=6*FjgU!H*iUjXPEPx(v>`~gv8%EM*{^LVY^Q%#Ym+wpRrBen'
            'A6()$?NsXj&FcIFs;^G!ur2eI++iaRuDZ=cT}?>>l`2~~OR&ZdAYG}<*}0AEO<doat@>g?psq-Dq;Pc|&qU*)FpSD_RIe#2;i0^k'
            'N(A2*jiPffN6&rCzf^=8!@c|g%c{#!4O}f`ExB}shQ|DQ7Nq!EhDG`qyJz{9$_rA&6NNS&h`m5h(%Xi5BJ-YVyB;8=;;sjvH^zP@'
            '(Lp9iD|paKkZQ?5rhn9iJ-WoTIlH^T9~i!k-pXnQbTOX)k>?v>2-siArGn-Lsla{0F0ZXyn5ZQeKXPJXFXr0a*_>dmv_#83*3LeL'
            '(KN2NkMW%H_DMkXj2hgpQ#QYSim6Vee#X`<<14QCODC_C*C3#F2p1XvWLIC%#s0(tp@vQCRW^0NG%Pa>bRba_bhCkd=hKaMJRWS*'
            '7{zWv?$5KKZ()tkPsz4pZaz$(v0l$g%wIm5wn^R}V^z)ktS13gxf{xj*?YpPU?>v=hPto<+Tu|i9UMyobP`~(ym$&fb*1=mFrV`s'
            's;n<f!UA7OlJQz!9&eYtf2SS{h~?@^+?0upstFrP7wqKx)yytvpsg$RY7;Tm`fKQ<jF!#JLQ%5(QC>o%$hb&%Lw3tC?_-DR9kr>}'
            'XM;MU(*)<V3Zik0t_mGzl}%fJa$L7!1TTkoqrs1ub{cAkbkJcRAPq+M2~34PPr=zx?i<DFbj;k<m7gkeFdl=?B>ca@xH|fmY5uc8'
            '0a79UOR}JI8)oG^_9~MVHQzFoC??oLgX}m?lGQTf&Uh@7fT(0qjehc0a_7s+p;gyf70+^@8Yje%r*p{RbvJ{KYBzd_puiV_b&d%`'
            'o4VU3F?NG6{&lWpwJa`0p8D%jw@g(}*jrV{+w%1v-K-U_|2ACxsXI@ZWff@a*_$__F+9IKRW6E$XQ!t<<y3j0%ODO_nPf$9-8SbA'
            'HLvvKJ>F|7;J4;<@8nLe?qTg4J|=m#VJGpU+@ByLbCzd2Mwku<l4urg{K@8tI;c+dp9W*h+&Kb)I)Yye8LEhh01PIa$Sqqfu!JQt'
            '3r;eMzTRz}73sf{;|POE`90LbsYV#F0r$DiCYIfp@B>l$z&2VE**8UXP3BFDM(zQMPi=N!Su9;MS+cZLSD9<ox6S(?)1IBXo}zDO'
            '_(V9Ldu@J}1dJ-{G{W>s!}dD2_2QQ>I~aXXTYlwz>ZXmV3wKZ(<`inc@{va^?$NEB9Nwzw(xwG(jx3B;fqf%|x@3nJd$oVPQyk4k'
            'yz6?qcuBfYj+0cp&h|AbW)EqUph68hxxGO)#Ec?{nPo8Y)l=ueS@<7hf;}$$hbQ1qk9G%l%Uvy>s_Jbv4E!4|^78}^5un9lnXOe4'
            'I8VPJp-cVcx<%hAYs|;`Oyni9vK4}Sc|g<=r{CkO^?43k2EcT^9$e*DaP{UYf3_!-_)e`=rh8_0?>~NdckkZ$`PsR5-W~M3<Nis1'
            ';GMreL6M5i8`(nX9=NLAQK{WTwZn3mltgls-KuA&XP}#4xCKdU?1K^mC!}%%vdu;B9E&|;{!0otm-im>?Vx|s17b}XEyG0+V^U1n'
            'uluGq_?7zk(}2P-robQO>V5z8=k9U;$b;VRd4Mn?UC*n?v&mfy1>&TCanV10i`^>No@{@4`tJ1X7Yx<W*B4x^6;~i#y*L4M7Gqr+'
            ')<m&pe3*$5s8F*Y*-fdyB9<@??}C={&!BEtjdgI|J-z6|tUUhayZ-4Bh1#0XYYGcmt~m>?zys>^_|`1yzKHLr>O9MHKvVc+ws+oi'
            'kB@)s9=<Dnj}Ar^>NKmK9;v4q1)9bHm9tiV+8fuz(%~vg`*HtW&wG!dGv}xIq^!cH-Tp`aYQObou6gSH*1CRpw%>eIeuh@t#t)(U'
            '+XxdFUtz-^Ky3XNqHmnYL-?;z;3q-yuMaJx8(0p#1#O_uZ|S#J2c2v9=eLXRYR7I|{qS$sjUNtZ+-2zT2PkNrIgi_Mum1DjeoKFI'
            'cCWwNFvWD%GK#pGYjSnH%nOaxXN1(8XFIR0QCI9+Ghu1vq%%Uff||K0h3av!J0)GZQcX%3C6unC49(qKkDa??fYC+oqzi}X5b)_&'
            'P|{*d?oT`|cN-wr{U!$Bx8i9F<k=e;&4YE@o0yUc3(qIv%}tQx3EL^BJLtKntx>{Jj9ij=r&0_GU`*yR<q|kVBF>pYY$}9KsRCr?'
            '<EgB7CYZp;+;?ePmZ#6uywvToPVjD2NUvu+#h8c_tb%2g`E1C#pTsOML?OCy@x)?YCpf|w7m;GB2yTF!PO5gl#K}x&;*!Ne_xR3j'
            'Zw!kQ=_&zjgzJ(5DN>9GvwXzatxgD^*XxqeHb$Hc8RTNjCqlWyRS6I1RSH%yZjXs?mp7wqnKx;C?Ko8!UpJIZRqBSU(ZT8lEi;TA'
            '=>R;DHDSm;NJBgL(2<o^OzR=Pl~<x>BUUz%Gy{VX4%fuUy=kzzo}Xmw&WXRMIG82Q*q@M?poVW&&LDrQ^}zf>5J84tGK@3x_+DNX'
            ';7r@}XT#MOr349_*USB9P4RO71uflKBKoFuaR`tOsjO$D$HEq?2q`h;qsCH2*8AUf3Kl@jd5lJfoPd?alUeFLDFm4BYc>}J8fzJy'
            '4lLj$jh0I>#qZqvsW3_^ZJf2FxOw@_rzfS^X=!H9|4d_|B884}Uhv`8U!_?EXrbv76JxiAVn3Ao_^!NqqA(i|WA#!GBe=W?h%#$='
            '&nerwl|_xJ#D|@y+nkhe43p%F@AseM|GdEedCAyD9{mx}G!`v4uN@Fid4!iBfVWg=1g`)pd~u8IW5GvFT#K~D!RQK1RK1x5{sdiX'
            '@#Z}UCR^c*l&|zxnVpq}9&wql(sN*Mx8W$vjC_@wAqb2)l%6G9KqM#e)^jOaD|ds<cvY2RNd;R*>&C)QW|&AQpETDmRZopN>Oh##'
            'PlU=}V-+|0C7tS0Z%Mu|YfdS^1`72~wKg1*aGvtKV24#SU(IN^rE;^;H=00HQ@o-9Nya0A9Sf7iAZ##gt`3y_DUOIU%S>G@DThoj'
            '7%*A4x+M7Tv3ztC%>6Zsd{lHRQlT%$2w05RZiKR`ri((yPlLgGF*F7mid#TEEZyMaEgnvEf@5MK2>pRVoGjf0V~{O;4h^5=bRI?='
            'Opahx{m_W@;#um5Db~`Z5+9LvT9<6Atz#VN=8=|Sq$x{%_IS9pil5`ZJnz)t$2Q+n0&vdx9NDFwTNfHsh!e6!Dc+XKcmk(Fv~vvj'
            'GM?d|zWNh1p7UQ@-8Z1^I0ecH-;#sK@04$wO4hc*@mu&G8vAX`DN6V2a-1s|_S!99x=Hg<Ls#%=ijrzks^pGEpL9t*03I*eKpwql'
            '#saHe(0x&W!e1u5mHii5O6xBnn6U@;o`KK|C(;mb3Okd37TULEYqAmJAXVQi7om~Wdhz7qg}^utkQJ-5tmBc5p;YVI;jHrM)K*@K'
            'p+<&=(>7<H0A-OO`vZaYH|GaKNp*f`{5JUxRMFqs@UQ!WoBrS!5+07y*H}zLmtCC*P3T=4HKAqWnQK|gRNT3w*cEF(s5mDSw+*x>'
            'lZfb+W$gl{;z(}YNO{rh0+6_(bmIIcfTI8lc#<Np{+EN5pzp1nF2G_dc;pD6GXzj@x_n+F8XGa7!e(9t6FG)kviELvV2jB#%5Gd+'
            'Sz+a!CYsmhSe`S)<XRww?U!`j0nu?_&AGO_M~V(~LH#zp1$o#Ux<vu!tOK`I#QOwm28o96%H+53r<wFbmJj73o0u~Jxduq*C~nWu'
            '=~`d;H%7?w*2Fv5YPUdt)3rE0J9~G*`3QU@xO<j$NYdWDe46sQ{_^}7xE0WdT$k5LD-xA@FwJ6|C4BcDQ)#~Dge(-=uQPQP0@hg-'
            'gh8uR*SwhW)Oo;U;dHdy9!33m-s6FIfCrCNT6w)sH&N6l3Z}_vk@}~+#EP}R0Q<lF_FK@FyO?woemW}Pv3@4BAp85<ezfj$tSMum'
            ')(Ck~rCG7`4cVccu11ZyzZgw?u{aQog+BG-yYlCVc5sWdA@`-&XyZ*Ee!SWOKx^g%7B&IK7gq=J_+5X6Ym4{(7_Tc@sGgvd%sEWZ'
            '%K%f+uTXG9>GS;&)e*@MFdRb(2jd2-_VJa*t1o}0K^$kv)AQo#6u!=WI1QOPYn6=;v{#}l!@M;;y1Y<p6PJTS!2=wK)*r=Wg(OED'
            'T@0F{^(R5)?(f4?g==4;Qycm=k~1aCq#p$(oWS%GN|9{JD(5J!qLqaz`z2K45k{uce6oUbAf5xpUm#dXygQ*%8PE0)_V;OO(qR8A'
            '&5TB;pqDO20O0*U%XThPj{agf59GGwImBgZaX|`7i?)hQdoX3&vWVA&;+HU*#P_L8TJu30{)X8{T<DOJ*I>qv$%gzIRDDnbO4c9z'
            'JN(^;zjZ4HJRp0%Yi&PO{-X`mzTtF%K!;<0?>J7qZgdmOgVb?dbTH*18Wt0m!`6ye^85fsG`MLy1FW5E*~Jxmd*a0_U}R7D+-iw;'
            '4I-Jn416L5w}fO8M8*a_ZHgC7@yrw*yp*@LBO8Bb_{75Vw9|McTKukk9|SXah7Jt>kN6`r{iAZnl{jI2>De>=Dax>ve@VBm{+N1Q'
            '_#O{6!GSWgK;~}D4QnZF{=EHc5l85!Wj}=f-#{0!?hR(zu5&`hnFNvPPTerqDW_pQlC!X$ZJNZNH_t%g5|9q`x7L99i-`>4FM6Ux'
            '7g~SPW&oyO@O?B^RuZoY*IP@pf`h#)a|2Nh2Gr(}B+DB`F$(4}mPyItCZ?2NN}#DXqi8sZ!kgQyrHCTY2<<y-ETkihIc|3R&ee<j'
            'P}Di0TVF8RKk0;C*U=y%<xcZ>?9Z+C)*)ce&ZoGG&5CKrHXS_Ij3t!&jQ6d2KmzWzTdiK@tP7X|i%*%#9>%x1#XZz)`M<+HYDVxR'
            '*tg$79zjw^+yLt3DL|Pa7~^;`3ISKlZ7dVUXYMDZ|IniYdu<bGa-XInRTFL1E{(CP&LI}LH6f9eGO|lexYGg10gI&|f9TM-P@RNH'
            'z~zX)6LG4xPi2Bh{8TwA$5cjsyh;)@;-Q27*pH;bGY@>qCU)!3r!D+k7DQ4q+|$ng+CSg$ipSr%uY7yyCzKs+2D57LAE~Uo5q!Yl'
            'uXsj(O6eKa)8URNxB!KpPQwrE#LmBAO;e{0?H*z>_bkIn!+msg_cp_m{I19iezIDCP>;}rFGeY7L!O_G!%&qnr9$yp=Q$|`zhUO1'
            'l8-zCE6oqWV%ZeDPIYDNy@6dgj>og2`!4`IPBE!^JmwPb<C(Y%eR0NF!~b>WVP%*mXD{+m;G737Y9+55r0n`Oj)DI~@#^N59XMgP'
            'c{_GNNru@%H(Gkq!X!pI$YO=oSW?We45Ta-KH;0XIn`5FA|o|A4x?2rGy)J4yuX%}?8Cs&ahZdoB=5mWGtUDVVm@WKG}Yug&&cet'
            'jOX(X{QADh3uLQ#=M_9M6K_EOwuwr7X$oM*t!X1r3};emEkDzgec?0mr|}ul5xSXdk-xNu^QIF?zRl%Py$88IW4h{cM~*Lv1UiNp'
            'Gb=Cd6Hf&B%u{p?KC>m^N~5u{5I|Wh_C;{i-v=@V&n5Z+&!RsN2i5Oi&e1L1tv74Vr#ltKW;_5roPSD2t4}a-WAvT@z?K0k)W3iy'
            '15g&kV*gs&-Gf@9o{zGMV4r3U=?_+hZ-wuv?O0n_>e)rj5-qJ_!xI16(b!Wx$T#e%FF3UgMF^-;7*i}72d_xm!`H6`5tm?6GZL(x'
            't_nqiGK3+G5MhSUAxD483B;U7>Y>|iJZ&njDxC+xvZ0TlYaN~s25tBj0O%nLb*+17jk2{mA0Xy<9;ZP=HFbAta9*25KMe(}_{%)U'
            '?s4zvZSTA)!!8Hq6LpoDb<vCO&~_J8R`%5N4?XOkA71tc-j8qq-@)L6pU!(1Kb;+;+vSVW+ru+>bN0(ATEC#dv+u>X;?=(9JO6>x'
            'I3_-zInR4cv4MHRf#CHn2K|#RmJUL$z`jy!<&oJ1aue+UjhU2&^Y{$Y^Qo??AeoPzN8v*FBV-NIwwiGnX?d-ua*VR#O%WxW)C$L)'
            '`y-6tn1~@J30R<?2zyo!+oGQ_?MvC(fft-0{7}#!6{l8pb1EYT;Y5HSz$sbLWUP_|r%M6>vNg0B8}5`UB^G;)p_yjklhv4#^U?Q-'
            'zmlM;W~;}gM5_cUw2S~k<p}|g=oVQjZz+5ZhKmV!xs0!jS53aIP`+4(O^yZstvIc;;)C4pO{2P~rn<1u&Zc0z4Sbn-GI<xl!#vBj'
            'eH?X&G0Y4_O=*rjcv4%@LsZ0ZZjJygKPrq!w!V&>Ah|c%wx@ilR8UHjkBaC^Jtn%CC?6^`l;TN~dHaLUwVUQ!&XXk3!;UbkPq4am'
            'rXBXYSBJ4O&Vl0C@z_&3o^o7=^K0+CcPs|opHBLxZM1T(#_<1<SO?h*1Gp#=!eo`K7AsiL4Tf#$oF3(1O7wOTnD-;YEFf6x@)PoY'
            '7`Z=ew+>d?J%wuMzffV41f{xgLS3hrDC|=*vJ?{cD2(I`PF2WV=(<`Wn#(Ll<&&MKpLuvJzVV|K3`?snM%q2*<b)7cW0qsE5@*Ae'
            'CVN!bLB{z(**emghcP3K`AIR**xNoryZ%4>+KN9UJqmL2RSrL{?SsTY!Ji$K2qtww$;GvH>M+2q5$7e@;&JMmv|+!c#L*}q0p@LU'
            'w~S7%rIcpKvb-cBdnTGpslJC270Af2QzbuY<V*d#AV)r1;!{Nd6;@T*|I2j!$~-aGdhW<%Y^$DDt5D_Eio2m9+21Q0XmrbF{`CR_'
            'XeqaJPU-pi41h<V@*>Rj%auiJp73;S4T{_}c83JCjtNK|9at{#p)>F(v99WZMvhn!X11Bz=tOnozb6e--U4(niVohdXw5IZd$M7$'
            '`M#1|+Fu7jDi0{!?n9+%O~@0js;RsXsGmn(3(-S~Gp!kibI5LN0!lze4eHoL-U?Xj#zx~fo&yXc54D1MeLH5U+F|*r=0?1fNA-YY'
            'cr`^%)h$bgU3og;9BF?Or*3UB7>(31PgN1vums7i^PFZC(U{dVF+Utu&R53y!lA0DRoXF%))Zd4n|f!PdS_4d=0d2o-=0+Tw$r#;'
            'eXUoVLj~`zJY!7Bg^K1n4;3PLR?`N5s{HBvlqMj@hNuAQ6`&`TwK4lm(WqT9XdURzq(`ni!`D0IFk&52O1qj%<JSC{9?Mmfxfts|'
            '1{2CHhjfQqygr0B6(m-s#*!4p6@S2sk98Hnz0$3&>2_pY*`^u<&|$x3$g0DBT{5)D`($&@Rg~FQw(!w57ENroDeD$BaI1`NvTN;?'
            'SM_I9uO~0QiUK2N$dcXNl3&7KJL1)TotrPiy9#L6tVUPLFk5Z0<ZI<!u0MI3_+xpT&GO`zSl{QZ?f0y!oD=v{*I)>QNzj_k`8x3T'
            'S_H@>OeB$WI95o7O;NpI<>a|vnZi+;n|`@nD?`}H<;8$<*N+19=SD36ty8oPN~Hw!75hu$7yfq^>B?Ctg{x9}X6^o8u&9Ch_H*w-'
            'S0R#BDRQcUViFpzkzi^XgvldVQrT0{*?7wVFra-Aqtyh>fn@zNEZXX>vMlo&B6eBUt*7F63nxQj$+SuS@{JmbsY6QGAd;<MKPKUm'
            '^;8-~sf;}1?-ESflZN9^01uft#q?Fw)TYKX(UOKs4L{+ghHvca2{Dc5)?M;&@2zOD^)cD+lc=#)Bk#pu<N)<w(&<4TXd@p*V-#i~'
            '9G3@Lkje}CX%w$?7gS{F1sSy3%Uko|LTe+j%W_g+yhYiL#2{-LOR`lwB@+O<f|k|p^bY6v&HP(9bq7M?Hks(eB%ZG*yDE*oTQ&Ob'
            '_2b?H4E~6YE(Ul(4&JnwMde}^n-`<0JFRE2)p{oWES&kgp76;9@fYNp7PzJpV@=tnHD#4+%H*1=$vD0G?RPV~@9KvCD}Mh|z(oCQ'
            'VEylh3?VVFAYnx+XhTrsFVovN+tR%838cH)I$la`^GW9LdgH{C_*eI#Cu|X)xUEmkm#r3?Pk!x8ZGG(EX^rivk|uR}@>4}k>NMo1'
            '%9_+^ip72f^MmeYJuiQ)U_i;gX1%yB-BBoWQAv?2+3i`JjMMCQ%9BO=B3VwVWyx<r&YN*!tER~FG*x7ltr2FP5JsEAHrs5<%~*)J'
            'mA9Yrw%yBaG2<t|x>I|{UNf#5LGKg}zUeCd_fe*Lq4H@i_FYFj+xcvo&lZ&ZCb`eo(0{U;Y%wL45-R6onI$*rcmIWKJ}3`ch*+)t'
            'ZLl};LRA$v`(?xa)7`N`9%j|wD%pt(;4YZ^HrZLr3bK|*OLPrys2p_s^iZ*K>b3b&KW(V3nW9Yxi&mr>%bPibWfCty0~VB%C(_2Y'
            'Pk`la*lcMy--O2YrvZFb->^UNjX2`MaO6Ej+N8_?j;RiWk=R1&bM;Ol+Z*jFT-nj=xW$?<8D~{;vxCn=#u6Hv*!Wau5adfF|C5lE'
            '(7L`u8BNKuXS$S2qh6<qak$m+r0B^iEYVqFYOK0N)laAA6Qk99#T6YeWzS3ZH9SE}3dV!M*C)PE$wn9cN9QT)Ho8}N3R?R3*UII^'
            'a^{~Pm)CAgPIPv*B_4YE=7o3Fq@}FVltw8j-LPxOXstY=9H)Y{Q^GGCotvW!MyZ?GX*<%<dUCRKI`JEZVN841MTX~{PK*{u+NqnJ'
            '?-V~I*Er?r*0ekf*jOf4pt6dj;Y61DX2&;}6BhQC_wpm4B`Ftp^eeLEx6*rmNq*u!B~Pn1o~d-YY}%8RJmh+1Y4&28hA<6`c=ie}'
            'BHTN`XR?i9ZnI<#SS<jgSjj#KX5&D7j1x36Oje1|$%%IZG4t0G%my7MZqXGeKY5tYq+Elos0mU!oMNDFvBH~LVyP9Q5w+u=$~s!v'
            'k}hIv926>w16rxE+2x9L@KAv(c`Z_Mz9Jx|_+~=&L4gniAKO-yHf@>k(Qz`RoXh$y)2vBDzA;>(v3&n&TS9p(PAN^J9z1n_+Fr)X'
            '1~)Dy3fbDDMJ2SQ9WH79^i<`JXNn}fcUj$GAC?2tiFRA<=ZOUJUr^;iGa@?_>MA4Ty1Z3kZAuqJxmMVGuQSh64hA(zEAG3U=X#eE'
            'T|T!o#Zt~`1XCxTSuo0f=4-2lOsD#B2~PH(X4wJsMwMXF-;xt2SEE9zVCH-r-nUU=*KoG(N&rR|OmLHcv%yIt81-oqffdWSPenhW'
            'ZzC_!eR1?o=9h5WM%qP=(<vFni^)E0Pd53rLTVhonTL&|*E>dfL1fSL677%`HLSUbOq(zZ)^;wd1Dzp7e%EY4&S6pfPyayl{`lgV'
            '3|bE6lv#xegaG>SwtB?=@G4`B6|?XzAuYb)t5fUYPBCNkEe1nnX?aj88X~iwS#MRawcK#>QKCuG(&QycezAkpjMwmJV$TE6gF8+Q'
            ')r22mWQ?e)uHr`BnehydiXBL-QwVS2J7tR~<vTb`R>0K%_siD!)=yFj?_y+AV>FqolSrat!|0}M@cu;Hto&qRq8jIJsll{;vlfkk'
            'YS^imut(;)Fv9G(RXNPAP}So@<5?3<XC&9Vw<lh@Um*j=IaL(k@}-=!JNQ6~g-+O=iWlPFzDCw=vTW0b70aQ}11eyUa~33?TJXeL'
            '27td9E|7o}m+AyPW(|^B9>KIu_6efAgr$dt!?A3*#Xjl>!i*%`QSuDUw(>$oe^aGLWQEVcpm*|qaPisva4Tj;`mdSr0?6!Y5@w_#'
            '%fcy`5s!`l@;sw_3cl{_B7}E&LhwDaggOU?U|xZFMkxnDeA5*$zEh0E7)nX}yEGJ?yrF~%)vRE^oc#<MUjL|f!DWDluT{=Fm5sU5'
            '4+RTHQR0u)2$q?xxCnD>DH`M3I2==cl65L>VYR6W6#WHtl3^tVbYxu^9SMU}AOo?QlD1+P3?^%)KKwz>azVkRLfX+NM3|#xFbO#p'
            '2mw%YCSJP4FC`SFQX{gup;6Dn%pO<J52g;5#)uZd8i<@6`Ub<^!UOSu!~G$_JKA72$C5AbOqOUhMt2`w$2Ds#su?uYIdc>&?L>B^'
            'soVr(lq)_8xxF+#?Js;7M=nf8V%fh$jw3h~jv-cO8-p|TZ>pSPy*FRyB~~f~0M)8UnT>(2)bHyZ3m$0W(;%>mgj0zS6a-N6*F#|l'
            'kL#AzK-s?X!f>lX6H*0hR#nBOS~v8#5XX@F((1a`)HW|?8^xYlpQjd4<v_Z*ysg%8oYMZAQ~=f0P3M1?@b+={x+3UBf+Y$Ub5`2s'
            'uZ6A~%7hK8AuU2Kqr31ww#$RKRo<(%bhmM9jkCkVN}_DB)asM9<8=i#U9NuHVnXIyqoxpz%C&^(GQb+Xw%(9#XO(j%`&3plA-nK1'
            'ef>&<f%$_(Hau3>sDSD`Wm;tpqpBnE3FUE#<bF!!`C=j6LK&e_FmrwV+`6*6l{Ixgd7e*K-`!&7TNmd2bTC$z@ZDxoHMoZMx8kmE'
            'Xj=B@kTu73!c-H_ta8D>Yu#!}APy8tQxPrN1^_Y`XhX^_vK4AmLeYmYpI5b@=$5Yv$b?&V75<c!^N&Y&=gIhgI{oERS*EH~EpnF%'
            't8rbfgJ;(}6;G9Bgm<%LG2C^pcXldytHo38`t*vo?pPin#<3$f*tNfnwk3Y7e-~S+3ioE+c+HvRuga?PvV1jzf-^4y=QfYWrdsq)'
            'v2&p~{cSiN=^|@Y=q(y}&i5o~(y4943`i*jp}Mb&h3e&)WB(@6EE%f~RurHoJ&^6XvO*o^aA;_iRF%~(rDgq=lG}P%&d|533s>e4'
            ')8!h;)i~F3M}8DZyVJr?G4x%>&~pJusWcn{#-m|S0;+$+BxxCgc(bM~hyiCjZMAeZ0M6K2s1GR-+M$S2&VZ>GYw&8#UCHL&s4Iyl'
            'eaDPINuCPG6d)OolW=5(fz!)Iv0&rV0ihc*sxm^VdQo;g?A-qC3`nhi&`Z7A#k073{rH=4lQnh7AOY<N28DO|hoM1nQH8c$zCqF@'
            'Y!Hq3lCbbS|6ULZiPaK=M`7e$CRCGMnHif=jMurVM%k5#XH|}>x_Wj+RP15CAWWP~mLbX_31YF#y?@9CxpP&)hzjPIRcdm3s5C5j'
            'WPH<~qq3PFvRtpjze)xye8^aH3)@DQ8@ZUK4UUvhwlsWhk5mY1r-D~?TVzl9kM2cfW4Pu5=B3Bh6$6n)KMcr1D@VYV_Ua5orM4CI'
            'iD`L?DN!Q}Jso-cxLtWC*^aNb(kjDHhD^cJMGuDtnQ`XXMcqmWQna0)8vz2;qZBA0@3&><8aCAs=g~N#-gOGQVN6;You@%CL2gW&'
            'Lfy2_wZbGhSUHV)ag4Uvzw;pb$>Z`nF^mWGB=Td@rV0d`qVeliFM#N$<F;3T21%n3Kb8-N#N4i}VZOEMYdn=Cui<bJ5Hz8@KR^4a'
            '|6_m9JHo$yJU%;ocMQ4%mqR%054xxumckP>Y^}9)9*ikHIyI@yJy{CdC^$L|Z%ngl`TkC$JSWE3qV{vT241=FkU?P4Dxk1Nwsfl)'
            '=@x@=Au^D5Z9d-Q>}s7nS)^(I{1KPQ<(Y@P#v$J%;a&}E%B~J|vFJH)MtlPto6Dc!l_{kQ#K%~&a8G3M(p$=ZXYjAaGQKQni}x};'
            'T0EVGV_-!<$?3-Vg}7NkLriaCj+oxYtNDaXl4u(YvsB#;iD#GuPWdk2nkX}~K<Ax6_!C_UL||@plriE(n5imd3$jv@kVBH$J&d2l'
            'X!MK|IUsyaPsyi*dy;@m{RK!AnA-#9jvfWkc)proY@Q{yJRLJ{+9G&ASaic1q{qAzLjgN-yyrew71eCfkwppo)|<TKKnWW;JB<#='
            'rB%w{Q0E30Y8dps5xjI+U{3B~_kADrXaGE*2CyF_So|GKS`@sdw&oWNgY`FngmZLqq5S_`z8r}p8qHNG7m*Se&NrM$6b#Di;Q-^('
            'W6l-|;wBy%L=e&$=>fSGu?^97Fd}Uf8k9k=2jB<v%?HOG7BmX_S8r*7c9<$IZ33IWJiwRB!@=cw?+9sYzB9^hLN%+)Q_`a^!Uf1y'
            '>!oyq^0DRt7n<GU<YKiwd0r03pj*$>5_0-uatAr_x98}12Gf@IgeRw7V<mr3yYQ$C_7_y=B>y@P1K8i$9Pn=)Cv8K{(HjKRk<Y``'
            'gG#f;)lw+Q&EvIN5fRQJG~1m0qd_k%Nr>Mo214$Z9%UCrh7QE7AvJ6dQ-OYHnoj0T_RjRCQ4@5jH0Tf|8~e-7pYTT#-4v=knZ=y$'
            'QK2eUKGJ2Mpxf&_Lp|Q?z!MVrS-j!7v5VP1F#Re_hBEl`?bHw^F*I*=jagS4VyMszT2MhKl(kv%E+86FHqI)uG3;H3{ufAK0!S~@'
            '-xtS(2{Dkwx1#Y}eEY4i;Oe^ge82s@_znka^GKMjT|C!Em&g07FruW@1~lkhp@t1SQTOSj^9HrIt>lc3+Xw=|c#a;WlBM$>-qfd*'
            'x$0p-AFpTdTIEv`!k4?!{c?A@U*}TyX)h<8tF1maojJqY*%vVr*tQz#d43KhG}Ou4Fk;z7SdR9o!69`4B6Z`LaUvUrl<rEN@<RJm'
            '=dNTGrTU&5Whu%HjA9pCVTf`d<(aPLvY4VY=twbfD9=QLk3iEfk|G<RNuY&5@s%M?8I4&ij=_Oh8I@;ou-9T}Y<a`%Z18CumrCeO'
            'hOq5xW+yzO$|A8tE=g7>U3`W@Eu19vw=5MZ%fPeL@gvs9#Uegogbr7DkItpF!<5rS;zf2UGgXcN`><>>U^Q9YRT5;Gdosw-G1jn`'
            'p`}BSsg~%<b0wrjA0&eQ;o$gJanbvaOU_mXf1t>4-UC75^g;<_RLVfeNRM1t>WaC~mA_Gr4XN%puAVptR^%f>&%#%57se~b1yBiA'
            'VT5u&M@lpeuP{zGPp(Rwxsfe-Q$M02MoMyK5iaxR@l9JV4~x!YZa&;QlDrFL3l(Q#PK7Lr`BxK%i>#96`H)9?NDxD@&H<nD*K*OM'
            'l1?fjAyl5e%Cn`jYu0}F<^Hq1m-{cY6G%?%siumwfoV^wd!QWwp0?Lm1!|d^1HIJVBz046>e~dXe7nB0V@V(%^ZO4U^q2~$#HX9Z'
            'TJ#bHPR)w+LX%Kqn(Abv`f#b5Rq%~%%5-0JPkOe)r-ZIzL;1I4@oi<+fe%~^&fZ_NWj<pvSO)Tt?CDb60GZJmgXLxad^|dutQ+ch'
            '7MXM|1C-xoEZ=R)6%kKRADjB4gz6C&Ds#bif+X{z*sIRxD9tE}<DhDyej-ZfjK@1UkJeXP)wf}6UNzi|)hBey86!LNPF`wGIj)tQ'
            '_3~GpN;fk*sg72YI_zhW%Om=iA(W^eO(lQuKQ<?;_Ajb@+N_X1I^{=&@juB;*ofwn@*(m7%j3W1j!dB-UL`REox%-TIkj2EwV3je'
            'FI{vPt<;rAX`tmCZ)r9TQ{OqiWUK0|-d1#1E7u!PaIkTBC`zYD4a#s(JM5wRDZFO8R>k|8G;0nY+6n+0f<yW36ItGjSW-UMqX2WJ'
            'Ki!T?xt5wCACAUrl=v`PU3<Us8j}gprmM>0)>yBlA=jLpcAmGU^tfYPv^|Y6<|Q?~F%8amOs>=_$n*d@>x%4s94ZU&t!ZwCqdmQ}'
            '!nJJS!cUqE=(vS=@%LIheaTvym$^~be5)Q!8impVOmUmyMMbi=lnSDjGT>m5_W2#Ul#`r{HxBZVe4+(tTbDG|NUIhZFE_vNJK?PK'
            '7&QPbe)s+un1fw7$k0fdd(ywS=%2pbW?#1{B>Gi1f^Oa7G$h9a%gYJ`-38gW8-&&F6B$r*ZzkPNd7*p>zmzaf%vHSXV3?H2?X~t&'
            'Df=W=u_aOE%o5Mabhny>I1~{uUwf)@d{Z`ncWUZ%K{cPJ9-Os`Jg9Y^K;<A%J7lOj4789B$&Lnf$v?Fs{9KM)6RVo*nBp7KhX9Zx'
            'eRsHHe_NiGYHo2|m)rJ<UV801uT8;Uh{;RA&|wAV=VEyEYrR9(V`Xe(MZXYPZ@rY^$42<<Cl{G9=1_{{W(zSUxKQ?m0s$=<8Y&{O'
            'S$*i#UN9GHbL&l|_cb{d$!vW=Bb{?YHebtL<q_FRVHO==tA=UJz(Ak;Elc`Ee~IF3i`m0UlwbWPcZzC#gwu724wt9A2vUk9`()y6'
            'pJ6*ta;M%p*|Z}7)zxMm1Zao<eZGvZ>}4I>ZdM-JIA;cC*dh0Pg<Y45^D`*+m!;g*6r3QL6K%QEMVg)eQ9bUXyK&V#Q%buc=en`E'
            'DVm@iD!BKQJE60&V?jv4A6tCL6iyoTV0(M{bMN%H|4x9=bJRc3Mzk!r31P$u5S1)!i?^%0Aex6W3j2W%;l-$*F8moso;$T4m@=65'
            'QdNjT9FdsOKkpx(^p5&yftXNairl2f>ajL&MYVpinZ;}x?{men%JeL`h*ovb^5iSs%fV00S{c;+pCh>WOi8u!|Hq@ms@V*KQ)Xfk'
            '-y|!_&2Z~umH|vVLdI10pp<$<_N+fejy^sq#=T%xQw(4bp!fMREELn;cmz!+$=hJQWL~=~85d}byY8dtuF=gk{GBIJ<Do9|a-Ej~'
            'bN=r?#23BupL^%(7!Ud<y|c>!mq96uvVW<$%d4D9xqGENiT_pI+?TpHS!VMJiQ*^$H_A)f*MQA0{_(C6{=XWQWhJOq=&<up{SWj$'
            '4SN04pS#EXBM&g|{pEn5SD`5VMLavG`BJHEEv)p74JeV@+>cPzT{^zyby%8Lvsw|;Y^0KO({;~IA$OR(FSOG>OToW3p+(907xWT$'
            '=aYs1O2J<iF#SBUy|scuwvX)ufA{~m(WRz(P1$UliwpA4tz`wLjI`LJ^su65>1gBCN9nk4vhHlD5U6I-lz&ZyEl_K&Y-JhtiGBI^'
            'l?^L0E85viEP<wYQ^D+&3=eg9fPC^<<!JV;1}#V$Aki_Y<ehK4D^nhQH{TIgzI0;<qfPjEb<lctt?c#w72?|U^-i5MP`qIwvE;Ri'
            'yr~Ife(<k$X)kxLA3-qk$^N#RBQD5IPCi$)gtDGZR~B(J4rU?a26}yJ%RZXxot>xTTUg%Cb~38t?0gOR>PyH~HS*LaQ=#9lx!B~V'
            '%xnx|WHY8hCgU0k^_djuQ>oEo`AXsUY;%pCRECaud0elCZKFm_oouzRuC-+0%G6InOeWEIFzC~62}5?TL3aV=CRhYk{Gl@rXIUsN'
            'e?7W9zdSza9`)Ofkp%1aAaenUzj!j9+_kGEy-%dsl;YROx9bBtLrLe7i{451bkIM%_$tYH0uEK<x_+FGi~Q9nrp(Buf%TMuq@{+h'
            '*1unzoyrtmdw!DmYx16orxeblg4?7USQ;;=P(~{56Q4as!i>pkoKi%$FNVV*&O%JFVyxR1=V+%+MWrvCs<0?atgt_exLKj^nBfde'
            'm%VZ|Iz^-WyPTv^8pf->*KkI0$`x6q$F>YlDvrjtbyV{fJ<;3jxZOw`SY6|X16-i{!*A)gjrMmxxbO@J8z+l`wio)6ng%E(n4+g`'
            '^dw(4p1IZ+RpBXgKyHM@wL(%{X>L<E)D7nW1HX*FuN;@m_94bgw4S+nmNWQ6JKylY*kDUQ5s)&U(PK1^Z&rAdEuj{$pznX+i6uOf'
            'M?Ew7;2Nj)ofJ&a(;d3~rBkewwQ(B8=V$)Xj~4zSOx51m*V=U8jWVt{lPmrfeE8bBv#!6Z*fg6dPsd*;WEYZR0SFy10xX_GM2+w+'
            'eUqYGJJSYBlg&Izoinm^8UVWSZ7`nUPuEDx#~lsoBD!w|7y?C65qpx-<gcYm=Y{26${uqxdAtY2G|i^7DZM@r!v_bWkjXd3aUO>('
            '`ld4ZoeU9y{;>A>9Fw%o@tlew&wI$#)>FKk#B&o*sbeDob+U<a*Mz*+Bm`Xc2#n`0d<E!GDbG?NVVi<mAvxgY-BM|cc;!)$!vBR$'
            'f%TEmnvD@EYC3Jt!-;y{qeLz%ZC+uwtk4r~Bm+)(w5s@$x;e^3u)?2XLabRDV&pZA`G@4u2rIe6_!O;<Tv3Z_a~|BlNUmWP{H2$i'
            'R*7~Ir%TTi`Efsc&OlMiGe+nYHXgyln6jZ9b3B73B3d*`2vcA0xAMMm^9uY3&Mw}`&d~Kao{QZFobKZ;S%L-dr0R2B$lX%PbpL4|'
            ')dCcJiau|L<V(#{M*0wIm*<>rd7maACJm}{UoTP63ZhjdU6e0{^%HkH%UWRtYWfxk)MUPb1lLWn5M7(dK8DMaNfgDi{jZcj@z#I-'
            '{VN8S&Wx$^4Lp^}5~g-L4@Yetr7^-V>D%DLB)kD-qTz1LbJ?Jnq0u@EQunhGUSj^dB*P*uDVfHqlDBP0sYx;cRZtHas($)T{n`jP'
            '*5WJxVrlUPRkpSO$c^D^h0*v+6h1;uHC*Z1Nk(w6fpFXcIh+S}VgyPufb=Q^wXDGjWDjIZstMnC_5JrRUcrgOZLmoIPeX<8(MWcb'
            '7r!sBPKJWZ`yPp$4dr%=YK%&+@@-RCp<qk$JIpxg`=+DE=1Lh}T~+Z$`kkvLPlkib<_zh|0a>lwzPgT(VsNgDRt~FEeR%0=gwfT&'
            'bnSTbjbNa#Y43Ig1kUr9sIOw45;OQrMrE40lI6!YMKk#&nu6%W0XeKbw^WpS9@@hpIG}o=6<<d<DJ6Q3Fkp}D1bc|uduT#z$y_$l'
            'MGaXxN5*q_52Z53-2AKTHcrBiEI8f*=AxzBH#}xw9+&6GO;+8|&neJcl?goyaTKJJr{!U<EEFB0o0KEg{W%_twYW#0QW?I*QBakh'
            'J4074oGJ-SR}uUFa}@v)VBx)8-Wg2p0v|)G!h{2<J^Iq_YYNnkd1+*UVo{amsshM{<R|DY!)1WEirQ+o<qzYexx+&8ALZsHgEKlR'
            'N&`i!l6Q#C+a%Ab={l_3|7On9Y>^zUHVlMDN@>1vji_#Grkod40#BB-wm`QrN2qP<s-#NB*6fw{MF8{2f<^H0HEacDPXKkPlynLZ'
            'xs~}N5Q#zoI@Ds~i$?9<xG0dJYjw$CTF=M|jxY?5Wl-vu49xm4iE>wdV}>1oV!4dNC?mt7qQl1f{9#>->OhW~n2hq2J~u3*ZY0%g'
            '+X#)iaXv9rqh_pf(~9vbB0DM$P_dD_A@YM1kn$LIGuPc18?vFhiX^CtyeORIO#v9@-xSPVLpADyGJp;E3=`)dFM^8|JgjoG)zv%-'
            'ktF4>o9~+#Y2t2*dMWoWCTZFxCZ;?}O;dM^$eE`H<Cf-FmWU+<Bit-Q_P3g_>CSvoj<l-`Gb(?>62_<woBg0TtWD|78QNcPjj=Y)'
            '(JjP06$CVWu*$Ejqm%@oQ-<iC1l#VxXTmXV?VIyn&l~i6=iX6w(8V%S{llKouR8h>O<CH_X~UkYvNV(?brReJ30-k;ahgH*r<4Ba'
            'QUAQDs?w~3YzCA(l^KpF$eL6{lTwYUrOe!kU9zq#$8N!yTx@o!nYp?q%O7!-9(fn9ANu7mG+list!Q6DCcE!rO|BBPhWczJs>N4O'
            ';NJ&HD`%-FI>r$iG(q@Y06-_HR_L3k$StH|4?TN^KyGklLFI!m<-pl5r)0jG$YM*x<J%ys25XFU#>0{XD67L@<Ss*`B5BI7^n~ni'
            '%GZVhH?Y<>W=~1j*@nE6xK0hq?us?Qa1`IaSk413jQ3S65p>Eu6ggzlA<lS2nq_UWu&wfrZL~_&CLGJt>U64=w{O2?0DH50Ke<Vz'
            '_$mv0c514R@S<F!GPM1^yizwWzlMhKx=a#Qt{yx#I0yckX&~G_(N(7+(N$sisq;`C@A0?jx9EYt{8-GYgyAOD>E&v|sd<IPZ7%$6'
            'Z%qZVlpj@Xd+CWiZjKN7SF<uyk~1RFHnnTg$no7}Ctsn0YDfKON{Ds%ABhFZ^oTd%PZ(cq(`CV>rm-pM7K&~5Sc2?N)uS=nmIaNh'
            'z@3AtARTEkw~mfgf~?w+4ev6IctBNstS%!FCnK`6<EY=w(#ciz%T|#Z={C+5@q#@kP^XsXRwX?AR(9Mbhs-pLfc(RNEgn(uh@!!z'
            'bGpq!+O-afA*+QlMVtrYAiN7wA!7@H5G3%o^lPP?G0cKBiSJXfC(<z>8W0z(+7iQGAXTR7J{viM3LHZPCKin}fC(HQpZ(&UcYpCP'
            'tfK1;&O9yE6ms}g>{bC$FqN6Z>xAEOuPeGSGGXhTJn5zEhl%G=8(PC_iB`)4>)K5zzWf;=;i#$c9TK{HEU)ytYZslyq$KBBxMT1e'
            'r?}mY!F|51MG}U-qGXuW;`d^S+|PVOjU{>F-YqvlUYdUD9>_H<!U(W2y2);>O+e^k(p0$v>Km9IRr?IMSUdLR(=abVZ#$rmR`Z!B'
            'gY{(ozTW}%+wkVLbr;W9i@?m?f@z2p=zlC9&@ciDHCGHjK@Wl?z8Chapj(*DgFVioP1Z{?m1h2pvB6E_9CMDRdEiH@B~VeRV`^Vx'
            '+1{8Y-5>*hxI?eqIQm2sso$Ck7;m>ZcVXfK<p&OyER!Jny}JVs4$u1odT0-du0Z4#j(?`)G9c*VW^NL-fi6{eRZcx5^`@oXm{pIA'
            'vAL<W1edDiZ3h^Z7^uLo^xCnZ!s1Atxme{r`~DLlR+Si17P6)%5+sL;yIi6dl+0K9y#2kAN|N*d031pq6Pc>X7Z)eME=06ii~yFn'
            '2k}y7CQ~!lIqb14uD@m}CRw1GK?!Mn>=TYD-4<7ir^I41ZPUG!xkjsuoVeI8#5d}6(L{cYSA*3kz%!DBV-io~$_IEFQ_A&&=^#Uu'
            'nCOBFHeQ||r+ed&w=N~UM&2U_d|@%eOME#Utw=t_1U1wE8S<M*ogv3Wp%Nlg;*aTQs0a7Bt|Yrls@w0<+lFQKG=0I{x^e%xB0$xQ'
            'E8lKCu8qj-)owkqS}QQX`@29^fY14n^HyQ7pI}7XMfa$8*b}=WEWsPC04?xB>DC{SWn=;f5$@ux@Cq#=IvZ+Yxp!K)Gb4k_U9k2G'
            '2=oy?7ZC<w$Z90k@C%Eu^$t&Ze=j(M+D@IO=VQh0m@Wx;qUXAts&<1+4O}H2k}B`VVGhNs>vlf6;uR`8^&58o@zE)A8e<+bLg*WP'
            '+vkU3_*Qj4QN6_;UykWFhj>%`OE4M=bW2J_W0-%FlitHL?t@W#m4the-Ng$(jD}KjR9Wb+uJbId+JSRS*Nf=)W6&M7l#=h23AtK_'
            '1X+hbLK~Q65Ja+)6c(j&H|LPIoKZIBk=M*wr^YuE$d@<9DHU~7Zw@wB{;_WU<)?IQUh_-59o3}}V26x}VngT(4-175U=#CZ@EU6p'
            'Bw@K)0!|;>!k6$LOlb^YK6tvu52sZNqP9PoG@z?JvfQ){(L*?!1N>%HG!9}PytSs!4=X%>97fY(d->F@jM|YXCZ}0@(lx!@c+#<='
            '%n&{*voEV&L!&YSC;vq5n5HXqGpyoG65Y{3S&>Q*+`<LbkQ`C6!m&&hi75anBj>#jG=@GK<V9=c1o+nL3Z#<jEy{Xqe_(J3{KR^5'
            'ZI7yx!?;=iR5hXla`H!{=c8IwY5(O5g=^Hw+GW{5S%}48aG`{!BliF$q2Bv0Mw?1iTU$q^l#Ua?eIjQg@|jD7G<Lr2o%B!p9*pSy'
            'eAYkgU1*l){5@FSt`jQzuo}<s-@{u!AtBRwhc`s<@0%b>LTYmyOnm(J4EP=X+l>QVe$hDu)CrU1Hl|M9RTfA5vj-~l+@EllzP}7I'
            '{&~3=!HVb9dVUiIBVP@51X~R1DRl-e2#T_76hz^O3Q_o}4}XE0MAP!2#Uu!+Ll=-NiTp&3)tyt|627`X{PSrgZ~bwy`oQa+V~+$B'
            'EZIFw$X(#?z_X(OzquNPYP6p*?e!w`fs<433qYCi?T9~(f6PJK3g{CTtGPe*vBtH#qY^UfEx`B_g0J=JxO@8cvip|aW=U&s?(BSX'
            'Adcekip}6ka7Nr<yWxR&{!+|-`q&iDUbRLcS!sd0p~dC>;fZL#?=hEw3KHOu_>qh5$$Kn$iKx_p?%w|XzWL&p{?Xtkd_xZpf9jr|'
            '_Kp!{o)sp8vmoQ)2W1fA4;-GI4tjrs6`j57opNk6^nUT2J|s==ii!~~d*qX67nsRbfZkX1qG`B3$>aB5sq9Y}p%z!F`@@gD0xvfG'
            '6RE+LOw4Be+!fhJd33BgP|-3kMo^keStuyGH8S_N-5M+AhdS-6m+jH3my>|}9PxQoIwB%@yzSJaN%P)$G7tiBj$$$Z-+TpnCWv<='
            'I?M13MziI3;bpOR@83xjg9isfA(f95)I%eAr*JONl8p`^^R#20ff{c*InaGL$!<HvgTx;G_(jc$e@Xmhs*fn3pfAq*Z{F<v(*3!Y'
            'pH?YY8z)|d2T2Duj5@#pxTv!jwqcb+OnM`Wx{Y81P#Ry({WaeEvZIhpFiqhL!4kCX{Em&CL)qAagd+oS0aT=+iyr{h7)?~DY)j`l'
            'S`(<v(<xO&z|7Mzm5!k~*upfnB|uXKD-Cw>t%fuFG{Kk`gYx`&c!Lu4rj`s5#6Jfi>4@*-I@(JRLmJLs`aqChyrQDt5h4Kv<cu7I'
            '_;is8?2uRLEOh<pr#}3@gA}%jm#AtY76vu*HCAJZv6!Z{09enTT<iNIKGv0s(DSz(hk5zmA-)%5ZkWJoXJNMdgb|;I&a?gJFVV?!'
            '$zDMljzonqNMY(8p7ilv8h!+2Y{gc_y(o@aXj6v<fTv{nl@=ZLlmmPN=u0GjPZ2gGzTp0;1%+7}%%@F>e^C%jsJ6{?$_W6`dcsI4'
            'MO3=5knwDkem%?zh)5aIJMg{imVKvXQ8~0rQ{|Z80|=w&8h~%xVakw04xsM{a_0kmR*}NgWgT(|Iv$du^w}%Y7*u})+@2Bq3op=5'
            'v_6rMqoC|@p!%I2^ikg2!nEeRAY2EzNhQ2sH9GA-WrH+^V;`o<D{XVKM%Dv)7V;+`gQVUPQ;bl1`Qp)Z%x7?4DOg{ZbZVS+np%Dy'
            '^6KC~cE(BbBGF}we^uyQc|}n~uSuM)F}+bdU!@etE}_M&i#8lxHzliZO;BE$KvRhba^9v^q;bS*BK|5~zO>q@{Vg=><R>#Pdt|(Q'
            '`I2xR|GqkSd5st5is0$ze>!;fT)si-W?ZGKkK0&#EnU5HXBH+H<mgsQ>^%BE+-ojshg516NjjwoCQa!1{=th^*SQQp?o0@Pik^Md'
            '>E-^x%a_;g)&8|nm4<dg`Pe)Z5Yn2SiI;i>^(m{$IEzn=fb|@7_E9F=p$RYq{4?4Nh7J7nsys~L+|wu~TkLcyNwQ_pG22tx9K{~h'
            '#|!Zo@`%Hy@(tJ%Tr#MZB{ygldwam4Y#~k`3yXB*QR(V&DXW44L79SKXFW@mlY{91#`U8g3iWMMz+PIoOfpCY|M(o5;S?$m%dc~}'
            'Sf(H{Q(79DQEPP$?cWL1137I6bH7<r+~pB~ZS=>2l_<h^=c?+$>2i))bNRv&I(k`_Hsqf=m|b4sfWEG8Z(f3mRW|)x;Ap8hJ3aoD'
            'jiTyqt}VLi2AHB3rq~Q?{ZR;O7?EQbTkqJVOP0!8EGS4J4OFca!^exXLfEci-B)w^s=%cg^eu@UhlKs5W2~)Ya>X{+I#i6dN7{|p'
            'CfQUMJ<<^C<dr;A6m>B!m%>@w1m{NLG=BI?=c@gOAFkcs((hyxyL(_)LysDAR9DK8YAg!w29(t^<^#!%3)WVvzN>7a@3HNsdI{<~'
            'JEjB7K}};!p~rtbV(DV;`REiuUHG)gC7zt2@;9QE<(tQ9RE<}erv=cvI2`MOS|!^^C8eJG(aj1yp~prot}5F}W$y@6IuN~?go&!s'
            'E$ewX#yKJ`KnpWZROOIGm!7RIOLJvToAy|D!h;vzN0qic6(Y1sW?_1Zb>dVB&c@IJOJ|piM!Q2-`Emh0#R@wFZ(XZX14C$FV15dz'
            'j&f3qrK$w!%cyw}CRn+o40k|8Ebp+1XlStlzYT0sNeqHnGBTW|$oYb$AE?XW?)&%WXFqq34^*&N(cFd88y13&hN$ZLD<A?RtPBN>'
            'cXUY`zHohi-ut=V`vn$Bckv-ty@LkfbS>+VD!@l-MbJ}vB6H~1P>1CZ?bd$ELpTmC_K;HPL}1P6cHuU;LT~QYN)W@X*kgM;TFlu='
            'AdGHdh|DhiL`|`n%WggyyysSQSV#SE_#4vfDON>mVyPnO(kzCyTlJ6?QLzLWSlxi{Xt9sxZEH;G+$ya}{q0(BK}97xv6ul$HTaKI'
            'rH?}~-;>$otO(r=ETKXtkRd`&mZdV9t0JvZUb~}QQ=vjZC`OY|Vica$YstxO*s7$8RaA{qQ+1^_ereFXc;}s;T@HE|SG1h#stcoH'
            'z!HsU7gwpMV<Jn5=sM|7NfMRBk9SE{$p(C!h98iFS5Q&4UnuLK8?M8`%!Y<6O*z}RE*sO#3n(!ylh-Y!n~e<-eNzYh+oMIW1_{WZ'
            'EFjonk^>s|56L44y|a()^2n|u=P-`v97H8=u1H^#cUPb6I4y^2qZ5`{>f5@7<;^VBEZ~dj&Qp%R<!thGp1}5oe<wx1Dg!mT30Eg;'
            'Hp<_|6QHFTCkDH6x|c`&Gv~Tm4RZi60s3Tc72gA~ArrL4^6*HRe{az8XoHO}z_AU`Gd%N3Z}<UvGr*tp$Kzv7*5~r$$4yBrjr$Hz'
            'p_Q$E<84UQ-GR~YV5g=eLcIZ<h#ashX)&L~5%A^7$|+U?QOGHULF1!U<y{yZ#W%1aX;$>PNM#~Zb;-K(8JRhh`AE%6<-i&_QpeJ7'
            'J|T^ZBT0Y)2EfKCt*+t=J731BTsmx3x|K^#_i29Rg+eGg-`yWL^txP!h<BCDgxAFiBX*lSyHZulQno~W>w?C$;zdj`!)QUAxCdcK'
            'n%f$>lWU5cY|E%JbK4S1%`(v*x~XAdt@)o;TOtkI0Dl$I^_gUfEPdt^t8E3pq#hN|5$QGLRe$M<qs)o?r@bt8B2j{X3~(bz7!i*a'
            '!!0L*l~68s-T?Qdl-Bc%v)bCIAueT8kFz<HJo0i@ohMzCBK>Q@xf9XPlO5;^%0XLP00J7$co?Y2S<AvC4fa450Ld+)V10fhWb#uA'
            '+K;ms4skdWH^CTPo<G7q0HrVoTnCe2h7?2!^+tEBAo25g_xPl1JMbxUk4CC1WxcxITqmqpIG{djr8%~GT<bi@@lil+@-z_mhn;Sv'
            'tkQq0ADZb+Dr$z*GUbXGSPp$FYtuS%U28?HF-R-0L{xCKD#TYz=B4RM>CfHyJsP&k>SIIVD5DqM@edi&!@XfijTknyBJKZw-Cb>O'
            '8#faE?q5NuA1oD;6Zeo@YeNN`T2726PGKj9GYmtbv|D+hH&?55qqzR>H*e&UORg;YQXFvk&?YOA%N@QAXP$Y6TciuT#>m`@caLAa'
            '#9zDVQpn0J-w1ViAG9}6|AK~GVdZdbVBBz6BH}DNC1H_mjyZO^_9!wzqTNQIXGRpi?$s`-eY@8!XA*dE-d*=DvJ++0=~llkwi`J&'
            'DE|0vl$->l#o=SM5_73Vs%^>&>sHu7oarX}l4LzR#C-vFWY{%8(c?WPLG-UWJ-awMga7{0d*A!eyEs*EdcVFuz5IAyO^?y|XvssK'
            'M-bwEb^N0vOboGw8F|*;2{}ltgT{$TFVEhCpbyTEZu~o|;8_x>!Kbdz!N4ag6b=T~KZj|2V`jsa*n%6K2D+Uhjhp+%BQJ1j#*X!{'
            'eyo&7iV-`x!)BtzoOZrLgc#oQ73jh95gF~6dkakfs}ZGRhDP^zZwU8WJI&lSVJ}1YI{aF|+|GB^Zdo3UNquTM&81(vYK%@M_S!=)'
            'Za{VptNeHV{%xDutCI%4A~xu8VTg_C>R|u%wb7&M1#K)z-oHKW4v<cg-r1bUeM=i!&6!(qus56WMc^o%)Ytom7XNJgR+Xo2tL}^$'
            'E%H^r$dkUggaj#WZ3%|_hWT;-C~ym3_bH&Wzu$iq776-Xx3=BSa{ItaYgo$w%RRCjSBtFXX$UmPNKYyye6kWcYV*E1X-g6cZzvx&'
            'QaY&KY?vkqQGtnZA3-zEVka;gRPVzZ%yoA5wtsy3p?7w1`tb}7?|;7jF;apuqbd%ehsD!LO?Ns!G<FcV89qmIF9K=?c*YYraB#>N'
            'V)(ZjFH(!FVapIR819tsBts?~c_*j;CQvc&`3*yv+0V0CHq}e!eQF3>pW*T?e^h@dRx1t4b~MGD@JOpx)+?U{#hFS1M>i34s{1+$'
            'LzA2KU&@q-)h1AOeO3CI>OF*BFDq82kIXGUp_tQ%i{5I;F$rzr>VuO%o#*7WVz<<W8M2vD5SKHa<fMdE$O50`(<(n)-DR3)3w5eP'
            'FyOOK%8-LGrqHO39Yd=l+Br<>@+))C$))X}QD7mupE*JheM+r8H3(<+KztgjMaL%2$-RTmyouCtSOYK~u;t-l<_5nKne#0u25EB0'
            '7HcOG!unlkS8A7y^1v&xqoNb&7OpEKadq+Tr~`+Q1S{!3SsS*br!+tbBRI)bKd1~Dc~L4HeFP&w12oON>j)!hE~Lk#1(FYfp89;L'
            '*vbaKOF7VgpZjIJm=&ORz=awa_U8oVZ9ksY{xQNh&*9>H$91b6^|OQxZ{c^5)G+KTso5jxFskkf6wPj`qwfp>FIUzIkESXdn-&w{'
            'BZUuhvd;1+I~<}^AtaEvyI_N>fP32BiFpRh4b83yE!=;w4K(e;)`QDt@(hEpv-NaqB7~g54g%uJ-=1JD_JpNu9)*|@{Vv6f@p&25'
            'WE?xpriAIvYkI&Cr}Cqy0(=cWkB1ZRZ%_0F*bYEvC(zlVV*hr)FWi0$4axo}KJAq?>5D8!{cy2{?eN|<{~$#5>fY%@B;TXoXA!;e'
            'TwF2E?T(iaA+Wfm>sA%xBQ@ze32ZGB{Ru9Gok(^juc-$$&X!5W32E=$w?4=?@N{a_IFbx2I9GbECt@=TYhKKnaj)!(r6>tc<de4Y'
            'XLu*bliV7vvC~csGIaWL(JNTpXQ_PkVO<T*5k`OYim)i~-YH1nE05H$Zl4aDAeugFvZrEBo)TY83k<&WnBPw|YfZaw(ZeXDQq64Y'
            'wi=RRM{DB1)c~~hsNr9*TiULJ;U==&M(`S?*KmO^zG8xe$x72kxtPv0Y?d;`F|Qb{`lakQ$%4RcFz>#o@70>5_@%9Aqx_oPa>uzS'
            '>S%rKl>Wk{DVU}O(+5uES7phelw8|(snMZVl4QWmcGXLk1lR4lN#I1taN>J(=Fs;SsNhlq-%C{!hBRN?rBid6E<?=vynx%*!rQl-'
            'k3T%1=k2Qw0yShj-Mf<>YgF}YBcM3m7DG^bZqnpMNwX~{mT24idoL-c<ch35(i+FEGczbLl_KtT&frd!6rcs8^1P>xQ&^g)aU+ol'
            'H8;-gQYUMs*?nWlOsd}+NOoT&J7gMr%&rqo;!uVQ<oe*HM}Kv*+*!GL%Y=Z+*gTp>DA&+F{sl$Tp|`g{JA7@%5L5ZsTo8RLjeNs!'
            'B8?B?u@+Of_|b5}_q^Qu>4y-Zdu-Z6V6r&+^k})ra|9u;#0F)^8DtO#=0#!78Kg%W;+=&1g!!i=c#NXG*(Rqnp|120rYARO`#V5='
            '3*Rd<gK}i~<7pc4Vz#}vx<gP7Gc5u{t!y!<(t8j0PSt<zI<oaOR_>OK>2FY%2qr>ZYj`#Zo&?k;j&(Ke9&@QRrA3T}Uw4I>YIdMH'
            'vn#i)Yco;APSGE(-EX6}PtlypQ?X`l{P-%I8*i6snylM2GwiYxswZZ)aZDpkMs0Vd8G^R$J31JcnUs`XQo{t_;~oF{k&&!^K`D0P'
            'H%g2Z!G|#MdkDm2URhNx!--&qU#eU+MyLQUvV_>xYKPwWu&c%lm5rs7m67@%hN4O&3c1aLx~=;5%PyhZqPkRbgdr64f{xlph#V-O'
            'FZhil90>-%!!6QoIQ`9^Fm<RM0xy)uPc~e+FL0F#j9kX3Ts=k<a4jYQA_mcRQEu1LO@HxRxpha5Odt5$-6?S`OULf2eR#Wu6SFBN'
            'rz{c1D#~x}viZlW$dUOQmoxSo#b9P6)ar0nCE8_qKcPfTsa|$$7wmy)mW*^ojwJug;Om8yXtg5eOp}ilQv@c4^C@!7uo?LX4w&&$'
            '-{ILVcI1=R0R;ptGs1+!vix{*;+n|nId0KHsmm5E2~^*0FQ5j4ZXr|j_PIoYKa}r$0d32zf$f%Cr#|^6XnXe)J+WGHGPu3K9xun~'
            'vG8O20Q>XLXMN0A`S$$iU#Fig`{%uj-qHCl?~GIi%=I>dnM}-KAaM9pPO?WBc9U~M?nKN4i=`R)ExMZbjNM{B0A1WqZZS#^qEQX-'
            'Q%?-6g|cmO)nU(8OmnYvzEegyyQw?LyFUd5?+BGXTnEtppgNKRH7t^Giq=YM2W5Tq>ErpOgy{-?A<w4Fv<B-aN3ugyJQBU$%>@A`'
            'Qa@eg4z^zq6$5wZ2C#7kppB?*X{K#p!>lB(6@zAMUDTU4jq52wCtg}T-F`QsrAlLGvr#v<7z3l4cY2xYHJpm`)&V7sUFXQsv(yM+'
            'nrZb|A{9*eD_|#tuD8`qHX3G22DO)v_Pr|o(0>PLkBYU5aDOJcz0yc^Zel*=%A*K-JCR{kiYZcZR?X^j3PpRlb)J~5wp+IQ=utLN'
            'I2u7$hQ(@>=0UO;NjI1jOQg}wL5%=ae^{)G39l%%Dkhk^0!KF2lU9%=M%HaN{-KuJej|(c&s}Nef4Im5o%7i8`n_SimimU^1S}<7'
            'Q6Svv_jm94yB;22RWd%%qRzARVtM#~y`Weu?lSroK8MXw3Xeo6)HodiHg(<(n)_`t^*ftqcf=v_41o9MS8QP?b#M*w#Y3i3Hzley'
            'p>9oz9@tnT3CDxUaIMVhujEWOnuHO*XcAPRycEV@>D@Y5x(j>tPSq{Iua!;txOdjOyy(4G+SLIysD8y^h!_5hD7_h~P%zSnEjFo5'
            '+SLu25W=t>7c=JmWMJ_u{b~=|JYB=Cu+A{!ss0*l$n{uUci_t;hz=gNq5y9X?et-twG(Uu&eJ=}^ca2kjR3i6(48S{6X~=JLbFC)'
            '^1+sm0v3Jz;Z=b9^g!oju^=6sKU>R!(RJV@M5`94_GsG`^7m(pIa#J@2h$2M?TO@192a_-m^?8*efzwx#O#e|hYyG!{d|j=$=@It'
            'I!&7A#T>blNt=>W1HtgJ91s@2T=h%rsYY&Ds7C)UpYv4}jbxVl+=6W;dOg<@&Op$hg;O%Q!VTg){k<Uep#?_TCq&FOTHLOe+4u(U'
            'OVmF8O3}R_KSlBbT@0Cx@F7%RN?LWkH6azk?TSLE<(l&zsfpCd$?P^ABkzVr!$;@On+y)iB|P|TeSwJ-j>-B9|DI;U<syfjJVky`'
            '2`pabyz%O}p00C3i@^FjWVA?qmWDQwQ$yL=E#g4%Jue8Ew`1+GLRnxL^(@=$c!6Kec+BXXt2z!9O*ua1JHB^xQ$h&73mvF?XJj8#'
            'F@mK68nBw^)eOW@-I^~y|N6dnhQ!Il?G;U$g(0izvY23`{Y2s34Z2*8B&K@3)Z;+^MumN(L2pO=_BdUplWC@I^kfPbGSDh-vRk#7'
            'CdC*JgH^hQt8lvVwtTV3*r^QX?mU~1n|=ac*#jIc;Pd6t`RT#Nv>Aj7+EefJjHmevhh^Q$$_ZLwv1M}{U_|d`1iE=lSZ@lrx<Fx&'
            '4Oc_KIjz#YW}{>h^d~{D;NLbP=0BPWT9N'
        ),
    },
}

def _load_local_pipeline_modules():
    # 1) Zaten import edilebiliyorsa (yerel .venv, repo Colab'da klonlanmis) dogrudan kullan.
    try:
        import local_pipeline
        modules = {}
        for _name in _LOCAL_PIPELINE_MODULE_NAMES:
            modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
        print('local_pipeline modulleri dogrudan import edildi:', list(modules))
        return modules
    except Exception:
        pass

    # 2) Drive'da veya proje kokunun bir ustunde daha once birakilmis bir kopya var mi?
    search_roots = []
    try:
        search_roots.append(Path(DRIVE_PROJECT_ROOT))
    except NameError:
        pass
    search_roots += [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for root in search_roots:
        candidate_dir = Path(root) / 'local_pipeline'
        if all((candidate_dir / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(candidate_dir.parent))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri onceden birakilmis kopyadan yuklendi:', candidate_dir)
                return modules
            except Exception:
                pass

    # 3) PROJECT_REPO_URL verilmisse git clone dene.
    if PROJECT_REPO_URL:
        clone_dir = Path('/content/youtube_automation_repo') if os.path.isdir('/content') else Path.cwd() / '_youtube_automation_repo'
        if not (clone_dir / 'local_pipeline' / (_LOCAL_PIPELINE_MODULE_NAMES[0] + '.py')).is_file():
            subprocess.run(['git', 'clone', '--depth', '1', PROJECT_REPO_URL, str(clone_dir)], check=False)
        if all((clone_dir / 'local_pipeline' / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(clone_dir))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri PROJECT_REPO_URL klonundan yuklendi:', clone_dir)
                return modules
            except Exception:
                pass

    # 4) Self-contained fallback: bu hucreye gomulu, build sirasinda
    # local_pipeline/<modul>.py'den birebir uretilmis base64 kopyalari, SHA-256
    # hash'lerini dogrulayarak diske yazip import et.
    fallback_dir = Path(tempfile.mkdtemp(prefix='youtube_automation_bootstrap_'))
    pkg_dir = fallback_dir / 'local_pipeline'
    pkg_dir.mkdir(parents=True, exist_ok=True)
    (pkg_dir / '__init__.py').write_text('', encoding='utf-8')
    for _name, _entry in _LOCAL_PIPELINE_PAYLOADS.items():
        if _entry.get('encoding') != 'zlib+b85':
            raise RuntimeError(f"Desteklenmeyen gomulu payload formati: {_entry.get('encoding')!r}")
        _source_bytes = zlib.decompress(base64.b85decode(_entry['data']))
        _actual_digest = hashlib.sha256(_source_bytes).hexdigest()
        if _actual_digest != _entry['sha256']:
            raise RuntimeError(
                f"Gomulu local_pipeline.{_name} payloadi bozuk/tutarsiz: "
                f"beklenen sha256={_entry['sha256']!r}, hesaplanan={_actual_digest!r}. "
                "Bu notebook yeniden uretilmeli (nb_builder ile)."
            )
        (pkg_dir / f'{_name}.py').write_text(_source_bytes.decode('utf-8'), encoding='utf-8')
    sys.path.insert(0, str(fallback_dir))
    modules = {}
    for _name in _LOCAL_PIPELINE_MODULE_NAMES:
        modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
    print('local_pipeline modulleri: repo/Drive/PROJECT_REPO_URL bulunamadi; gomulu (self-contained, hash dogrulanmis) kopyalar kullanildi.')
    return modules

_local_pipeline_modules = _load_local_pipeline_modules()
video_analysis = _local_pipeline_modules['video_analysis']
render_utils = _local_pipeline_modules['render_utils']
google_ai = _local_pipeline_modules['google_ai']



## 1. Test Videosu

İki seçenek: (A) kendi dosyanı yükle, (B) bir YouTube URL'sinden kısa bir klip indir. Gerçekçi bir karşılaştırma için **3–6 dakikalık** bir örnek yeterli (tüm videoyu değil).

**YouTube "Sign in to confirm you're not a bot" hatası alırsan** (Colab IP'leri sık engellenir), iki yoldan biri:
- **En garantili — SECENEK A (upload):** Videoyu kendi bilgisayarına indir (ör. tarayıcı uzantısıyla ya da yerelde `yt-dlp`), Colab sol paneldeki 📁 **Files**'tan yükle, `SOURCE_MODE = 'upload'` yap ve `UPLOADED_PATH`'i dosya adına eşitle.
- **YouTube'dan indirmeye devam etmek istersen — cookies:** Chrome/Firefox'a "Get cookies.txt LOCALLY" gibi bir uzantı kur, YouTube'da oturum açıkken `cookies.txt` dışa aktar, Colab Files'a yükle ve aşağıda `COOKIES_FILE = 'cookies.txt'` yaz. Bu, yt-dlp'ye "gerçek bir tarayıcı oturumu" gösterip bot kontrolünü çoğu zaman aşar.

In [ ]:
# SECENEK A: Kendi dosyani kullan. Colab ve yerel Jupyter otomatik algilanir.
# SECENEK B: YouTube URL'sinden indir.

SOURCE_MODE = 'upload'  # 'upload' veya 'youtube'
YOUTUBE_URL = ''
VIDEO_FILENAME = 'THE ABSOLUTE BEST OF DAEMON TARGARYEN - ABC CINEMA (1080p).mp4'
COOKIES_FILE = ''
CLIP_START_SEC = 0
CLIP_DURATION_SEC = 240

if IS_COLAB:
    SEARCH_ROOTS = [Path('/content/sample_data'), Path('/content')]
else:
    SEARCH_ROOTS = [
        PROJECT_ROOT / 'sample_data',
        PROJECT_ROOT,
        Path.home() / 'Downloads',
        Path.home() / 'Videos',
    ]

UPLOADED_PATH = str(SEARCH_ROOTS[0] / VIDEO_FILENAME)
RAW_VIDEO = str(RUNTIME_DIR / 'source_raw.mp4')
VIDEO_PATH = str(RUNTIME_DIR / 'source_clip.mp4')
AUDIO_PATH = str(RUNTIME_DIR / 'source_clip.wav')

def resolve_uploaded_path(path):
    """Tam yol yoksa Colab veya yerel arama klasorlerinde kaynak videoyu bulur."""
    requested = Path(path)
    if requested.is_file():
        return str(requested.resolve())

    print(f'UYARI: tam yol bulunamadi -> {requested}')
    generated_names = {'source_clip.mp4', 'input_video.mp4'}
    exact_matches = []
    fallback_matches = []
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        exact = root / VIDEO_FILENAME
        if exact.is_file():
            exact_matches.append(exact)
        for extension in ('*.mp4', '*.mkv', '*.webm', '*.mov'):
            fallback_matches.extend(
                item for item in root.rglob(extension)
                if item.name not in generated_names
            )

    candidates = exact_matches or sorted(set(fallback_matches))
    print('Bulunan kaynak videolar:', [str(item) for item in candidates])
    if len(candidates) == 1:
        selected = str(candidates[0].resolve())
        print('Otomatik kullanilan video:', selected)
        return selected
    if len(candidates) > 1:
        raise FileNotFoundError(
            f'Birden fazla video bulundu. VIDEO_FILENAME veya UPLOADED_PATH degerini netlestir: {candidates}'
        )
    raise FileNotFoundError(
        f'Video bulunamadi. Aranan klasorler: {[str(root) for root in SEARCH_ROOTS]}'
    )

def download_youtube(url, out_path, cookies_file=''):
    base_cmd = [
        'yt-dlp',
        '-f', 'bv*[ext=mp4]+ba[ext=m4a]/b[ext=mp4]/b',
        '--merge-output-format', 'mp4',
        '--no-playlist',
        '-o', out_path,
    ]
    attempts = [['--cookies', cookies_file]] if cookies_file else []
    attempts += [[], ['--extractor-args', 'youtube:player_client=android']]
    last_stderr = ''
    for extra in attempts:
        proc = subprocess.run(base_cmd + extra + [url], capture_output=True, text=True)
        if proc.returncode == 0:
            print('Indirme basarili.')
            return
        last_stderr = proc.stderr
    print(last_stderr[-3000:])
    raise RuntimeError('yt-dlp indirme basarisiz oldu; upload modunu veya cookies dosyasini kullan.')

if SOURCE_MODE == 'youtube':
    assert YOUTUBE_URL, 'YOUTUBE_URL bos olamaz'
    download_youtube(YOUTUBE_URL, RAW_VIDEO, cookies_file=COOKIES_FILE)
else:
    RAW_VIDEO = resolve_uploaded_path(UPLOADED_PATH)

file_size_mb = os.path.getsize(RAW_VIDEO) / (1024 ** 2)
print(f'Kaynak video: {RAW_VIDEO}')
print(f'Dosya boyutu: {file_size_mb:.2f} MB')
if file_size_mb < 1:
    raise RuntimeError('Video dosyasi cok kucuk; yukleme veya kopyalama tamamlanmamis olabilir.')

if FFPROBE_BIN:
    probe = subprocess.run(
        [
            FFPROBE_BIN, '-v', 'error',
            '-show_entries', 'stream=index,codec_type,codec_name,width,height,duration',
            '-of', 'json', RAW_VIDEO,
        ],
        capture_output=True,
        text=True,
    )
    if probe.returncode != 0:
        raise RuntimeError(f'FFprobe kaynak videoyu okuyamadi: {probe.stderr[-3000:]}')
    streams = json.loads(probe.stdout).get('streams', [])
    print('Bulunan akislar:', streams)
    if not any(stream.get('codec_type') == 'video' for stream in streams):
        raise RuntimeError('Dosyada video akisi bulunamadi.')
    if not any(stream.get('codec_type') == 'audio' for stream in streams):
        raise RuntimeError('Dosyada ses akisi bulunamadi; transcript uretilemez.')
else:
    print('FFprobe yok; akis kontrolu FFmpeg donusumu sirasinda yapilacak.')

# Kanonik klip+ses cikarimi (master notebook ve verify_local.py ile ayni fonksiyon).
video_analysis.extract_clip_and_audio(
    FFMPEG_BIN, RAW_VIDEO, VIDEO_PATH, AUDIO_PATH, CLIP_START_SEC, CLIP_DURATION_SEC,
)

print('Klip hazir:', VIDEO_PATH)
print('Ses hazir:', AUDIO_PATH)

## 2. Transcript Çıkarımı — Aday Modeller

Adaylar: `faster-whisper` (tiny / small / large-v3-turbo, lokal & ücretsiz) ve `OpenAI gpt-4o-mini-transcribe` (API, ücretli — anahtar varsa).

In [ ]:
import gc

transcripts = {}
transcript_segments = {}

try:
    cuda_check = subprocess.run(['nvidia-smi'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    cuda_available = cuda_check.returncode == 0
except FileNotFoundError:
    cuda_available = False
local_gpu_opt_in = os.environ.get('YOUTUBE_AUTOMATION_USE_GPU') == '1'
cuda_available = cuda_available and (IS_COLAB or local_gpu_opt_in)
TRANSCRIBE_DEVICE = 'cuda' if cuda_available else 'cpu'
TRANSCRIBE_COMPUTE_TYPE = 'float16' if TRANSCRIBE_DEVICE == 'cuda' else 'int8'
# Karar (gercek test sonuclarina gore): large-v3-turbo GPU'da varsayilan
# (~%91 ortalama kelime guveni), small CPU fallback (~%85). tiny (~%72) sadece
# karsilastirma icin tutuluyor, production icin onerilmez.
TRANSCRIPT_MODELS = ['tiny', 'small']
if TRANSCRIBE_DEVICE == 'cuda':
    TRANSCRIPT_MODELS.append('large-v3-turbo')
else:
    print('GPU bulunamadi; large-v3-turbo cok yavas olacagi icin atlandi. CPU fallback: small.')

print('Transkripsiyon cihazi:', TRANSCRIBE_DEVICE, '| modeller:', TRANSCRIPT_MODELS)

# Bu notebook'un katkisi: AYNI kanonik transcribe_audio() fonksiyonunu birden
# fazla model boyutuyla calistirip karsilastirmak.
for size in TRANSCRIPT_MODELS:
    candidate_name = f'faster-whisper-{size}'
    t0 = time.time()
    text, segment_rows = video_analysis.transcribe_audio(
        AUDIO_PATH, model_size=size, device=TRANSCRIBE_DEVICE, compute_type=TRANSCRIBE_COMPUTE_TYPE,
    )
    latency = time.time() - t0
    transcripts[candidate_name] = text
    transcript_segments[candidate_name] = segment_rows
    log_result(
        'transcript', candidate_name, quality_score=None, cost_usd=0.0, latency_sec=latency,
        notes=f'{len(text)} karakter, {len(segment_rows)} segment, VAD+timestamps',
    )
    gc.collect()

if OPENAI_API_KEY:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    t0 = time.time()
    # gpt-4o-mini-transcribe: whisper-1'in yerini alan guncel, daha ucuz model.
    with open(AUDIO_PATH, 'rb') as f:
        resp = client.audio.transcriptions.create(model='gpt-4o-mini-transcribe', file=f)
    latency = time.time() - t0
    transcripts['openai-gpt-4o-mini-transcribe'] = resp.text
    cost = (CLIP_DURATION_SEC / 60) * 0.003  # yaklasik; platform.openai.com/docs/pricing
    log_result('transcript', 'openai-gpt-4o-mini-transcribe', quality_score=None, cost_usd=cost,
               latency_sec=latency, notes=f'{len(resp.text)} karakter')
else:
    print('OpenAI key yok, openai-gpt-4o-mini-transcribe atlandi.')


In [ ]:
# Ciktilari yan yana oku ve kalite puanini (1-5) kendin gir.
for name, text in transcripts.items():
    print('=' * 20, name, '=' * 20)
    print(text[:800], '...\n')

transcript_quality = {
    # ornek: 'faster-whisper-tiny': 2, 'faster-whisper-small': 3, 'faster-whisper-large-v3-turbo': 4
}

for r in results:
    if r['task'] == 'transcript' and r['candidate'] in transcript_quality:
        r['quality_score_1_5'] = transcript_quality[r['candidate']]
print('Kalite puanlarini transcript_quality sozlugune girip bu hucreyi tekrar calistir.')

## 3. Sahne Tespiti — Aday Yöntemler

Adaylar: PySceneDetect `ContentDetector` ve `AdaptiveDetector`. Kalite ölçütü: bulunan sahne sayısının videoyu mantıklı şekilde bölüp bölmediği (elle kontrol).

In [ ]:
scene_results = {}

# Kanonik detect_scenes() iki dedektorle karsilastirmali calistirilir.
for name, detector_key in [('content-detector', 'content'), ('adaptive-detector', 'adaptive')]:
    t0 = time.time()
    scene_list = video_analysis.detect_scenes(VIDEO_PATH, detector=detector_key)
    latency = time.time() - t0
    scene_results[name] = scene_list
    log_result('scene-detection', name, quality_score=None, cost_usd=0.0, latency_sec=latency,
               notes=f'{len(scene_list)} sahne bulundu')


In [ ]:
scene_quality = {
    # ornek: 'content-detector': 4, 'adaptive-detector': 3
}
for r in results:
    if r['task'] == 'scene-detection' and r['candidate'] in scene_quality:
        r['quality_score_1_5'] = scene_quality[r['candidate']]
print('Sahne sayilarini yukarida gordukten sonra scene_quality sozlugunu doldurup bu hucreyi tekrar calistir.')

## 4. Ses Enerjisi Analizi (Destek Sinyali)

Bu adım tek bir sabit yöntemle (librosa RMS enerji tepe noktaları) çalışır — aday model karşılaştırması gerektirmez, sadece hot-moment skorlamasına ek sinyal sağlar. Bu yüzden burada kalite/maliyet karşılaştırması yapılmaz.

In [ ]:
import numpy as np

t0 = time.time()
# Kanonik compute_audio_peaks(): 16kHz yukleme + RMS + min-mesafeli tepe gruplama.
audio_info = video_analysis.compute_audio_peaks(AUDIO_PATH, top_n=10, min_distance_sec=1.5)
times, rms = audio_info['times'], audio_info['rms']
audio_peak_details = audio_info['peaks']
audio_peaks = [item['time_sec'] for item in audio_peak_details]
latency = time.time() - t0
log_result('audio-energy', 'librosa-rms', quality_score='n/a', cost_usd=0.0, latency_sec=latency,
           notes=f'gruplanmis en yuksek enerjili anlar: {[round(t, 1) for t in audio_peaks]}')


## 5. Yerel + Opsiyonel LLM Hot-Moment Skorlama

Önce transcript, sahne ve ses sinyallerinden API gerektirmeyen yerel hot-moment adayları üretilir. Aynı adaylar, API anahtarı varsa LLM sağlayıcılarıyla karşılaştırılır.

In [ ]:
t0 = time.time()
TRANSCRIPT_PREFERENCE = [
    'faster-whisper-large-v3-turbo',
    'faster-whisper-small',
    'faster-whisper-tiny',
]
BEST_TRANSCRIPT_CANDIDATE = next(
    name for name in TRANSCRIPT_PREFERENCE
    if name in transcript_segments and transcripts.get(name, '').strip()
)
print('Kullanilan transcript:', BEST_TRANSCRIPT_CANDIDATE)

LOCAL_WINDOW_SEC = 30.0
LOCAL_STRIDE_SEC = 10.0
TOP_HOT_MOMENTS = 5

analysis_duration = min(float(CLIP_DURATION_SEC), float(times[-1]))
content_scenes = scene_results.get('content-detector', [])
scene_boundaries = [end for _, end in content_scenes[:-1]] if content_scenes else []

# Kanonik skorlama: master notebook da AYNI score_hot_moments() fonksiyonunu cagirir.
hot_moments = video_analysis.score_hot_moments(
    transcript_segments[BEST_TRANSCRIPT_CANDIDATE], times, rms, scene_boundaries,
    analysis_duration, CLIP_START_SEC, top_n=TOP_HOT_MOMENTS,
    window_sec=LOCAL_WINDOW_SEC, stride_sec=LOCAL_STRIDE_SEC,
)

# Kanonik onizleme klibi uretimi (her ana bir 720p klip).
video_analysis.generate_preview_clips(FFMPEG_BIN, VIDEO_PATH, hot_moments, RESULTS_DIR / 'hot_moment_previews')

# Kanonik manifest: status HER ZAMAN 'review_required' (asla otomatik approved).
hot_moments_manifest = video_analysis.build_hot_moments_manifest(
    hot_moments, transcript_candidate=BEST_TRANSCRIPT_CANDIDATE,
    window_sec=LOCAL_WINDOW_SEC, stride_sec=LOCAL_STRIDE_SEC,
    weights=video_analysis.DEFAULT_WEIGHTS,
    raw_video_path=RAW_VIDEO, analysis_clip_path=VIDEO_PATH,
    clip_start_sec=CLIP_START_SEC, clip_duration_sec=CLIP_DURATION_SEC,
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
hot_moments_path = RESULTS_DIR / 'hot_moments.json'
with open(hot_moments_path, 'w', encoding='utf-8') as hot_file:
    json.dump(hot_moments_manifest, hot_file, ensure_ascii=False, indent=2)

local_latency = time.time() - t0
log_result(
    'hotmoment-scoring', 'local-multisignal-v2', quality_score=None, cost_usd=0.0,
    latency_sec=local_latency,
    notes=str([(item['start_sec'], item['end_sec'], item['score']) for item in hot_moments]),
    status='review_required',
)
print('Yerel hot-moment dosyasi:', hot_moments_path, '(status: review_required)')
display(pd.DataFrame(hot_moments)[['rank', 'aligned_start_sec', 'aligned_end_sec', 'score', 'reason', 'preview_path', 'transcript']])

# Opsiyonel LLM karsilastirmasinda yerel yontemin sectigi ayni pencereleri kullan.
windows = [item['transcript'] for item in hot_moments if item['transcript']]
print(f'{len(windows)} zaman kodlu pencere olusturuldu.')

SCORING_PROMPT = (
    "Asagidaki video transkript parcasini oku. 0-100 arasi bir 'hot moment' (vurucu, "
    "paylasilabilir, dikkat cekici an) skoru ver ve 1 cumlelik gerekce yaz. "
    "SADECE JSON don: {{\"score\": <int>, \"reason\": \"...\"}}\n\nMetin:\n{chunk}"
)


In [ ]:
def parse_json_score(raw):
    try:
        start = raw.find('{'); end = raw.rfind('}') + 1
        return json.loads(raw[start:end])
    except Exception:
        return {'score': None, 'reason': raw[:200]}

llm_outputs = {'local-multisignal-v2': hot_moments}

# --- OpenAI gpt-5.6-luna (gpt-4o-mini'nin yerini alan dusuk maliyetli model) ---
if OPENAI_API_KEY:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    t0 = time.time()
    scored = []
    for chunk in windows:
        resp = client.chat.completions.create(
            model='gpt-5.6-luna',
            messages=[{'role': 'user', 'content': SCORING_PROMPT.format(chunk=chunk)}],
            temperature=0,
        )
        scored.append(parse_json_score(resp.choices[0].message.content))
    latency = time.time() - t0
    llm_outputs['openai-gpt-5.6-luna'] = scored
    approx_tokens = sum(len(c) for c in windows) / 4
    cost = (approx_tokens / 1_000_000) * 1.0  # yaklasik input fiyati, guncelini platform.openai.com/docs/pricing adresinden kontrol et
    log_result('hotmoment-scoring', 'openai-gpt-5.6-luna', quality_score=None, cost_usd=cost, latency_sec=latency,
               notes=str([s.get('score') for s in scored]))
else:
    print('OpenAI key yok, gpt-5.6-luna atlandi.')

# --- Anthropic claude-haiku-4-5 ---
if ANTHROPIC_API_KEY:
    import anthropic
    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    t0 = time.time()
    scored = []
    for chunk in windows:
        resp = client.messages.create(
            model='claude-haiku-4-5',
            max_tokens=200,
            messages=[{'role': 'user', 'content': SCORING_PROMPT.format(chunk=chunk)}],
        )
        scored.append(parse_json_score(resp.content[0].text))
    latency = time.time() - t0
    llm_outputs['anthropic-claude-haiku-4-5'] = scored
    approx_tokens = sum(len(c) for c in windows) / 4
    cost = (approx_tokens / 1_000_000) * 1.00  # yaklasik input fiyati, guncelini platform.claude.com/docs/en/pricing adresinden kontrol et
    log_result('hotmoment-scoring', 'anthropic-claude-haiku-4-5', quality_score=None, cost_usd=cost, latency_sec=latency,
               notes=str([s.get('score') for s in scored]))
else:
    print('Anthropic key yok, claude-haiku-4-5 atlandi.')

# --- Google: SABIT MODEL YOK. Her cagri kanonik router uzerinden gider -------
# (local_pipeline/google_ai.py: Rota A = gemini-3.6-flash -> gemini-3.5-flash ->
#  gemini-3.5-flash-lite -> gemma-4-31b-it, ilk basarida durur, kota defteri
#  tutulur, hatalar siniflandirilir ve sanitize edilir.)
#
# Ucretsiz katman veri politikasi: buraya SADECE yerel olarak uretilmis transcript
# pencereleri gonderilir; ham video ASLA yuklenmez.
GOOGLE_CREDENTIAL = google_ai.resolve_secret('google_api', get_secret)
print('Gemini kimlik bilgisi:', google_ai.describe_credential_state(GOOGLE_CREDENTIAL))

if GOOGLE_CREDENTIAL.configured:
    _g_client = google_ai.build_google_client(GOOGLE_CREDENTIAL)
    _g_manifest = google_ai.build_google_capability_manifest(
        _g_client, credential=GOOGLE_CREDENTIAL)
    _g_ledger = google_ai.QuotaLedger()
    _g_selection = google_ai.select_google_route(
        'hotmoment_review', manifest=_g_manifest, ledger=_g_ledger)
    print(google_ai.format_route_line(_g_selection, GOOGLE_CREDENTIAL))

    t0 = time.time()
    scored, _g_models, _g_failed = [], [], None
    for chunk in windows:
        _g_result = google_ai.run_google_hotmoment_review(
            SCORING_PROMPT.format(chunk=chunk), client=_g_client,
            manifest=_g_manifest, ledger=_g_ledger,
            required_fields=(), checks=(),
        )
        if _g_result['status'] != 'ok':
            _g_failed = _g_result['status_reason']
            break
        _g_models.append(_g_result['model'])
        scored.append(parse_json_score(_g_result['output']))
    latency = time.time() - t0

    if _g_failed:
        # Cikti UYDURULMAZ: basarisizlik acikca 'failed' olarak loglanir.
        log_result('hotmoment-scoring', 'google-router', quality_score=None, cost_usd=0.0,
                   latency_sec=latency, notes=f'Google rotasi basarisiz: {_g_failed}',
                   status='failed')
        print('Google hot-moment incelemesi basarisiz:', _g_failed)
    else:
        _g_label = 'google-' + (sorted(set(_g_models))[0] if _g_models else 'router')
        llm_outputs[_g_label] = scored
        # Ucretsiz katman: dolar maliyeti 0; gercek butce RPM/TPM/RPD kotasidir.
        log_result('hotmoment-scoring', _g_label, quality_score=None, cost_usd=0.0,
                   latency_sec=latency,
                   notes=str([s.get('score') for s in scored])
                         + f' | kullanilan modeller: {sorted(set(_g_models))}')
else:
    print('Google key yok (GeminiKey / GOOGLE_API_KEY), Google adayi atlandi.')
    print(google_ai.credential_help_text('google_api'))

In [ ]:
for name, scored in llm_outputs.items():
    print('=' * 20, name, '=' * 20)
    for s in scored:
        print(s)
    print()

hotmoment_quality = {
    # ornek: 'local-multisignal-v2': 4, 'openai-gpt-5.6-luna': 4
}
for r in results:
    if r['task'] == 'hotmoment-scoring' and r['candidate'] in hotmoment_quality:
        r['quality_score_1_5'] = hotmoment_quality[r['candidate']]
print('Skorlarin tutarliligini/isabetini degerlendirip hotmoment_quality sozlugunu doldur, sonra tekrar calistir.')

## 6. Karşılaştırma Tablosu ve Dışa Aktarım

In [ ]:
df = pd.DataFrame(results)
# Not: satir bazli status artik log_result tarafindan ayarlaniyor; eksikleri guvenli varsayilana tamamla.
if 'status' not in df.columns:
    df['status'] = 'experimental'
else:
    df['status'] = df['status'].fillna('experimental')
display(df)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
out_path = RESULTS_DIR / '01_video_analysis_hotmoment_sonuclar.csv'
df.to_csv(out_path, index=False)
print('Kaydedildi:', out_path)
print('\nBu dosyayi indirip experiments/results/ altina koy; roadmap Faz 1deki provider registry karari bu sonuclara gore verilecek.')

## 7. Tanılama Paketini İndir

Bu hücre transcriptleri, sahne zamanlarını, ses tepe noktalarını ve karşılaştırma CSV'sini tek bir ZIP dosyasında toplar.

In [ ]:
import zipfile

scene_report = {}
for detector_name, scenes in globals().get('scene_results', {}).items():
    scene_report[detector_name] = {
        'count': len(scenes),
        # detect_scenes() zaten (start_sec, end_sec) float ikilileri dondurur.
        'scenes': [{'start_sec': round(start, 3), 'end_sec': round(end, 3)} for start, end in scenes],
    }

diagnostic_report = {
    'status': 'experimental',
    'status_reason': 'provider comparison notebook output; not a final production artifact',
    'source': {
        'raw_video': globals().get('RAW_VIDEO'),
        'video_path': globals().get('VIDEO_PATH'),
        'audio_path': globals().get('AUDIO_PATH'),
        'clip_start_sec': globals().get('CLIP_START_SEC'),
        'clip_duration_sec': globals().get('CLIP_DURATION_SEC'),
    },
    'transcripts': {
        name: {
            'character_count': len(text),
            'full_text': text,
            'segments': globals().get('transcript_segments', {}).get(name, []),
        }
        for name, text in globals().get('transcripts', {}).items()
    },
    'scene_detection': scene_report,
    'audio_peaks_sec': globals().get('audio_peaks', []),
    'audio_peak_details': globals().get('audio_peak_details', []),
    'local_hot_moments': globals().get('hot_moments', []),
    'llm_hotmoment_outputs': globals().get('llm_outputs', {}),
    'benchmark_results': globals().get('results', []),
}

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
report_path = RESULTS_DIR / '01_video_analysis_tanilama.json'
zip_path = RESULTS_DIR / '01_video_analysis_paylasim_paketi.zip'

with open(report_path, 'w', encoding='utf-8') as report_file:
    json.dump(diagnostic_report, report_file, ensure_ascii=False, indent=2)

with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(report_path, arcname=report_path.name)
    if Path(out_path).exists():
        archive.write(out_path, arcname=Path(out_path).name)
    if Path(globals().get('hot_moments_path', '')).exists():
        archive.write(hot_moments_path, arcname=Path(hot_moments_path).name)
    for preview_file in (RESULTS_DIR / 'hot_moment_previews').glob('*.mp4'):
        archive.write(preview_file, arcname=f'hot_moment_previews/{preview_file.name}')

print('Paylasim paketi hazir:', zip_path)

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as exc:
    print('Otomatik indirme baslatilamadi. Files panelinden elle indir:', zip_path)
    print('Detay:', exc)


## 8. Karar (elle doldurulacak)

Yukarıdaki tabloyu inceledikten sonra bu hücreyi düzenleyip kararını ve kısa gerekçeni yaz. Bu karar Faz 1'de `packages/providers/vision-analysis` ve `packages/providers/llm` adaptörlerinin varsayılan sağlayıcısını belirleyecek.

- **Transcript:** _(seçilen aday)_ — gerekçe: _...
- **Sahne tespiti:** _(seçilen aday)_ — gerekçe: _...
- **Hot-moment skorlama:** _(seçilen aday)_ — gerekçe: _...
